# **Projeto Integrador VI**
# **🍺 Inteligência de Risco em Comodato de Chopp**

**2° Semestre/2026**

**Curso:** Ciência de Dados para Negócio | **Instituição:** FATEC Votorantim  
**Alunos:** Ana Elisa · Arthur Nunes · Bruno Araujo · Caio Corrá · Lucas Camelo · Nicole Fava

---

## 🗺️ Pipeline do Projeto

| Fase  | Nome             | Objetivo                                            |
| :---: | :--------------  | :-------------------------------------------------- |
| **0** | ⚙️ Configuração | Imports, **painel de hiperparâmetros**, utilitários  |
| **1** | 🎯 Negócio      | Contexto, hipótese, KPIs                            |
| **2** | 💾 Dados        | Unity Catalog (table) · CSV/SQL como alternativa    |
| **3** | 🔍 EDA          | Auditoria, perfil 360°, risco, aging                |
| **4** | 🔧 Features     | `df_ml_completo` (usuários) vs `df_treino` (modelo) |
| **5** | 🤖 Modelagem    | Pipelines, MODEL_REGISTRY, CV                       |
| **6** | 📊 Avaliação    | ROC, PR, matrizes, importâncias                     |
| **7** | 🚀 Deploy       | Versionamento, `predict_by_id()`                    |  
#
> **Meta:** Classificar clientes como **Alto Risco** (≥ 20% atrasos financeiros _ou_ de comodato).

# ⚙️Fase 0 - Configuração

## 🎛️ Painel de Controle do Experimento

Toda variável capaz de alterar o resultado de um treino está declarada em **uma única célula**. As fases seguintes não contêm números mágicos: elas apenas **leem** o painel. Essa é a condição para que o registro no MLflow seja fiel, porque um hiperparâmetro escrito no meio do notebook é um hiperparâmetro que o log não enxerga.

### 📐 A divisão em três blocos

| Bloco | O que é | Efeito de mudar |
| :--- | :--- | :--- |
| **1 · `CONSTANTES`** | Identidade, caminhos, semente, paleta. | Nenhum sobre o aprendizado. `RANDOM_STATE` mora aqui de propósito: é dispositivo de **reprodutibilidade**, não de otimização — variá-lo para "melhorar" a métrica é escolher o ruído favorável, não um modelo melhor. |
| **2 · `HP_ARQUITETURA`** | A **forma** do modelo: definição do alvo, features, pré-processamento e capacidade de cada algoritmo. | Muda o que o modelo **é**. Exige retreino. Rodadas com arquiteturas diferentes comparam objetos diferentes. |
| **3 · `HP_TREINAMENTO`** | **Como** se aprende, valida e decide: partição, CV, métricas coletadas, critério de campeão, limiares. | Muda a **estimativa** da performance e qual modelo vence — não muda o modelo em si. |

A fronteira entre os blocos 2 e 3 é a pergunta útil na hora de interpretar um resultado: *o modelo ficou melhor, ou só a forma de medi-lo mudou?*

### 🔁 Ciclo de fine-tuning

1. Altere um valor em `HP_ARQUITETURA` ou `HP_TREINAMENTO`.
2. Atualize `EXPERIMENT_TAG` e `EXPERIMENT_NOTA` — é por essa etiqueta que a rodada será reencontrada.
3. Re-execute a partir da célula do painel.
4. Compare no MLflow filtrando por `tags.experiment_tag` ou `tags.config_hash`.

### 🔐 `CONFIG_HASH`

Cada rodada recebe um SHA-256 truncado da configuração inteira, além de dois hashes parciais (`arquitetura` e `treinamento`). Dois runs com o mesmo hash usaram exatamente os mesmos hiperparâmetros. É o que permite provar, no painel, que uma diferença de métrica veio da configuração — e não de uma execução esquecida com valores antigos.

> ⚠️ **Os aliases** (`LIMITE_ATRASO`, `CV_FOLDS`, `TEST_SIZE`, …) são *views* somente-leitura do painel, mantidos para as fases 3-7 continuarem legíveis. Nunca reatribua um alias diretamente: altere o dicionário e re-execute a célula, senão o MLflow registra uma configuração diferente da que de fato rodou.

In [ ]:
# ─── Bibliotecas Padrão ───────────────────────────────────────────────────────
import warnings, pickle, os, json, hashlib
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, cast

warnings.filterwarnings("ignore")

# ─── Manipulação de Dados ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Visualização ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter

# ─── Machine Learning ─────────────────────────────────────────────────────────
from joblib import Memory
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

# ─── MLflow para Experiment Tracking ──────────────────────────────────────────
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# ── Exibição Visual do Pipeline ──────────────────────────────────────────────
from sklearn import set_config
from IPython.display import display

set_config(display="diagram")
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.family": "DejaVu Sans"})


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║                                                                            ║
# ║   PAINEL DE CONTROLE DO EXPERIMENTO                                        ║
# ║                                                                            ║
# ║   Toda variável que muda o resultado de um treino mora AQUI. Nenhum        ║
# ║   número mágico é escrito adiante no notebook: as fases 4-7 só LEEM        ║
# ║   destes dicionários. Para uma nova rodada de fine-tuning, altere um       ║
# ║   valor abaixo, mude EXPERIMENT_TAG e re-execute — o MLflow registra a     ║
# ║   configuração inteira e o hash que a identifica.                          ║
# ║                                                                            ║
# ║   Três blocos, com fronteiras deliberadas:                                 ║
# ║     1. CONSTANTES        → identidade e infraestrutura. NÃO se tuna.       ║
# ║     2. HP_ARQUITETURA    → a FORMA do modelo (features, preproc, capacidade║
# ║                            do algoritmo). Mudar aqui muda o que o modelo   ║
# ║                            É — exige retreino e invalida comparação com    ║
# ║                            runs anteriores no eixo de arquitetura.         ║
# ║     3. HP_TREINAMENTO    → COMO se aprende, valida e decide (split, CV,    ║
# ║                            scoring, critério de campeão, thresholds).      ║
# ║                            Mudar aqui muda a estimativa, não o modelo.     ║
# ║                                                                            ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 1 · CONSTANTES  —  identidade, caminhos, semente, apresentação
#  Não são hiperparâmetros: nenhuma delas é objeto de busca. RANDOM_STATE mora
#  aqui porque é dispositivo de REPRODUTIBILIDADE, não de otimização — variá-lo
#  para "melhorar" a métrica é escolher o ruído favorável, não um modelo melhor.
# ══════════════════════════════════════════════════════════════════════════════
RANDOM_STATE = 42          # semente única, propagada a split/CV/estimadores
MODEL_VERSION = "1.0.4"    # versão do pipeline

# Identificador humano desta rodada de tuning. TROQUE a cada experimento —
# é a etiqueta pela qual você vai reencontrar a rodada no painel do MLflow.
EXPERIMENT_TAG = "rodada_v1.0.4"
EXPERIMENT_NOTA = (
    "Primeira rodada com hiperparâmetros centralizados no painel de controle. "
    "Valores idênticos aos da v1.0.3 — serve de âncora para comparar os tunings seguintes."
)

DATA_DIR = "."             # sobrescrito na Fase 2B pelo caminho do Volume
MODEL_DIR = "modelos"      # saída dos PKLs

# ─── Paleta de Cores (apresentação — sem efeito sobre o modelo) ───────────────
CORES_MODELO = {
    "Regressão Logística": "#3498DB",
    "Árvore de Decisão": "#2ECC71",
    "Random Forest": "#E74C3C",
}
CORES_RISCO = {
    "BAIXO": "#2ECC71",
    "MÉDIO": "#F39C12",
    "ALTO": "#E74C3C",
}

Path(MODEL_DIR).mkdir(exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 2 · HIPERPARÂMETROS DE ARQUITETURA
#  A forma do modelo. Subdividido em três camadas que respondem, na ordem:
#    2.1 alvo    → o que é "alto risco"? (define o problema)
#    2.2 dados   → quais linhas e colunas o modelo vê? (define o espaço)
#    2.3 preproc → como cada coluna é transformada? (define a representação)
#    2.4 modelos → qual a capacidade de cada algoritmo? (define a família)
# ══════════════════════════════════════════════════════════════════════════════
HP_ARQUITETURA: Dict[str, Any] = {

    # ── 2.1 Definição do alvo ─────────────────────────────────────────────────
    # Não é um parâmetro estatístico: é a REGRA DE NEGÓCIO. Mudar limite_atraso
    # redefine o que se está prevendo, então métricas de rodadas com limites
    # diferentes NÃO são comparáveis entre si — são problemas distintos.
    "alvo": {
        "coluna": "ALTO_RISCO",
        "limite_atraso": 0.20,      # > 20% de atrasos ⇒ ALTO_RISCO
        "combinador": "OU",         # atraso financeiro OU de comodato
    },

    # ── 2.2 Espaço de dados ───────────────────────────────────────────────────
    "dados": {
        # Filtro de elegibilidade (cold-start): clientes com histórico curto
        # demais para que as taxas de atraso signifiquem alguma coisa.
        # ⚠️ Aplicado como FREQUENCIA_COMPRAS > min_compras (exclusivo):
        #    min_compras=2 mantém quem tem 3 ou mais compras.
        "min_compras": 2,

        "features_num": [
            "FREQUENCIA_COMPRAS",
            "TICKET_MEDIO",
            "TOTAL_GASTO",
            "DIAS_DESDE_PRIMEIRA_COMPRA",
            "DIAS_DESDE_ULTIMA_COMPRA",
            "TOTAL_PARCELAS",
            "TOTAL_COMODATOS",
            # ⚠️ As duas abaixo compartilham origem aritmética com o alvo.
            # Removê-las é o experimento de ablação nº 1 (ver LIMITACOES_CONHECIDAS).
            "MEDIA_DIAS_ATRASO_PAG",
            "MEDIA_DIAS_ATRASO_COM",
        ],
        "features_cat": [
            "PERFIL",
            "CIDADE",
            "PAGAMENTO",
        ],
        # Colunas que existem em df_ml_completo mas nunca entram no modelo:
        # ou são identidade, ou são o próprio alvo disfarçado.
        "features_proibidas": [
            "ID_PESSOA", "NOME_CLIENTE",
            "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",   # constroem o alvo
            "PARCELAS_ATRASADAS", "COMODATOS_ATRASADOS",       # numeradores do alvo
        ],
    },

    # ── 2.3 Pré-processamento ─────────────────────────────────────────────────
    "preproc": {
        "scaler_numerico": "standard",    # standard | minmax | robust | none
        "encoder_categorico": "onehot",
        "onehot_drop": "first",           # evita colinearidade na Reg. Logística
        "onehot_handle_unknown": "ignore",# categoria nova em produção não quebra
        "remainder": "drop",
    },

    # ── 2.4 Capacidade dos algoritmos ─────────────────────────────────────────
    # Um dicionário por modelo, com os kwargs que vão direto ao construtor.
    # 'ativo' liga/desliga o candidato sem apagar a configuração.
    "modelos": {
        "Regressão Logística": {
            "ativo": True,
            "classe": "LogisticRegression",
            "params": {
                "C": 1.0,                  # ↓C = mais regularização
                "penalty": "l2",
                "solver": "lbfgs",
                "max_iter": 1000,
                "class_weight": "balanced",
            },
        },
        "Árvore de Decisão": {
            "ativo": True,
            "classe": "DecisionTreeClassifier",
            "params": {
                "max_depth": 6,            # principal controle de overfitting
                "min_samples_split": 2,
                "min_samples_leaf": 1,
                "criterion": "gini",
                "class_weight": "balanced",
            },
        },
        "Random Forest": {
            "ativo": True,
            "classe": "RandomForestClassifier",
            "params": {
                "n_estimators": 200,
                "max_depth": 8,
                "min_samples_split": 2,
                "min_samples_leaf": 3,
                "max_features": "sqrt",
                "bootstrap": True,
                "class_weight": "balanced",
                # n_jobs=1: no serverless a contagem de cores é baixa e não
                # configurável; -1 não escala e ainda disputa CPU com o CV.
                "n_jobs": 1,
            },
        },
    },
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 3 · HIPERPARÂMETROS DE TREINAMENTO
#  Como se aprende, como se mede e como se decide. Mudar algo aqui altera a
#  ESTIMATIVA da performance (e qual modelo vence), não o modelo em si.
# ══════════════════════════════════════════════════════════════════════════════
HP_TREINAMENTO: Dict[str, Any] = {

    # ── 3.1 Partição Treino/Teste ─────────────────────────────────────────────
    "split": {
        "test_size": 0.30,
        "stratify": True,      # obrigatório: 86,8% de positivos
        "shuffle": True,
    },

    # ── 3.2 Validação Cruzada ─────────────────────────────────────────────────
    "cv": {
        "n_splits": 5,
        "shuffle": True,
        "estrategia": "StratifiedKFold",
        # n_jobs=None (sequencial): o serverless não dá temp dir gravável para
        # o loky, e com ~333 linhas o paralelismo não paga o próprio overhead.
        "n_jobs": None,
        "return_train_score": True,   # habilita o gap treino−validação
    },

    # ── 3.3 Métricas coletadas na CV ──────────────────────────────────────────
    # A classe positiva é MAIORIA (86,8%). Nesse regime accuracy e AUC enganam:
    # "todos são risco" já acerta 86,8%. Por isso o conjunto inclui métricas
    # insensíveis a prevalência (balanced_accuracy, MCC) e as versões _macro.
    "scoring_cv": {
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
        "f1": "f1",
        "recall": "recall",
        "precision": "precision",
        "balanced_accuracy": "balanced_accuracy",
        "matthews_corrcoef": "matthews_corrcoef",
        "f1_macro": "f1_macro",
        "recall_macro": "recall_macro",
    },

    # ── 3.4 Critério de seleção do campeão ────────────────────────────────────
    # MCC, não AUC: só sobe quando AMBAS as classes são bem classificadas —
    # exatamente o que AUC e Accuracy escondem com 86,8% de positivos.
    # O baseline majoritário tem MCC = 0 por construção, o que dá um piso claro.
    # ⚠️ Este é o ÚNICO lugar onde o campeão é definido; as Fases 6 e 7 leem daqui.
    #
    # ⚠️⚠️ SOBRE 'origem' — leia antes da segunda rodada de tuning:
    #   'teste' = escolhe o campeão pelo holdout (~19 negativos). Válido para
    #             UMA rodada. A cada configuração adicional que você compara
    #             pelo test_*, o holdout deixa de ser estimativa não-viesada:
    #             um único cliente reclassificado move a specificity ~5 p.p.,
    #             então após várias rodadas o "melhor test MCC" é ruído escolhido.
    #   'cv'    = escolhe pela validação cruzada do TREINO, deixando o holdout
    #             intocado para uma única medição final de confirmação.
    #             É a opção correta quando se compara muitas configurações.
    "selecao": {
        "metrica": "MCC",           # chave em avaliar_no_teste() / metricas_cv
        "origem": "teste",          # teste | cv  (veja o aviso acima)
        "maior_melhor": True,
        "criterio_desempate": "Balanced_Accuracy",
    },

    # ── 3.4b Intervalos de confiança ──────────────────────────────────────────
    # Com ~19 negativos no teste, o IC é a parte mais informativa do resultado.
    # z=1.96 ⇒ 95%. Declarado aqui porque é decisão metodológica citada na Fase 6.
    "intervalo_confianca": {
        "z": 1.96,
        "nivel": 0.95,
    },

    # ── 3.5 Baseline de referência ────────────────────────────────────────────
    # Sem esta âncora nenhuma métrica significa nada.
    "baseline": {
        "ativo": True,
        "strategy": "most_frequent",
    },

    # ── 3.6 Thresholds de decisão (produção) ──────────────────────────────────
    # Aplicados DEPOIS do treino, sobre predict_proba. Tunáveis sem retreinar:
    # baixar 'classificacao' aumenta recall e reduz precisão.
    "threshold": {
        "classificacao": 0.50,   # ≥ ⇒ classificado como ALTO_RISCO
        "faixa_baixo": 0.35,     # prob < 0.35            ⇒ BAIXO
        "faixa_medio": 0.65,     # 0.35 ≤ prob < 0.65     ⇒ MÉDIO; ≥ 0.65 ⇒ ALTO
        # Grade varrida na Fase 5 para escolher o ponto de operação por curva.
        "sweep_inicio": 0.20,
        "sweep_fim": 0.80,
        "sweep_passo": 0.05,
    },

}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 4 · METAS DE NEGÓCIO
#  Deliberadamente FORA dos dois blocos acima. Metas não são hiperparâmetro:
#  foram fixadas na Fase 1, antes do experimento, e não devem variar entre
#  rodadas de tuning — mover a trave depois de ver o resultado não é tuning.
#  Ficam fora também do CONFIG_HASH: alterá-las não muda nem o modelo nem o
#  protocolo, e não deveria fazer o MLflow reportar diferença onde não houve.
# ══════════════════════════════════════════════════════════════════════════════
METAS_KPI: Dict[str, float] = {
    "auc_roc": 0.75,
    "recall_classe1": 0.70,
    "accuracy": 0.65,
}


# ══════════════════════════════════════════════════════════════════════════════
#  ALIASES DE COMPATIBILIDADE
#  As fases 3-7 e as funções utilitárias continuam lendo estes nomes curtos.
#  São VIEWS somente-leitura do painel acima — nunca reatribua um alias direto:
#  altere o dicionário e re-execute esta célula, senão o MLflow registra uma
#  configuração diferente da que de fato rodou.
# ══════════════════════════════════════════════════════════════════════════════
LIMITE_ATRASO = HP_ARQUITETURA["alvo"]["limite_atraso"]
TARGET        = HP_ARQUITETURA["alvo"]["coluna"]
MIN_COMPRAS   = HP_ARQUITETURA["dados"]["min_compras"]
FEATURES_NUM  = HP_ARQUITETURA["dados"]["features_num"]
FEATURES_CAT  = HP_ARQUITETURA["dados"]["features_cat"]
ALL_FEATURES  = FEATURES_NUM + FEATURES_CAT

TEST_SIZE     = HP_TREINAMENTO["split"]["test_size"]
CV_FOLDS      = HP_TREINAMENTO["cv"]["n_splits"]
SCORING_CV    = HP_TREINAMENTO["scoring_cv"]
THRESHOLD_PADRAO = HP_TREINAMENTO["threshold"]["classificacao"]


# ══════════════════════════════════════════════════════════════════════════════
#  MAPA DE MÉTRICAS: chave interna (PT) → chave logada no MLflow (ASCII)
#  Fonte de verdade ÚNICA, usada por _logar_metricas_teste() na Fase 5 e pelo
#  order_by do search_runs() na Fase 6. Não deduza a chave MLflow com .lower():
#  'Recall' é logado como 'recall_classe1', e 'Precisão' tem acento — que o
#  MLflow rejeita em cláusula order_by.
# ══════════════════════════════════════════════════════════════════════════════
MAPA_METRICA_MLFLOW: Dict[str, str] = {
    "AUC": "auc", "AUC_ic_low": "auc_ic_low", "AUC_ic_high": "auc_ic_high",
    "AUC_se": "auc_se", "AP": "ap_classe1", "AP_classe0": "ap_classe0",
    "F1": "f1_classe1", "Recall": "recall_classe1",
    "Recall_ic_low": "recall_classe1_ic_low", "Recall_ic_high": "recall_classe1_ic_high",
    "Precisão": "precision_classe1", "Specificity": "specificity_classe0",
    "Specificity_ic_low": "specificity_ic_low", "Specificity_ic_high": "specificity_ic_high",
    "Precisão_classe0": "precision_classe0", "F1_classe0": "f1_classe0",
    "Accuracy": "accuracy", "Balanced_Accuracy": "balanced_accuracy",
    "MCC": "mcc", "Brier": "brier", "F2": "f2_classe1",
}

# Métricas realmente disponíveis em cada origem de seleção. A lista de CV é
# menor: metricas_cv_pt não carrega Specificity, Brier, F2 nem Accuracy.
METRICAS_DISPONIVEIS: Dict[str, set] = {
    "teste": set(MAPA_METRICA_MLFLOW),
    "cv": {"AUC", "AUC_std", "AP", "F1", "Recall", "Precisão", "Balanced_Accuracy", "MCC"},
}


# ══════════════════════════════════════════════════════════════════════════════
#  ASSINATURA DA CONFIGURAÇÃO
#  Hash determinístico de (arquitetura + treinamento). Duas rodadas com o mesmo
#  hash usaram exatamente os mesmos hiperparâmetros — é o que permite provar,
#  no painel do MLflow, que uma diferença de métrica veio da configuração e não
#  de uma execução esquecida com valores antigos.
# ══════════════════════════════════════════════════════════════════════════════
def _serializar_config(*blocos: dict) -> str:
    """JSON canônico (chaves ordenadas) dos blocos de hiperparâmetros."""
    return json.dumps(blocos, sort_keys=True, ensure_ascii=False, default=str)


def calcular_hash_config(arquitetura: dict, treinamento: dict, tamanho: int = 12) -> str:
    """SHA-256 truncado da configuração — identidade curta do experimento."""
    return hashlib.sha256(
        _serializar_config(arquitetura, treinamento).encode("utf-8")
    ).hexdigest()[:tamanho]


def achatar_config(d: dict, prefixo: str = "", sep: str = ".") -> Dict[str, Any]:
    """
    Achata dicionário aninhado em pares chave→valor de um nível só.
    Necessário porque mlflow.log_params() não aceita valores aninhados; listas
    viram string separada por vírgula para permanecerem filtráveis no painel.
    """
    plano: Dict[str, Any] = {}
    for chave, valor in d.items():
        nome = f"{prefixo}{sep}{chave}" if prefixo else str(chave)
        if isinstance(valor, dict):
            plano.update(achatar_config(valor, nome, sep))
        elif isinstance(valor, (list, tuple)):
            plano[nome] = ", ".join(map(str, valor))
        else:
            plano[nome] = valor
    return plano


CONFIG_HASH = calcular_hash_config(HP_ARQUITETURA, HP_TREINAMENTO)
CONFIG_HASH_ARQ = calcular_hash_config(HP_ARQUITETURA, {})
CONFIG_HASH_TRN = calcular_hash_config({}, HP_TREINAMENTO)

# Snapshot congelado: é ESTE objeto que vai para o MLflow como artefato.
CONFIG_EXPERIMENTO: Dict[str, Any] = {
    "experiment_tag": EXPERIMENT_TAG,
    "nota": EXPERIMENT_NOTA,
    "model_version": MODEL_VERSION,
    "random_state": RANDOM_STATE,
    "config_hash": CONFIG_HASH,
    "config_hash_arquitetura": CONFIG_HASH_ARQ,
    "config_hash_treinamento": CONFIG_HASH_TRN,
    "hp_arquitetura": HP_ARQUITETURA,
    "hp_treinamento": HP_TREINAMENTO,
}


# ─── Validação defensiva do painel ────────────────────────────────────────────
# Erro de digitação em hiperparâmetro é silencioso e caro: o notebook roda até
# o fim e você só descobre semanas depois que a rodada media outra coisa.
def validar_config() -> list:
    """Checa coerência interna do painel. Retorna lista de problemas."""
    problemas = []
    thr = HP_TREINAMENTO["threshold"]
    if not 0 < thr["classificacao"] < 1:
        problemas.append("threshold.classificacao deve estar em (0, 1).")
    if not thr["faixa_baixo"] < thr["faixa_medio"]:
        problemas.append("threshold.faixa_baixo deve ser menor que faixa_medio.")
    if not 0 < HP_TREINAMENTO["split"]["test_size"] < 1:
        problemas.append("split.test_size deve estar em (0, 1).")
    if HP_TREINAMENTO["cv"]["n_splits"] < 2:
        problemas.append("cv.n_splits deve ser >= 2.")
    if not 0 < HP_ARQUITETURA["alvo"]["limite_atraso"] < 1:
        problemas.append("alvo.limite_atraso deve estar em (0, 1).")

    # Sobreposição feature × lista de proibidas = vazamento silencioso do alvo
    proibidas = set(HP_ARQUITETURA["dados"]["features_proibidas"])
    vazamento = proibidas.intersection(set(ALL_FEATURES))
    if vazamento:
        problemas.append(f"features proibidas presentes no modelo: {sorted(vazamento)}")

    if len(set(ALL_FEATURES)) != len(ALL_FEATURES):
        problemas.append("há features duplicadas entre features_num e features_cat.")

    sel = HP_TREINAMENTO["selecao"]
    if sel["origem"] not in {"teste", "cv"}:
        problemas.append("selecao.origem deve ser 'teste' ou 'cv'.")
    else:
        # A métrica precisa EXISTIR na origem escolhida. Sem esta checagem,
        # selecionar_campeao() cai no default -inf para todos os modelos e
        # elege o primeiro da ordem de inserção — silenciosamente, sem erro.
        validas = METRICAS_DISPONIVEIS[sel["origem"]]
        for campo in ("metrica", "criterio_desempate"):
            if sel[campo] not in validas:
                problemas.append(
                    f"selecao.{campo}='{sel[campo]}' não existe nas métricas de "
                    f"'{sel['origem']}'. Disponíveis: {sorted(validas)}"
                )

    thr_sweep = HP_TREINAMENTO["threshold"]
    if not thr_sweep["sweep_inicio"] < thr_sweep["sweep_fim"]:
        problemas.append("threshold.sweep_inicio deve ser menor que sweep_fim.")
    if thr_sweep["sweep_passo"] <= 0:
        problemas.append("threshold.sweep_passo deve ser positivo.")

    ativos = [n for n, c in HP_ARQUITETURA["modelos"].items() if c.get("ativo")]
    if not ativos:
        problemas.append("nenhum modelo ativo em HP_ARQUITETURA['modelos'].")

    return problemas


_PROBLEMAS_CONFIG = validar_config()

# ─── Relatório do painel ──────────────────────────────────────────────────────
_modelos_ativos = [n for n, c in HP_ARQUITETURA["modelos"].items() if c.get("ativo")]

print("=" * 78)
print(f"{'PAINEL DE CONTROLE DO EXPERIMENTO':^78}")
print("=" * 78)
print(f"  Rodada         : {EXPERIMENT_TAG}")
print(f"  Versão pipeline: {MODEL_VERSION}   |   Semente: {RANDOM_STATE}")
print(f"  Hash da config : {CONFIG_HASH}   (arq {CONFIG_HASH_ARQ} · trn {CONFIG_HASH_TRN})")
print("-" * 78)
print("  ARQUITETURA")
print(f"    ├─ Alvo         : {TARGET}  (atraso > {LIMITE_ATRASO:.0%}, regra {HP_ARQUITETURA['alvo']['combinador']})")
print(f"    ├─ Elegibilidade: compras > {MIN_COMPRAS}")
print(f"    ├─ Features     : {len(ALL_FEATURES)}  ({len(FEATURES_NUM)} num + {len(FEATURES_CAT)} cat)")
print(f"    ├─ Pré-proc     : {HP_ARQUITETURA['preproc']['scaler_numerico']} + "
      f"{HP_ARQUITETURA['preproc']['encoder_categorico']}(drop={HP_ARQUITETURA['preproc']['onehot_drop']})")
print(f"    └─ Candidatos   : {len(_modelos_ativos)} ativos → {', '.join(_modelos_ativos)}")
print("  TREINAMENTO")
print(f"    ├─ Split        : {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} estratificado")
print(f"    ├─ Validação    : {CV_FOLDS}-Fold {HP_TREINAMENTO['cv']['estrategia']}")
print(f"    ├─ Métricas CV  : {len(SCORING_CV)} coletadas")
print(f"    ├─ Campeão por  : {HP_TREINAMENTO['selecao']['metrica']} "
      f"({HP_TREINAMENTO['selecao']['origem']}), desempate por {HP_TREINAMENTO['selecao']['criterio_desempate']}")
print(f"    └─ Threshold    : {THRESHOLD_PADRAO:.2f}  "
      f"(faixas: <{HP_TREINAMENTO['threshold']['faixa_baixo']:.2f} BAIXO · "
      f"<{HP_TREINAMENTO['threshold']['faixa_medio']:.2f} MÉDIO · ALTO)")
print("-" * 78)

if _PROBLEMAS_CONFIG:
    print("  ❌ CONFIGURAÇÃO INCONSISTENTE — corrija antes de treinar:")
    for p in _PROBLEMAS_CONFIG:
        print(f"     • {p}")
    raise ValueError(f"{len(_PROBLEMAS_CONFIG)} problema(s) no painel de controle.")

print(f"  ✅ Painel validado  |  pandas {pd.__version__} · numpy {np.__version__} · mlflow {mlflow.__version__}")

if HP_TREINAMENTO["selecao"]["origem"] == "teste":
    print("-" * 78)
    print("  ⚠️  selecao.origem = 'teste': o campeão é escolhido pelo holdout.")
    print("      Isso é válido para uma rodada isolada. Se você vai comparar")
    print("      VÁRIAS configurações de tuning, troque para 'cv' — senão o")
    print("      holdout vira parte do processo de escolha e deixa de ser uma")
    print("      estimativa honesta da performance futura.")
print("=" * 78)
print("\n  💡 Para uma nova rodada de fine-tuning:")
print("     1. altere o valor em HP_ARQUITETURA ou HP_TREINAMENTO")
print("     2. atualize EXPERIMENT_TAG e EXPERIMENT_NOTA")
print("     3. re-execute a partir desta célula — o MLflow registra config + hash")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FUNÇÕES UTILITÁRIAS
#
#  Escopo reduzido nesta versão: formatação e categorização, que são do
#  DOMÍNIO DA APRESENTAÇÃO deste notebook.
#
#  As antigas build_dimensoes(), calc_atraso_financeiro(), calc_atraso_comodato()
#  e agrupar_cidades() foram REMOVIDAS daqui e vivem agora em
#  Extracao_Dados_Consolidados.ipynb. O motivo é de fronteira: todas operavam no
#  grão transacional (tabelas_db, parcelas, contratos), que este notebook não
#  carrega mais — ele lê dataset_consolidado.csv, já no grão do cliente.
#
#  Mantê-las aqui significaria duas implementações da mesma regra de negócio em
#  arquivos diferentes, que divergem no dia em que alguém corrige só uma.
# ══════════════════════════════════════════════════════════════════════════════

# ─── Formatadores ─────────────────────────────────────────────────────────────
def fmt_moeda(x, pos=None):
    """Formata valores como moeda brasileira abreviada."""
    if x >= 1e6:
        return f"R$ {x*1e-6:.1f}M"
    if x >= 1e3:
        return f"R$ {x*1e-3:.1f}k"
    return f"R$ {x:.0f}"


def fmt_numero(x, pos=None):
    """Formata inteiros com separador de milhar ponto (.)"""
    return f"{int(x):,}".replace(",", ".")


# ─── Aging ────────────────────────────────────────────────────────────────────
# A ORDEM importa: aging é categórica ORDINAL. Sem declarar a ordem, os gráficos
# saem em ordem alfabética e "1-3 Dias" aparece depois de "+30 Dias".
# As colunas AGING_PAGAMENTO/AGING_COMODATO já vêm categorizadas do dataset;
# categorizar_atraso() fica disponível para recategorizar em outra escala.
ORDEM_AGING = [
    "Sem Atraso",
    "1-3 Dias",
    "4-7 Dias",
    "8-15 Dias",
    "16-20 Dias",
    "21-30 Dias",
    "+30 Dias",
]


def categorizar_atraso(dias) -> str:
    """Converte dias de atraso em faixa categórica de aging."""
    d = pd.to_numeric(dias, errors="coerce")
    if pd.isna(d) or d <= 0:
        return "Sem Atraso"
    if d <= 3:
        return "1-3 Dias"
    if d <= 7:
        return "4-7 Dias"
    if d <= 15:
        return "8-15 Dias"
    if d <= 20:
        return "16-20 Dias"
    if d <= 30:
        return "21-30 Dias"
    return "+30 Dias"


print("✅ Funções utilitárias carregadas!")
print("   ├─ fmt_moeda() | fmt_numero()")
print("   ├─ categorizar_atraso(dias) → faixa de aging")
print("   └─ ORDEM_AGING (ordem canônica das faixas)")
print("\n   ℹ️  build_dimensoes(), calc_atraso_*() e agrupar_cidades() migraram")
print("      para Extracao_Dados_Consolidados.ipynb — operam no grão")
print("      transacional, que este notebook não carrega mais.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════════════════
#  CONFIGURAÇÃO DO MLFLOW EXPERIMENT TRACKING (Databricks Free Edition · Serverless Compute)
# ═══════════════════════════════════════════════════════════════════════════════════════════

# ─── Limpeza defensiva de run órfão ───────────────────────────────────────────
# mlflow.set_tags()/log_metric() chamados FORA de um run não falham: eles abrem um run implícito que nunca é encerrado. Se a versão anterior deste notebook rodou na sessão, esse run fantasma ainda está ativo e capturaria tudo que viesse depois. Encerramos antes de começar.
if mlflow.active_run() is not None:
    print(f"Encerrando run órfão da sessão anterior: {mlflow.active_run().info.run_id}")
    mlflow.end_run()

# ─── Identificação do Experimento ─────────────────────────────────────────────
EXPERIMENT_NAME = "/Users/brunofsaraujo@hotmail.com/Chopp_Cia_Experimentos"

# Define (ou cria) o experimento antes de qualquer operação de logging. Ordem importa: tags/runs criados antes disto iriam para o experimento default.
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

# ─── Autolog DESLIGADO — logging 100% manual ─────────────────────────────────
# Dois motivos concretos:
#   1) O MLflow DESABILITA o autolog durante cross_validate() (a função está na 
#      lista interna _apis_autologging_disabled). Ou seja: as métricas de CV
#      nunca chegariam ao MLflow por autolog — só o fit() final seria capturado.
#   2) log_post_training_metrics (default=True) intercepta roc_auc_score e
#      classification_report chamados após o fit e loga com nomes automáticos
#      (ex.: "roc_auc_score_X_test"), duplicando as chaves que logamos à mão e
#      tornando o painel "Compare" ilegível.
# Logging manual dá chaves determinísticas e comparáveis entre os 3 modelos.
mlflow.sklearn.autolog(disable=True)

# ─── Tags do PROJETO (nível de experimento, não de run) ──────────────────────
# mlflow.set_tags() só é válido dentro de um run ativo. Para marcar o experimento inteiro usa-se set_experiment_tags().
TAGS_PROJETO = {
    "projeto": "Chopp & Cia",
    "instituicao": "FATEC Votorantim",
    "disciplina": "Projeto Integrador VI",
    "tipo_modelo": "classificacao_binaria",
    "framework": "scikit-learn",
    "ambiente": "databricks-free-serverless",
    "unidade_analise": "ID_PESSOA",
}
try:
    mlflow.set_experiment_tags(TAGS_PROJETO)
except Exception as e:
    print(f"Tags de experimento não aplicadas ({type(e).__name__}) — seguirão como tags de run.")

# ─── Convenção de Nomes de Métricas ───────────────────────────────────────────
# Chaves padronizadas para que o botão "Compare" do Databricks alinhe os runs.
PREFIXO_CV, PREFIXO_TESTE = "cv", "test"

# ─── Registro da limitação metodológica conhecida ────────────────────────────
# Documentado explicitamente para auditoria
LIMITACOES_CONHECIDAS = {
    "sem_corte_temporal": (
        "Features e variável-alvo são calculadas sobre a MESMA janela temporal. O desenho correto exigiria features até uma data de corte e alvo observado após ela. Não aplicado devido ao tamanho da amostra (476 clientes)."
    ),
    "features_derivadas_do_alvo": (
        "MEDIA_DIAS_ATRASO_PAG/COM e TOTAL_PARCELAS/COMODATOS compartilham origem com o alvo ALTO_RISCO. As métricas devem ser lidas como capacidade de REPRODUZIR a regra de negócio, não como poder preditivo prospectivo."
    ),
    "classe_positiva_majoritaria": (
        "86,8 porcento dos clientes são ALTO_RISCO. Acurácia e AUC isoladas são enganosas; a comparação deve ser sempre contra o baseline DummyClassifier."
    ),
}

print("✅ MLflow configurado com sucesso!")
print(f"   ├─ Experimento : {EXPERIMENT_NAME}")
print(f"   ├─ Experiment ID: {experiment.experiment_id}")
print(f"   ├─ Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   ├─ Autolog     : DESABILITADO (logging manual controlado)")
print(f"   └─ Artefatos   : MLflow (sem dependência de disco local efêmero)")
print(f"\n   ⚠️  {len(LIMITACOES_CONHECIDAS)} limitações metodológicas registradas para auditoria.")


# 🎯 Fase 1 - Entendimento do Negócio

### 📌 Contexto do Projeto e do Negócio

O objeto de estudo é uma **distribuidora de chopp** localizada estrategicamente na cidade de **Ponta Porã (MS)**, na fronteira entre o Brasil e o Paraguai. 

Devido à sua localização transfronteiriça e ao modelo de negócio baseado em recorrência e empréstimo de equipamentos (comodatos), a empresa enfrenta desafios singulares em relação ao comportamento de compra, perfil de cliente e gestão de inadimplência. 

Nosso objetivo principal é utilizar técnicas de Ciência de Dados e Machine Learning para analisar esse histórico, identificar padrões de risco e auxiliar a tomada de decisão comercial da distribuidora.

### Hipótese Central

> Clientes com histórico de atrasos (financeiro **e/ou** comodato) podem ser
> identificados preventivamente com base no perfil + comportamento.

### KPIs de Sucesso

| Métrica         |  Meta  | Justificativa                           |
| :-------------- | :----: | :-------------------------------------- |
| AUC-ROC         | ≥ 0.75 | Discriminação geral                     |
| Recall classe 1 | ≥ 0.70 | **Minimizar falsos negativos (custo alto)** |
| Accuracy        | ≥ 0.65 | Baseline razoável para dataset pequeno  |

### Separação de Datasets

| Dataset          | Propósito                                                                                      |
| :--------------- | :--------------------------------------------------------------------------------------------- |
| `df_ml_completo` | Todos os clientes com features calculadas — usado como "tabela de usuários" para lookup por ID |
| `df_treino`      | Subconjunto filtrado (veteranos) — vai para `X_train` / `X_test` do modelo                     |


> ### ⚠️ Nota Metodológica (revisão de QA)
>
> A tabela de KPIs acima foi definida **antes** da análise da distribuição real da variável-alvo. Após a construção do dataset, verificou-se que **86,8% dos clientes elegíveis são ALTO_RISCO**. Isso tem duas consequências que precisam ficar registradas:
>
> 1. **As metas de Accuracy (≥ 0,65) e Recall (≥ 0,70) são atingidas por um classificador constante** que responde "alto risco" para todos, sem olhar dado algum — esse baseline entrega Accuracy 0,868 e Recall 1,000. Metas abaixo do baseline não medem valor do modelo.
> 2. Por isso a Fase 5 passa a registrar um **`DummyClassifier` como run de referência no MLflow**, e a seleção do modelo campeão usa **MCC** e **Balanced Accuracy** — métricas que só sobem quando *ambas* as classes são bem classificadas.
>
> **Limitação conhecida e assumida:** as features `MEDIA_DIAS_ATRASO_PAG` e `MEDIA_DIAS_ATRASO_COM` compartilham origem aritmética com a variável-alvo, e não há corte temporal separando a janela das features da janela do alvo. As métricas devem portanto ser lidas como **capacidade de reproduzir a regra de negócio já conhecida**, não como poder preditivo prospectivo. Essa limitação é registrada como artefato em todos os runs (`limitacoes_metodologicas.json`).

# 💾 Fase 2 - Dados

Para a realização deste estudo, foi concedido acesso direto ao banco de dados relacional (SQL Server) do sistema de gestão da empresa (`DB_POWER_SYS`).

A seguir, iniciamos o processo de extração dos dados. O script de código abaixo estabelece a conexão segura com o banco de dados e realiza um mapeamento inteligente. Para otimizar o processamento e a memória, o algoritmo executa uma limpeza prévia automática: varre todas as tabelas, identifica as que contêm dados úteis e expurga colunas completamente vazias (100% nulas), entregando um relatório gerencial sobre a saúde inicial da nossa base de dados.

--- 

> Comente os blocos destacados se for usar CSVs já exportados

In [0]:
# # ─── FASE 2A: Extração via SQL Server
# # Comente este bloco inteiro se for usar CSVs já exportados

# from sqlalchemy import create_engine, text
# from urllib.parse import quote_plus
# from IPython.display import display, Markdown

# # 1. Configuração da Conexão
# # params = quote_plus(
# #     r"Driver={ODBC Driver 17 for SQL Server};"
# #     r"Server=.\SQLEXPRESS;"
# #     r"Database=DB_POWER_SYS;"
# #     r"Trusted_Connection=yes;"
# #     r"Encrypt=yes;"
# #     r"TrustServerCertificate=yes;"
# # )

# params = quote_plus(
#     r"Driver={ODBC Driver 17 for SQL Server};Server=localhost;"
#     r"Database=DB_POWER_SYS;Trusted_Connection=yes;"
#     r"Encrypt=yes;TrustServerCertificate=yes;"
# )

# engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# # Containers para o relatório e contadores globais
# tabelas_db = {}
# dados_relatorio = []
# total_vazias = 0
# total_colunas_ativas = 0
# total_colunas_removidas = 0

# try:
#     query_tabelas = text(
#         "SELECT TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE'"
#     )

#     with engine.connect() as conn:
#         lista_tabelas = pd.read_sql(query_tabelas, conn)

#         for _, linha in lista_tabelas.iterrows():
#             nome_tab = linha["TABLE_NAME"]
#             schema = linha["TABLE_SCHEMA"]
#             tabela_full = f"[{schema}].[{nome_tab}]"

#             try:
#                 # Carregamento dos dados
#                 df = pd.read_sql(text(f"SELECT * FROM {tabela_full}"), conn)

#                 if df.empty:
#                     total_vazias += 1
#                 else:
#                     cols_originais = len(df.columns)

#                     # Limpeza: remove colunas onde todas as linhas são nulas
#                     df_limpo = df.dropna(axis=1, how="all")
#                     cols_ativas = len(df_limpo.columns)
#                     cols_removidas = cols_originais - cols_ativas

#                     # Somatório para o resumo geral
#                     total_colunas_ativas += cols_ativas
#                     total_colunas_removidas += cols_removidas

#                     # Armazenamento
#                     tabelas_db[nome_tab] = df_limpo

#                     # Registra dados para o Markdown
#                     dados_relatorio.append(
#                         {
#                             "Tabela": f"`{nome_tab}`",
#                             "Linhas": len(df_limpo),
#                             "Colunas Originais": cols_originais,
#                             "Colunas Removidas": cols_removidas,
#                             "Colunas Ativas": cols_ativas,
#                         }
#                     )

#             except Exception as e:
#                 print(f"Erro na tabela {nome_tab}: {e}")

#     # ==============================================================================
#     # GERAÇÃO DO RELATÓRIO EM MARKDOWN
#     # ==============================================================================
#     df_resumo = pd.DataFrame(dados_relatorio)

#     relatorio = f"""
# ### 📊 Relatório de Mapeamento e Limpeza de Dados
# Abaixo estão listadas as tabelas identificadas no banco de dados **DB_POWER_SYS** que possuem registros ativos. 

# > **Nota Metodológica:** Para otimização de memória e performance do modelo de Machine Learning, todas as colunas que possuíam 100% de valores nulos (vazios) foram descartadas automaticamente durante a ingestão dos dados.

# | Tabela | Registros | Colunas Originais | Colunas Removidas | Colunas Ativas |
# | :--- | :---: | :---: | :---: | :---: |
# """
#     # Adicionando as linhas da tabela detalhada no Markdown
#     for _, r in df_resumo.iterrows():
#         relatorio += f"| {r['Tabela']} | {r['Linhas']} | {r['Colunas Originais']} | **{r['Colunas Removidas']}** | {r['Colunas Ativas']} |\n"

#     # Adicionando a Tabela de Resumo Geral
#     relatorio += f"""
# ### 📌 Resumo Geral do Banco de Dados
# | Métrica | Quantidade |
# | :--- | :---: |
# | Tabelas com dados carregadas | **{len(tabelas_db)}** |
# | Tabelas vazias ignoradas | **{total_vazias}** |
# | Total de Colunas Ativas na Base | **{total_colunas_ativas}** |
# | Total de Colunas Vazias Removidas | **{total_colunas_removidas}** |
# #
# > **💡 Dica de Uso:** Para acessar e trabalhar com qualquer uma das tabelas importadas nas próximas células do Notebook, utilize a estrutura `tabelas_db['NOME_DA_TABELA']` (Exemplo: `tabelas_db['TB_PRODUTO']`).
# """

#     # Exibe o visual
#     display(Markdown(relatorio))

# except Exception as e:
#     display(Markdown(f"## ❌ Erro Crítico na Conexão\n{e}"))

#### ⚙️ Engenharia de Atributos: Criação de Views Estratégicas
Para otimizar a extração de dados e garantir que o modelo de Machine Learning receba as informações já pré-processadas e focadas no objetivo do negócio, três visões (`Views`) foram construídas diretamente no banco de dados SQL Server da distribuidora. 

O objetivo dessas views é isolar os dados transacionais e facilitar o cálculo das métricas que compõem o perfil de risco e comportamento do cliente. A seguir, detalhamos a estrutura e a finalidade de cada uma:

#### 1. View de Vendas (`vw_calculo_score_vendas`)
**Objetivo:** Mapear a recorrência de compras, o volume financeiro gerado e o engajamento do cliente com os produtos. Esta base filtra apenas pedidos ativos e consolidados (Status 11).
* **`ID_PESSOA` / `ID_PRODUTO`:** Identificadores fundamentais do cliente e do item consumido.
* **`QTD_VENDA` e `QTD_DEVOLUCAO`:** Medem o volume bruto consumido e se há um alto índice de devolução de produtos por parte daquele cliente.
* **`VL_FINANCEIRO`:** A receita líquida/bruta gerada pelo item, base para cálculo do Ticket Médio.
* **`DT_PEDIDO` e `DT_ACERTO`:** Permitem medir a frequência de compra (tempo entre um pedido e outro) e o ciclo de vida do cliente (Análise RFM - Recência, Frequência e Valor Monetário).

#### 2. View de Comodato (`vw_calculo_score_comodato`)
**Objetivo:** Avaliar o risco logístico e o compromisso do cliente com os equipamentos (chopeiras) emprestados. Filtramos apenas comodatos encerrados (Status 24) para analisar o histórico real de devolução.
* **`ID_COMODATO` / `ID_PEDIDO` / `ID_CLIENTE`:** Chaves de relacionamento para cruzar o empréstimo da máquina com a venda do chopp.
* **`ID_PRODUTO` e `QTD_PRODUTO`:** Identifica qual equipamento foi cedido e em qual quantidade.
* **`DT_EMPRESTIMO`:** Data inicial da cessão do equipamento.
* **`DT_VENCIMENTO` vs. `DT_RECOLHE`:** O "coração" desta view. O cruzamento destas duas datas permite ao modelo calcular matematicamente os **Dias de Atraso** na devolução. Uma data de recolhimento superior ao vencimento sinaliza um desvio de conduta e aumenta o risco do cliente.

#### 3. View Financeira (`vw_calculo_score_financeiro`)
**Objetivo:** Mensurar o risco de crédito (inadimplência) e a pontualidade nos pagamentos. A view consolida as contas a receber já baixadas (Status 17) que possuem vínculo com um pedido.
* **`ID_PESSOA` / `ID_PEDIDO`:** Relacionamento com o cliente e a venda que gerou a cobrança.
* **`NR_PARCELA` e `VL_PARCELA`:** Identifica se a compra foi fracionada e qual o valor do compromisso financeiro.
* **`DT_VENCIMENTO` vs. `DT_RECEBIMENTO`:** Assim como no comodato, a diferença entre essas datas gera a variável de **Atraso Financeiro**. 
* **`VL_RECEBIDO`:** Usado para cruzar com o valor da parcela e identificar se houve pagamento parcial, descontos ou cobrança de juros/multas.
* **`DT_BAIXA` e `TP_BAIXA`:** Dados de auditoria que indicam quando e como o título foi efetivamente liquidado no sistema.

---

> As principais tabelas de suporte identificadas assim como essas Visões criadas foram exportadas no formato CSV para facilitar a replicação do uso deste Pipeline sem que seja necessário fazer a conexão direta com o banco de dados

In [0]:
# # ─── FASE 2A: Extração via SQL Server
# # Comente este bloco inteiro se for usar CSVs já exportados

# df_vendas = pd.read_sql("SELECT * FROM dbo.vw_calculo_score_vendas", engine)
# df_financeiro = pd.read_sql("SELECT * FROM dbo.vw_calculo_score_financeiro", engine)
# df_comodato = pd.read_sql("SELECT * FROM dbo.vw_calculo_score_comodato", engine)

# tabelas_dim = [
#     "TB_CLIENTE",
#     "TB_PESSOA",
#     "TB_PRODUTO",
#     "TB_PEDIDO_ITEM",
#     "TB_CLIENTE_ENDERECO",
#     "TB_FORMA_PAGTO",
#     "TB_TIPO_ESTABELECIMENTO",
# ]
# tabelas_db = {t: pd.read_sql(f"SELECT * FROM dbo.{t}", engine) for t in tabelas_dim}

# # Export para CSV (persistência)
# for nome_csv, df_exp in [
#     ("score_vendas", df_vendas),
#     ("score_financeiro", df_financeiro),
#     ("score_comodato", df_comodato),
# ]:
#     df_exp.to_csv(f"{nome_csv}.csv", index=False, sep=";", encoding="utf-8-sig")

# for nome, df_tab in tabelas_db.items():
#     df_tab.to_csv(f"{nome}.csv", index=False, sep=";", encoding="utf-8-sig")

# print("✅ Dados extraídos do SQL e exportados para CSV!")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FASE 2B — CARGA VIA CSV  ·  DESATIVADA
#
#  A origem oficial dos dados passou a ser a tabela do Unity Catalog, lida na
#  Fase 2C (célula abaixo). Este bloco fica como alternativa para rodar o
#  notebook FORA do Databricks, onde não há `spark`.
#
#  Para usar: descomente este bloco e comente a Fase 2C. As duas células
#  produzem o MESMO df_clientes — nunca deixe as duas ativas ao mesmo tempo,
#  ou a segunda sobrescreve a primeira sem aviso.
# ══════════════════════════════════════════════════════════════════════════════

# # ─── FASE 2B: Carga do Dataset Consolidado ────────────────────────────────────
# #
# #  A extração (Extracao_Dados_Consolidados.ipynb) entrega UM arquivo:
# #
# #      dataset_consolidado.csv  —  1 linha por CLIENTE (ID_PESSOA), 38 colunas
# #
# #  Antes eram 10 CSVs (3 views transacionais + 7 dimensões) que este notebook
# #  cruzava e agregava por conta própria. A agregação foi movida para a extração,
# #  onde é auditada: lá a soma do faturamento por cliente é conferida contra a
# #  soma das linhas de origem, o que impede a duplicação silenciosa que um join
# #  em cadeia entre itens, parcelas e contratos de comodato produz.
# #
# #  Consequência para este notebook: a Fase 3 (EDA) e a Fase 4 trabalham no
# #  mesmo grão — o do cliente. Não há mais reagregação aqui.
# # ──────────────────────────────────────────────────────────────────────────────
#
# DATA_DIR = '/Volumes/projetointegrador/default/projetointegrador'
# ARQUIVO_DATASET = "dataset_consolidado.csv"
#
# LEITURA_PARAMS: Dict[str, Any] = {
#     "sep": ";",
#     "encoding": "utf-8-sig",
#     "low_memory": False,
# }
#
# print("📂 Carregando o dataset consolidado...")
# print(f"   └─ {os.path.join(DATA_DIR, ARQUIVO_DATASET)}\n")
#
# df_clientes = pd.read_csv(os.path.join(DATA_DIR, ARQUIVO_DATASET), **LEITURA_PARAMS)
#
# # ── Conversão de tipos ────────────────────────────────────────────────────────
# # CSV não carrega schema: sem isto TOTAL_GASTO chega como string e `.sum()`
# # concatena texto em vez de somar.
# for _col in ["PRIMEIRA_COMPRA", "ULTIMA_COMPRA"]:
#     if _col in df_clientes.columns:
#         df_clientes[_col] = pd.to_datetime(df_clientes[_col], errors="coerce")
#
# _COLS_NUM = [
#     "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA", "FREQUENCIA_COMPRAS",
#     "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
#     "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
#     "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS",
#     "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
#     "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS",
#     "RISCO_FINANCEIRO", "RISCO_COMODATO", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
# ]
# for _col in _COLS_NUM:
#     if _col in df_clientes.columns:
#         df_clientes[_col] = pd.to_numeric(df_clientes[_col], errors="coerce").fillna(0)
#
# # Aging é categórica ORDINAL: sem a ordem explícita, os gráficos saem em ordem
# # alfabética e "1-3 Dias" aparece depois de "+30 Dias".
# ORDEM_AGING = ["Sem Atraso", "1-3 Dias", "4-7 Dias", "8-15 Dias",
#                "16-20 Dias", "21-30 Dias", "+30 Dias"]
# for _col in ["AGING_PAGAMENTO", "AGING_COMODATO"]:
#     if _col in df_clientes.columns:
#         df_clientes[_col] = pd.Categorical(
#             df_clientes[_col], categories=ORDEM_AGING, ordered=True
#         )
#
# # ── Verificação do contrato com a extração ───────────────────────────────────
# # Falha aqui é muito mais barata de diagnosticar do que uma coluna faltando
# # 30 células adiante, no meio da engenharia de atributos.
# _ESPERADAS = [
#     "ID_PESSOA", "NOME_CLIENTE", "PERFIL", "CIDADE", "PAGAMENTO",
#     "FREQUENCIA_COMPRAS", "TICKET_MEDIO", "TOTAL_GASTO",
#     "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
#     "TOTAL_PARCELAS", "TOTAL_COMODATOS",
#     "MEDIA_DIAS_ATRASO_PAG", "MEDIA_DIAS_ATRASO_COM",
#     "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",
# ]
# _ausentes = [c for c in _ESPERADAS if c not in df_clientes.columns]
# if _ausentes:
#     raise KeyError(
#         f"Colunas ausentes em {ARQUIVO_DATASET}: {_ausentes}. "
#         f"Reexecute Extracao_Dados_Consolidados.ipynb — o contrato mudou."
#     )
#
# if df_clientes["ID_PESSOA"].duplicated().any():
#     raise ValueError(
#         f"{ARQUIVO_DATASET} tem ID_PESSOA duplicado — deveria ser 1 linha por "
#         f"cliente. A agregação da extração falhou."
#     )
#
# # Categórica que virou constante é o sintoma do bug do prefixo N' (literais
# # Unicode do SQL Server chegando como "N'F'" em vez de "F"). O modelo treina
# # sem erro e a feature simplesmente não informa nada.
# for _cat in ["PERFIL", "CIDADE", "PAGAMENTO"]:
#     if df_clientes[_cat].nunique() <= 1:
#         print(f"   ⚠️  {_cat} tem um único valor — feature sem poder discriminante. "
#               f"Verifique _limpar() na extração.")
#
# print(f"   ✅ {df_clientes.shape[0]:,} clientes × {df_clientes.shape[1]} colunas\n")
#
# print(f"📊 Cobertura por trilha:")
# for _flag, _rotulo in [("TEM_VENDAS", "vendas"), ("TEM_FINANCEIRO", "financeiro"),
#                        ("TEM_COMODATO", "comodato")]:
#     if _flag in df_clientes.columns:
#         _n = int(df_clientes[_flag].sum())
#         print(f"   ├─ com {_rotulo:<12}: {_n:>6,} clientes ({_n/len(df_clientes)*100:>5.1f}%)")
#
# print(f"\n📅 Janela: {df_clientes['PRIMEIRA_COMPRA'].min():%d/%m/%Y} → "
#       f"{df_clientes['ULTIMA_COMPRA'].max():%d/%m/%Y}")
# print(f"💰 Faturamento total: R$ {df_clientes['TOTAL_GASTO'].sum():,.2f}")
# print(f"\n   Categóricas: PERFIL {df_clientes['PERFIL'].nunique()} valores | "
#       f"CIDADE {df_clientes['CIDADE'].nunique()} | PAGAMENTO {df_clientes['PAGAMENTO'].nunique()}")
#
# display(df_clientes.head())

### 🗄️ Fase 2C — Carga via Unity Catalog

O dataset consolidado foi publicado como **tabela gerenciada** no Unity Catalog:

```
projetointegrador.projetointegrador.dataset
```

Ler da tabela em vez do CSV traz duas coisas que importam aqui:

| | CSV no Volume | Tabela no Unity Catalog |
| :--- | :--- | :--- |
| **Schema** | perdido — tudo volta como `string` | preservado: `TOTAL_GASTO` é `double`, datas são `timestamp` |
| **Versionamento** | sobrescrito a cada carga | Delta versiona; dá para auditar o dado exato de uma rodada passada |
| **Acesso** | só por caminho de arquivo | consultável por SQL, governado por permissões do catálogo |

A conversão de tipos permanece na célula mesmo com o schema vindo pronto: a Fase 2C precisa produzir **exatamente o mesmo `df_clientes`** que a 2B produzia a partir do CSV. Sem isso, as duas origens divergiriam em silêncio e o resultado dependeria de qual célula foi executada.

> A **Fase 2B** (célula acima) fica comentada como alternativa para rodar o notebook fora do Databricks, onde não existe `spark`. Nunca deixe as duas ativas ao mesmo tempo.

In [ ]:
# ─── FASE 2C: Carga do Dataset Consolidado (Unity Catalog) ────────────────────
#
#  Lê a tabela gerenciada do Unity Catalog em vez do CSV. Duas vantagens que
#  importam para este projeto:
#
#   · A tabela carrega o SCHEMA. TOTAL_GASTO volta como double e as datas como
#     timestamp, sem a reconversão manual que o CSV exige — e que é fonte
#     silenciosa de divergência entre execuções.
#   · O CSV vivia num Volume; a tabela é versionada (Delta) e consultável por
#     SQL, o que permite auditar o dado exato de uma rodada passada.
#
#  A Fase 2B (leitura do CSV) fica logo acima, comentada, como alternativa para
#  rodar fora do Databricks.
# ──────────────────────────────────────────────────────────────────────────────

CATALOGO = "projetointegrador"
SCHEMA = "projetointegrador"
TABELA = "dataset"

TABELA_FULL = f"{CATALOGO}.{SCHEMA}.{TABELA}"

print("📂 Carregando o dataset consolidado do Unity Catalog...")
print(f"   └─ {TABELA_FULL}\n")

# .toPandas() traz tudo para o driver. Aceitável e proposital aqui: são ~1.5k
# linhas × 39 colunas, e o restante do notebook (sklearn, matplotlib) é pandas.
df_clientes = spark.read.table(TABELA_FULL).toPandas()

# ── Normalização de tipos ─────────────────────────────────────────────────────
# A tabela já traz o schema correto, mas normalizamos assim mesmo: a célula
# precisa produzir o mesmo resultado tendo vindo da tabela OU do CSV comentado
# acima, senão as duas origens divergem em silêncio.
for _col in ["PRIMEIRA_COMPRA", "ULTIMA_COMPRA"]:
    if _col in df_clientes.columns:
        df_clientes[_col] = pd.to_datetime(df_clientes[_col], errors="coerce")

_COLS_NUM = [
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA", "FREQUENCIA_COMPRAS",
    "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS",
    "RISCO_FINANCEIRO", "RISCO_COMODATO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
]
for _col in _COLS_NUM:
    if _col in df_clientes.columns:
        df_clientes[_col] = pd.to_numeric(df_clientes[_col], errors="coerce").fillna(0)

# Aging é categórica ORDINAL: sem declarar a ordem, os gráficos saem em ordem
# alfabética e "1-3 Dias" aparece depois de "+30 Dias".
for _col in ["AGING_PAGAMENTO", "AGING_COMODATO"]:
    if _col in df_clientes.columns:
        df_clientes[_col] = pd.Categorical(
            df_clientes[_col], categories=ORDEM_AGING, ordered=True
        )

# ── Verificação do contrato com a extração ───────────────────────────────────
# Falha aqui é muito mais barata de diagnosticar do que uma coluna faltando
# 30 células adiante, no meio da engenharia de atributos.
_ESPERADAS = [
    "ID_PESSOA", "NOME_CLIENTE", "PERFIL", "CIDADE", "PAGAMENTO",
    "FREQUENCIA_COMPRAS", "TICKET_MEDIO", "TOTAL_GASTO",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "TOTAL_PARCELAS", "TOTAL_COMODATOS",
    "MEDIA_DIAS_ATRASO_PAG", "MEDIA_DIAS_ATRASO_COM",
    "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO", "CORE_BUSINESS",
]
_ausentes = [c for c in _ESPERADAS if c not in df_clientes.columns]
if _ausentes:
    raise KeyError(
        f"Colunas ausentes em {TABELA_FULL}: {_ausentes}. "
        f"Recarregue a tabela a partir do dataset_consolidado.csv gerado por "
        f"Extracao_Dados_Consolidados.ipynb — o contrato mudou."
    )

if df_clientes["ID_PESSOA"].duplicated().any():
    raise ValueError(
        f"{TABELA_FULL} tem ID_PESSOA duplicado — deveria ser 1 linha por cliente. "
        f"Provável append onde deveria ter sido overwrite ao recarregar a tabela."
    )

# Categórica que virou constante é o sintoma do bug do prefixo N' (literais
# Unicode do SQL Server chegando como "N'F'" em vez de "F"). O modelo treina
# sem erro e a feature simplesmente não informa nada.
for _cat in ["PERFIL", "CIDADE", "PAGAMENTO"]:
    if df_clientes[_cat].nunique() <= 1:
        print(f"   ⚠️  {_cat} tem um único valor — feature sem poder discriminante. "
              f"Verifique _limpar() na extração.")

print(f"   ✅ {df_clientes.shape[0]:,} clientes × {df_clientes.shape[1]} colunas\n")

print(f"📊 Cobertura por trilha:")
for _flag, _rotulo in [("TEM_VENDAS", "vendas"), ("TEM_FINANCEIRO", "financeiro"),
                       ("TEM_COMODATO", "comodato"), ("CORE_BUSINESS", "core business")]:
    if _flag in df_clientes.columns:
        _n = int(df_clientes[_flag].sum())
        print(f"   ├─ {_rotulo:<14}: {_n:>6,} clientes ({_n/len(df_clientes)*100:>5.1f}%)")

print(f"\n📅 Janela: {df_clientes['PRIMEIRA_COMPRA'].min():%d/%m/%Y} → "
      f"{df_clientes['ULTIMA_COMPRA'].max():%d/%m/%Y}")
print(f"💰 Faturamento total: R$ {df_clientes['TOTAL_GASTO'].sum():,.2f}")
print(f"\n   Categóricas: PERFIL {df_clientes['PERFIL'].nunique()} valores | "
      f"CIDADE {df_clientes['CIDADE'].nunique()} | PAGAMENTO {df_clientes['PAGAMENTO'].nunique()}")

display(df_clientes.head())

# 🔍 Fase 3 - Exploração e Qualidade dos Dados (EDA)

Nesta fase, realizamos um mergulho profundo no histórico transacional e cadastral para compreender a dinâmica de faturamento, o comportamento volumétrico e, principalmente, os padrões ocultos de risco de inadimplência no comodato de chopp. A **Análise Exploratória de Dados (EDA)** não é apenas uma etapa visual, mas uma auditoria estatística vital para extrair inteligência de negócio e validar as hipóteses levantadas na Fase 1.

Análises desta fase:

- **3.1** Auditoria de integridade relacional
- **3.2** Perfil 360° do cliente (identidade × localização × pagamento)
- **3.3** Análise de risco por segmento (financeiro vs. comodato)
- **3.4** Análise de aging (severidade do atraso)
- **3.5** Série histórica (R$ inadimplente vs. unidades retidas)
- **3.6** Radiografia do Risco Integrado por Cliente


## 3.1 🔗 Auditoria de integridade relacional

Antes de partirmos para as visualizações gráficas globais, realizamos uma análise focada em ranqueamento para identificar os nossos **Clientes VIPs**. 

O objetivo desta etapa é extrair informação direta da base através da consolidação das métricas financeiras cruzadas com os dados cadastrais. Para cada cliente, o algoritmo a seguir calcula:
* **Total Comprado:** Somatório do valor financeiro de todas as compras. (Métrica principal de ranqueamento).
* **Ticket Médio:** A relação entre o Faturamento Total e a Quantidade de Pedidos Únicos daquele cliente.
* **Produto Mais Comprado:** Identificação do item favorito do cliente (baseado no volume de unidades `QTD_VENDA` consumidas historicamente).
* **Perfil:** O Nome Fantasia e o Segmento do Estabelecimento a que pertence.

In [ ]:
# ─── 3.1 Auditoria de Integridade Relacional ─────────────────────────────────
# O ranking de clientes VIP agora nasce pronto: df_clientes já chega com uma
# linha por ID_PESSOA e as métricas de RFM consolidadas na extração. Não há mais
# groupby/merge aqui porque o grão transacional não é mais carregado.

top_10_clientes = df_clientes.sort_values("TOTAL_GASTO", ascending=False).head(10)

# Mantemos exatamente as mesmas colunas do relatório original. SEGMENTO ocupa o
# lugar de DS_TIPO_ESTABELECIMENTO e PRODUTO_FAVORITO já vem resolvido pelo nome.
colunas_exibicao = [
    "DS_FANTASIA",
    "SEGMENTO",
    "PRODUTO_FAVORITO",
    "FREQUENCIA_COMPRAS",
    "TOTAL_GASTO",
    "TICKET_MEDIO",
]
top_10_clientes = top_10_clientes[colunas_exibicao].copy()

# Blindagem contra cadastro incompleto — a origem pode trazer fantasia/segmento nulos.
top_10_clientes["DS_FANTASIA"] = top_10_clientes["DS_FANTASIA"].fillna("NÃO INFORMADO")
top_10_clientes["SEGMENTO"] = top_10_clientes["SEGMENTO"].fillna("OUTROS")
top_10_clientes["PRODUTO_FAVORITO"] = top_10_clientes["PRODUTO_FAVORITO"].fillna("N/D")

top_10_clientes.columns = [
    "Nome do Cliente",
    "Segmento",
    "Produto Mais Comprado",
    "Total de Pedidos",
    "Total Comprado (R$)",
    "Ticket Médio (R$)",
]

display(
    top_10_clientes.style.format(
        {"Total Comprado (R$)": "R$ {:,.2f}", "Ticket Médio (R$)": "R$ {:,.2f}"}
    )
    .hide(axis="index")
    .set_caption("3.1 - Top 10 Clientes por Volume de Compras")
)

### 💡 Conclusões e Ações Futuras (Análise de Top Clientes)

A visualização do ranqueamento dos maiores clientes nos permitiu identificar rapidamente oportunidades de melhoria na qualidade dos dados. A partir desta primeira análise, tiramos duas conclusões cruciais que guiarão as próximas etapas de *Engenharia de Atributos*:

1. **Inconsistências na Segmentação Cadastral:** Ao cruzar o Nome Fantasia (`DS_FANTASIA`) com o Segmento (`DS_TIPO_ESTABELECIMENTO`), fica evidente que o cadastro no sistema muitas vezes não reflete a realidade do cliente (ex: estabelecimentos comerciais classificados erroneamente como "Consumidor Final"). Para que o modelo de Machine Learning não aprenda padrões incorretos, será necessário realizar um tratamento/higienização focado em corrigir esses rótulos, ou simplesmente descartá-los da análise.

2. **Ruído no Mix de Produtos:** Observamos que itens de apoio, como "Copos Descartáveis", aparecem no topo do ranking de produtos mais comprados por grandes clientes. Como o escopo central do nosso modelo preditivo é o comportamento atrelado ao consumo de **Chopp** e ao risco de comodato de **Chopeiras**, a presença desses produtos periféricos gera "ruído" financeiro e transacional. 
   
> **🎯 Próxima Ação:** Fazer um levantamento da base de produtos (`TB_PRODUTO`) e aplicar um filtro definitivo nas tabelas de vendas, retirando os itens que não sejam pertinentes ao foco principal da análise (Chopp e equipamentos relacionados).

In [ ]:
# ─── Filtro Core Business: Chopp e Chopeiras ──────────────────────────────────
# ⚠️ LIMITAÇÃO DO NOVO GRÃO: o recorte por ID_PRODUTO (regex CHOPP|CHOPEIRA|
# BARRIL|...) exigia a linha de item de pedido. Como df_clientes já chega
# agregado por cliente, esse filtro deixou de ser possível NESTE ponto do
# notebook — ele passou a ser aplicado na origem, durante a extração que gera o
# dataset consolidado. O que se perdeu aqui é a auditoria do impacto do filtro
# (quantas linhas sobreviveram por trilha); o que permanece é o resultado dele.

print(f"{'='*55}")
print(f"{'📋 3.1 ESCOPO CORE BUSINESS (aplicado na extração)':^55}")
print(f"{'='*55}")
print("  ℹ️  O recorte de produtos core (chopp, chopeiras, barris, cilindros)")
print("      é feito na consulta que materializa o dataset consolidado.")
print("      df_clientes já representa somente a carteira do core business.\n")

print(f"  👥 Clientes na base          : {fmt_numero(len(df_clientes)):>12}")
print(f"  🧾 Pedidos (soma frequência) : {fmt_numero(df_clientes['FREQUENCIA_COMPRAS'].sum()):>12}")
print(f"  📦 Itens vendidos            : {fmt_numero(df_clientes['TOTAL_ITENS'].sum()):>12}")
print(f"  🍺 Unidades vendidas         : {fmt_numero(df_clientes['QTD_TOTAL_VENDIDA'].sum()):>12}")
print(f"  💰 Faturamento Core Total    : {fmt_moeda(df_clientes['TOTAL_GASTO'].sum()):>12}")
print(f"  🎯 Ticket médio da carteira  : {fmt_moeda(df_clientes['TICKET_MEDIO'].mean()):>12}")

In [ ]:
# ─── 3.1 Auditoria de Integridade Relacional ─────────────────────────────────
# A auditoria de chaves entre views saiu de cena (não há mais ID_PEDIDO aqui).
# O que sobrevive — e é o que de fato alimenta o modelo — é o pré-diagnóstico
# das taxas de atraso, agora recomposto a partir dos contadores por cliente.
print("=" * 60)
print(f"{'📊 3.1 - AUDITORIA DA BASE CONSOLIDADA':^60}")
print("=" * 60)

n_clientes = len(df_clientes)
n_com_vendas = int(df_clientes["TEM_VENDAS"].sum())
n_com_fin = int(df_clientes["TEM_FINANCEIRO"].sum())
n_com_com = int(df_clientes["TEM_COMODATO"].sum())

print(f"\n👥 Clientes únicos (ID_PESSOA)     : {n_clientes:>6,}")
print(f"📦 Clientes com histórico de vendas: {n_com_vendas:>6,}")
print(f"💰 Clientes com trilha financeira  : {n_com_fin:>6,}")
print(f"🛢️  Clientes com comodato ativo     : {n_com_com:>6,}")

# Taxas ponderadas pelo volume: somamos numerador e denominador de toda a
# carteira em vez de tirar média das taxas individuais — assim um cliente com
# 1 parcela não pesa o mesmo que um com 200.
tot_parcelas = df_clientes["TOTAL_PARCELAS"].sum()
tot_parc_atraso = df_clientes["PARCELAS_ATRASADAS"].sum()
tot_comodatos = df_clientes["TOTAL_COMODATOS"].sum()
tot_com_atraso = df_clientes["COMODATOS_ATRASADOS"].sum()

pct_fin = (tot_parc_atraso / tot_parcelas * 100) if tot_parcelas > 0 else 0
pct_com = (tot_com_atraso / tot_comodatos * 100) if tot_comodatos > 0 else 0

print(f"\n{'='*60}")
print(f"{'⚠️  PRÉ-DIAGNÓSTICO DE ATRASOS (base completa)':^60}")
print(f"{'='*60}")
print(f"   🔴 Taxa de atraso Financeiro: {pct_fin:.1f}%  ({tot_parc_atraso:,.0f} de {tot_parcelas:,.0f} parcelas)")
print(f"   🟠 Taxa de atraso Comodato  : {pct_com:.1f}%  ({tot_com_atraso:,.0f} de {tot_comodatos:,.0f} comodatos)")
print(f"\n   📅 Última compra registrada : {df_clientes['ULTIMA_COMPRA'].max()}")

### 💡 Conclusão do Diagnóstico de Pedidos e Dinâmica de Negócio

A auditoria da granularidade dos pedidos nos revelou informações valiosas tanto sobre a saúde do banco de dados quanto sobre a dinâmica da operação da distribuidora. Destacamos três pontos cruciais:

1. **A Base de Fatos (Vendas):** O nosso universo de análise validado (Core Business) é composto por exatos **4.288 pedidos únicos**. Este é o número oficial que guiará as conversões.

2. **🚨 Alerta de Integridade (Gap Financeiro):** Ao cruzarmos com a tabela do Financeiro, localizamos apenas **4.240 pedidos**. Isso significa que **48 pedidos de venda oficializados estão sem nenhum registro de cobrança ou parcela atrelada**. 
   * *Ação Recomendada:* Este é um alerta crítico para a equipe de TI e Controladoria da distribuidora. Esses 48 pedidos podem ser erros de integração sistêmica, vendas 100% bonificadas/cortesia que não geraram contas a receber, ou até mesmo perdas financeiras. Isolaremos esses IDs para investigação futura.

3. **Dinâmica Operacional do Comodato:** A tabela de logística de equipamentos registrou pouco mais de **2.000 pedidos**, representando aproximadamente metade (50%) do volume total de vendas. 
   * *Insight de Negócio:* Isso não é um erro sistêmico, mas sim o reflexo da realidade do negócio! Isso indica que nem toda venda exige o empréstimo de um equipamento. Uma parcela gigantesca da operação é composta por clientes que já possuem suas próprias chopeiras e compram apenas a "recarga" (barris de chopp), ou vendas que não demandam empréstimo de comodato.

---


### 🔍 Hipótese de Risco Integrado e Validação de Chaves

A hipótese inicial para a modelagem de risco era criar uma visão integrada de **"Dupla Inadimplência"**. O objetivo era cruzar as bases para identificar e quantificar os pedidos que apresentavam o pior cenário possível: **atraso no pagamento financeiro E retenção indevida do equipamento (chopeira/cilindro)**.

Para construir essa visão, a lógica natural em um banco de dados relacional é utilizar a chave primária da transação (`ID_PEDIDO`) para unir a tabela de Vendas (Consumo) com as tabelas de Financeiro e Comodato. 

Antes de gerar os gráficos e consolidar os atrasos, realizamos um diagnóstico estrutural para garantir que os IDs de pedido da tabela de Vendas conversam perfeitamente com os IDs da tabela de Comodato.

In [ ]:
# ─── 3.1 Auditoria de Integridade Relacional ─────────────────────────────────

# ==============================================================================
# DIAGNÓSTICO ESTRUTURAL: COBERTURA DAS TRILHAS POR CLIENTE
# ==============================================================================
# ⚠️ O raio-X original comparava conjuntos de ID_PEDIDO entre vendas e comodato
# para provar que o ERP usa numeração independente por processo. Sem o grão
# transacional essa prova não é mais reproduzível aqui. A pergunta de negócio
# subjacente, porém, continua respondível: quantos clientes existem em cada
# combinação de trilhas — é isso que as flags TEM_* expressam.

flags = ["TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO"]

so_vendas = df_clientes[
    (df_clientes["TEM_VENDAS"] == 1)
    & (df_clientes["TEM_FINANCEIRO"] == 0)
    & (df_clientes["TEM_COMODATO"] == 0)
]
vendas_e_fin = df_clientes[
    (df_clientes["TEM_VENDAS"] == 1)
    & (df_clientes["TEM_FINANCEIRO"] == 1)
    & (df_clientes["TEM_COMODATO"] == 0)
]
vendas_e_com = df_clientes[
    (df_clientes["TEM_VENDAS"] == 1)
    & (df_clientes["TEM_FINANCEIRO"] == 0)
    & (df_clientes["TEM_COMODATO"] == 1)
]
trilha_completa = df_clientes[
    (df_clientes["TEM_VENDAS"] == 1)
    & (df_clientes["TEM_FINANCEIRO"] == 1)
    & (df_clientes["TEM_COMODATO"] == 1)
]

print("=" * 60)
print("🔍 RAIO-X DE COBERTURA: TRILHAS POR CLIENTE")
print("=" * 60)
print(f"📦 Clientes com Vendas:     {int(df_clientes['TEM_VENDAS'].sum()):>6,}")
print(f"💰 Clientes com Financeiro: {int(df_clientes['TEM_FINANCEIRO'].sum()):>6,}")
print(f"🛢️ Clientes com Comodato:   {int(df_clientes['TEM_COMODATO'].sum()):>6,}")
print("-" * 60)
print(f"✅ Trilha COMPLETA (Vendas + Fin + Comodato): {len(trilha_completa):>6,}")
print(f"➖ Somente Vendas:                            {len(so_vendas):>6,}")
print(f"➖ Vendas + Financeiro (sem comodato):        {len(vendas_e_fin):>6,}")
print(f"➖ Vendas + Comodato (sem financeiro):        {len(vendas_e_com):>6,}")

# Distribuição completa das 2³ combinações possíveis — evidencia se existe
# alguma célula inesperada (ex.: comodato sem nenhuma venda associada).
combos = (
    df_clientes.groupby(flags).size().reset_index(name="QTD_CLIENTES")
    .sort_values("QTD_CLIENTES", ascending=False)
)

print("\n" + "=" * 60)
print("👀 MATRIZ DE COMBINAÇÕES (V / F / C → nº de clientes)")
print("=" * 60)
for _, r in combos.iterrows():
    marca = lambda v: "✔" if v == 1 else "·"
    print(
        f"  Vendas {marca(r['TEM_VENDAS'])} | Financeiro {marca(r['TEM_FINANCEIRO'])} "
        f"| Comodato {marca(r['TEM_COMODATO'])}  →  {int(r['QTD_CLIENTES']):>5,} clientes"
    )

cobertura = len(trilha_completa) / len(df_clientes) * 100
print(f"\n🎯 Cobertura da trilha completa: {cobertura:.1f}% da carteira")

#### 💡 Conclusão: Descoberta de Processo e Reposicionamento da Análise

O diagnóstico estrutural nos trouxe uma revelação crítica que altera a abordagem técnica do projeto: **praticamente não há intersecção de `ID_PEDIDO` entre a tabela de Vendas e a tabela de Comodato.** **Interpretação de Negócio (Insight Fiscal/Sistêmico):**
Essa desconexão não é um erro de extração de dados, mas sim o reflexo de uma regra de negócio enraizada no ERP da distribuidora. Em operações que envolvem venda de mercadorias (Chopp) e empréstimo de ativos (Chopeiras), os sistemas costumam separar a transação por razões fiscais e contábeis:
* É gerado um **Pedido A** exclusivo para o faturamento e cobrança do consumo (gera receita).
* É gerado um **Pedido B** exclusivo para a remessa de comodato do equipamento (não gera receita, apenas controle de ativo na rua).

**Decisão Metodológica:**
Como o sistema quebra a experiência do cliente em dois IDs distintos, torna-se inviável analisar o "Risco Duplo" no nível do *Pedido*. A partir de agora, **analisaremos a inadimplência financeira e o risco de retenção de comodato como duas trilhas paralelas e independentes**, avaliando cada uma dentro de sua própria realidade sistêmica. Futuramente, cruzamentos integrados deverão ser feitos no nível do Cliente (`ID_PESSOA`), e não mais no nível da transação.

> **Descoberta Estrutural Confirmada:** Pedidos de Venda e Comodato têm `ID_PEDIDO`
> **distintos** no ERP por razões fiscais. O cruzamento correto é no nível do **cliente**
> (`ID_PESSOA`), não do pedido.

## 📊 3.2 - Perfil do Cliente (Visão 360º)

Após o Tratamento transacional, iniciamos a **Análise Exploratória Visual (EDA)** com foco em mapear a identidade e o comportamento de consumo da nossa base. 

Como a variável de "Segmento/Estabelecimento" apresentou inconsistências, estruturamos esta etapa utilizando três dimensões cadastrais de alta confiabilidade. O painel a seguir consolida esse mapeamento, cruzando cada perfil com três métricas fundamentais de negócio: **Quantidade de Pedidos** (Frequência), **Valor Total** (Receita Bruta) e o **Ticket Médio**.

As três perspectivas analisadas são:

1. **Tipo de Cliente (Identidade):** Segmentação entre *Pessoa Física*, *Pessoa Jurídica* e *Estrangeiro*. Isso nos permite isolar o comportamento do consumidor B2B, do cliente final e do público transfronteiriço (devido à fronteira seca com o Paraguai).
2. **Setor de Entrega (Geografia):** Mapeamento do volume transacional pelas rotas e bairros cadastrados. Entender onde está a demanda é o primeiro passo para calcular o risco logístico atrelado ao empréstimo e recolhimento dos equipamentos em comodato.
3. **Forma de Pagamento (Perfil Financeiro):** Avaliação do método de acerto padrão do cliente (ex: dinheiro à vista, transferências, prazos). Este comportamento será uma das principais variáveis (*features*) de entrada para o futuro modelo preditivo de inadimplência.

In [ ]:
# ─── 3.2 Perfil 360° do Cliente ───────────────────────────────────────────────
# As dimensões PERFIL/CIDADE/PAGAMENTO já vêm resolvidas no dataset consolidado,
# então a montagem de dimensões e os merges desapareceram. Como o grão agora é o
# cliente, QTD_PEDIDOS vira a soma de FREQUENCIA_COMPRAS e FATURAMENTO a soma de
# TOTAL_GASTO — o ticket médio segue sendo faturamento / pedidos.
df_360 = df_clientes.copy()
df_360["CIDADE"] = df_360["CIDADE"].fillna("NÃO PREENCHIDO")


def add_labels(ax, is_money=False):
    """Anota valores no topo das barras."""
    for p in ax.patches:
        val = p.get_height()
        if val > 0:
            label = fmt_moeda(val) if is_money else f"{int(val):,}"
            ax.annotate(
                label,
                (p.get_x() + p.get_width() / 2, val),
                ha="center",
                va="bottom",
                fontweight="bold",
                fontsize=9,
                xytext=(0, 4),
                textcoords="offset points",
            )


fig, axes = plt.subplots(3, 3, figsize=(24, 18))
plt.subplots_adjust(hspace=0.55, wspace=0.3)

segmentos = ["PERFIL", "CIDADE", "PAGAMENTO"]
titulos_col = ["Perfil do Cliente", "Localização", "Forma de Pagamento (Top 7)"]
paletas = ["Blues_r", "Oranges_r", "Greens_r"]

for j, (seg, titulo) in enumerate(zip(segmentos, titulos_col)):
    df_t = (
        df_360.groupby(seg, observed=True)
        .agg(
            QTD_PEDIDOS=("FREQUENCIA_COMPRAS", "sum"),
            FATURAMENTO=("TOTAL_GASTO", "sum"),
        )
        .reset_index()
    )
    df_t["TICKET_MEDIO"] = df_t["FATURAMENTO"] / df_t["QTD_PEDIDOS"].replace(0, np.nan)
    df_t["TICKET_MEDIO"] = df_t["TICKET_MEDIO"].fillna(0)

    if seg == "CIDADE":
        # AMAMBAI passou a existir como categoria própria no dataset consolidado.
        ordem = [
            "PONTA PORÃ",
            "PEDRO JUAN CABALLERO",
            "AMAMBAI",
            "OUTRAS CIDADES",
            "NÃO PREENCHIDO",
        ]
        df_t = df_t.set_index(seg).reindex(ordem).reset_index().fillna(0)
    elif seg == "PAGAMENTO":
        df_t = df_t.sort_values("FATURAMENTO", ascending=False).head(7)

    for row, (col_y, is_money, ylabel) in enumerate(
        [
            ("QTD_PEDIDOS", False, "Qtd. Pedidos"),
            ("FATURAMENTO", True, "Total (R$)"),
            ("TICKET_MEDIO", True, "Média (R$)"),
        ]
    ):
        ax = axes[row, j]
        sns.barplot(
            data=df_t, x=seg, y=col_y, ax=ax, hue=seg, palette=paletas[j], legend=False
        )
        subtitulo = ["Vol. Pedidos", "Faturamento", "Ticket Médio"][row]
        ax.set_title(
            f"{subtitulo}: {titulo}", fontweight="bold", fontsize=11, color="#2C3E50"
        )
        ax.set_ylabel(ylabel)
        ax.set_xlabel("")
        if is_money:
            ax.yaxis.set_major_formatter(FuncFormatter(fmt_moeda))
        ax.tick_params(axis="x", rotation=25 if j > 0 else 0)
        add_labels(ax, is_money)

plt.suptitle(
    "3.2 - Dashboard Perfil 360° — Identidade · Localização · Pagamento",
    fontsize=18,
    fontweight="bold",
    y=1.01,
)
sns.despine()
plt.tight_layout()

os.makedirs("imagens", exist_ok=True)
plt.savefig(
    "imagens/Dashboard_perfil_360.png",
    dpi=300,            # alta resolução
    bbox_inches="tight" # corta bordas extras
)

plt.show()

### 💡 Conclusão da Análise de Perfil (Insights e Georreferência Corrigida)

Com a correção da georreferência e a consolidação do *Dashboard 360º*, refinamos nosso entendimento do negócio e mapeamos os seguintes insights estratégicos:

1. **Identidade vs. Faturamento:** Pessoas Físicas lideram o volume absoluto de pedidos. No entanto, **Pessoas Jurídicas** e **Estrangeiros** sustentam os maiores Tickets Médios, confirmando o peso financeiro da operação transfronteiriça.
2. **Geografia e Correlação:** **Ponta Porã** é o núcleo da operação. O perfil Estrangeiro correlaciona-se diretamente com **Pedro Juan Caballero**. O volume residual em "Outras Cidades" exige atenção especial na logística de recolhimento das chopeiras.
3. **Liquidez (Pagamentos):** A predominância de **Dinheiro e Pix** garante o giro rápido do caixa. Modalidades "A Prazo" ficam restritas ao público B2B e internacional (maior ticket).

**🚀 Decisão Metodológica para Modelagem:**
A variável de localização (Cidades) foi saneada (97,7% de preenchimento) 

## ⚠️ 3.3 - Análise de Risco por Segmento

Compreendendo que o faturamento (Financeiro) e a remessa (Comodato) operam de forma independente no ERP, mudamos a granularidade do nosso cruzamento. Em vez de cruzar ID com ID, vamos analisar o **Risco do Pedido** sob a ótica das dimensões do cliente.

Para cada pedido de Venda/Financeiro e cada pedido de Empréstimo/Comodato, mapeamos:
1. **Identidade:** Pessoa Física, Jurídica ou Estrangeiro.
2. **Localização:** Ponta Porã, Pedro Juan Caballero, Outras e Não Informado.
3. **Liquidez:** Principais Formas de Pagamento.

**Objetivo:** Identificar qual perfil de cliente apresenta o maior percentual de atraso no pagamento (Risco de Crédito) e qual apresenta o maior percentual de retenção de equipamento (Risco de Ativo).

In [ ]:
# ─── 3.3 Análise de Risco por Segmento ────────────────────────────────────
# ⚠️ MUDANÇA DE UNIDADE: antes a taxa era medida por PEDIDO (proporção de
# pedidos atrasados); agora é medida por CLIENTE — a média de RISCO_FINANCEIRO
# é a fração da carteira sinalizada como arriscada naquele segmento. A leitura
# gerencial é a mesma (onde dói mais), mas a magnitude não é comparável com a
# versão anterior do notebook.

fig, axes = plt.subplots(3, 1, figsize=(14, 16))
plt.subplots_adjust(hspace=0.5)

dimensoes_plot = [
    ("PERFIL", "Perfil do Cliente"),
    ("CIDADE", "Localização Geográfica"),
    ("PAGAMENTO", "Forma de Pagamento (Top 6)"),
]

for i, (dim, titulo) in enumerate(dimensoes_plot):
    df_plot = (
        df_clientes.groupby(dim, observed=True)
        .agg(
            RISCO_FIN=("RISCO_FINANCEIRO", "mean"),
            RISCO_COM=("RISCO_COMODATO", "mean"),
        )
        .reset_index()
    )
    df_plot[["RISCO_FIN", "RISCO_COM"]] *= 100

    if dim == "PAGAMENTO":
        top6 = df_clientes[dim].value_counts().nlargest(6).index
        df_plot = df_plot[df_plot[dim].isin(top6)]

    df_plot = (
        df_plot.assign(TOTAL_RISK=lambda x: x["RISCO_FIN"] + x["RISCO_COM"])
        .sort_values("TOTAL_RISK", ascending=False)
        .drop(columns="TOTAL_RISK")
    )

    df_melt = df_plot.melt(
        id_vars=dim,
        value_vars=["RISCO_FIN", "RISCO_COM"],
        var_name="Tipo",
        value_name="Taxa",
    )
    df_melt["Tipo"] = df_melt["Tipo"].map(
        {"RISCO_FIN": "Atraso Financeiro", "RISCO_COM": "Atraso Comodato"}
    )

    ax = axes[i]
    sns.barplot(data=df_melt, x=dim, y="Taxa", hue="Tipo", ax=ax, palette=["#E74C3C", "#F39C12"])
    ax.set_title(f"Clientes em Risco por {titulo}", fontweight="bold", fontsize=13)
    ax.set_ylabel("% de Clientes em Risco")
    ax.set_xlabel("")
    ax.set_ylim(0, df_melt["Taxa"].max() * 1.3 + 1)
    ax.yaxis.set_major_formatter(PercentFormatter())

    if dim == "PAGAMENTO":
        ax.tick_params(axis="x", rotation=20)
    for p in ax.patches:
        v = p.get_height()
        if v > 0.5:
            ax.annotate(
                f"{v:.1f}%",
                (p.get_x() + p.get_width() / 2, v),
                ha="center",
                va="bottom",
                fontweight="bold",
                fontsize=9,
                xytext=(0, 4),
                textcoords="offset points",
            )
    ax.legend(title="", loc="upper right", frameon=True)

plt.suptitle(
    "Termômetro de Risco: Financeiro vs. Retenção de Equipamento",
    fontsize=16,
    fontweight="bold",
    y=1.01,
)
sns.despine()
plt.tight_layout()

os.makedirs("imagens", exist_ok=True)
plt.savefig(
    "imagens/Termometro_Risco.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Recorte complementar: o PERFIL_RISCO já classifica cada cliente nas quatro
# combinações possíveis, então cruzá-lo com as dimensões mostra ONDE mora o
# risco duplo — informação que antes exigia unir as duas trilhas na mão.
print("\n" + "=" * 68)
print(f"{'🎯 COMPOSIÇÃO DO PERFIL DE RISCO POR SEGMENTO (%)':^68}")
print("=" * 68)
for dim, titulo in dimensoes_plot:
    tab = pd.crosstab(df_clientes[dim], df_clientes["PERFIL_RISCO"], normalize="index") * 100
    print(f"\n▸ {titulo}")
    display(tab.round(1))

### 💡 Conclusão: Termômetro de Risco e Comportamento por Segmento

A quebra do risco em trilhas paralelas (Financeiro vs. Comodato) revelou que a inadimplência e a retenção de equipamentos possuem motivadores diferentes. A partir do dashboard, extraímos os seguintes *insights* para a modelagem e para o negócio:

1. **Risco por Perfil do Cliente (Identidade):**
   * **Comodato:** O atraso na devolução de equipamentos é um problema crônico e generalizado, mantendo-se na faixa de **70%** para todos os públicos, com uma leve tendência de alta nas Pessoas Físicas.
   * **Financeiro:** Observamos uma clara divisão de fronteira. Clientes brasileiros (Pessoa Física e Jurídica) apresentam um risco de atraso financeiro na casa dos **55%**, enquanto o público **Estrangeiro** apresenta maior pontualidade, com o risco caindo para cerca de **47%**.

2. **Risco por Localização Geográfica (Logística):**
   * **Comodato (Distância = Risco):** Cidades vizinhas ("Outras Cidades") apresentam a maior taxa de atraso na devolução. Isso indica que a logística reversa perde eficiência quanto mais o equipamento se afasta do eixo Ponta Porã - Pedro Juan Caballero.
   * **Financeiro:** Curiosamente, o núcleo da operação (**Ponta Porã**) lidera a incidência de atrasos de pagamento. 
   * *Atenção Metodológica:* A categoria "Não Preenchido" apresentou altas taxas de atraso, mas, como mapeamos anteriormente, seu volume absoluto de pedidos é muito baixo. Para o treinamento do modelo preditivo, precisaremos tratar ou desconsiderar esses *outliers* para não enviesar o algoritmo.

3. **Risco por Forma de Pagamento (Liquidez):**
   * **O Fator Dinheiro:** Pedidos pagos em Dinheiro em espécie apresentam um risco ligeiramente menor de atraso na devolução da chopeira, possivelmente associados a transações mais rápidas e de balcão.
   * **A Oportunidade do PIX:** O pagamento via **Pix** desponta com o **menor risco de atraso financeiro** em comparação aos demais métodos (como boletos ou cartões a prazo). 
   * *Ação de Negócio:* Este é um *insight* altamente acionável. A distribuidora pode criar políticas de incentivo (como descontos no chopp ou isenção de taxa de entrega) para forçar a migração dos clientes para o Pix. Embora o risco do comodato ainda exista, o caixa da empresa fica garantido no tempo certo.

## ⏳ 3.4 Análise de "Aging"

Não basta saber *quem* atrasa; precisamos entender a **gravidade** desse atraso. Um cliente que atrasa a devolução da chopeira por 2 dias representa um gargalo logístico leve; um cliente que atrasa mais de 30 dias representa um alto risco de perda de ativo.

Para aprofundar nossa modelagem de risco, calculamos o tempo exato (em dias) de cada atraso e os agrupamos nas seguintes faixas de tolerância:
* **Leve:** 1 a 3 dias | 4 a 7 dias
* **Médio:** 8 a 15 dias | 16 a 20 dias
* **Crítico:** 21 a 30 dias | + de 30 dias

Abaixo, exploramos a volumetria absoluta de pedidos atrasados nessas faixas, cruzando o tempo de atraso (Financeiro e Comodato) com os perfis de cliente, localidade e métodos de pagamento.

In [ ]:
# ==============================================================================
# 3.4 AGING DA CARTEIRA (GRÃO DE CLIENTE)
# ==============================================================================
# ⚠️ MUDANÇA DE UNIDADE: o dashboard anterior contava PEDIDOS em atraso e
# empilhava por dimensão. Como AGING_PAGAMENTO/AGING_COMODATO já chegam
# calculados por cliente (faixa do pior atraso observado), a contagem passa a
# ser de CLIENTES. Perdeu-se o detalhe de quantos pedidos caíram em cada faixa;
# ganhou-se a leitura direta de quantos clientes estão em cada estágio.

# As colunas são pd.Categorical ordenadas — value_counts(sort=False) já devolve
# na ordem cronológica das faixas, sem precisar reindexar na mão.
aging_fin = df_clientes["AGING_PAGAMENTO"].value_counts(sort=False)
aging_com = df_clientes["AGING_COMODATO"].value_counts(sort=False)

df_aging = pd.DataFrame({"Financeiro": aging_fin, "Comodato": aging_com}).fillna(0)

print("=" * 62)
print(f"{'⏳ 3.4 - AGING DA CARTEIRA POR CLIENTE':^62}")
print("=" * 62)
print(f"  {'Faixa':<14} {'Financeiro':>12} {'Comodato':>12}")
print("-" * 62)
for faixa in df_aging.index:
    print(
        f"  {str(faixa):<14} {int(df_aging.loc[faixa, 'Financeiro']):>12,} "
        f"{int(df_aging.loc[faixa, 'Comodato']):>12,}"
    )

# ==============================================================================
# GRÁFICO 1: Aging comparado (Financeiro vs. Comodato)
# ==============================================================================
fig, ax = plt.subplots(figsize=(16, 8))
x = np.arange(len(df_aging.index))
width = 0.38

b_fin = ax.bar(x - width / 2, df_aging["Financeiro"], width,
               color="#E74C3C", edgecolor="white", label="Financeiro")
b_com = ax.bar(x + width / 2, df_aging["Comodato"], width,
               color="#F39C12", edgecolor="white", hatch="///", label="Comodato")

for barras in (b_fin, b_com):
    for b in barras:
        h = b.get_height()
        if h > 0:
            ax.annotate(
                f"{int(h):,}",
                (b.get_x() + b.get_width() / 2, h),
                ha="center", va="bottom", fontsize=10, fontweight="bold",
                color="#2C3E50", xytext=(0, 3), textcoords="offset points",
            )

ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in df_aging.index], fontsize=11, fontweight="bold", rotation=15)

# Escala simétrica-log preservada do dashboard original: a faixa "Sem Atraso"
# concentra a maior parte da carteira e achataria as faixas críticas.
ax.set_yscale("symlog", linthresh=10)
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_yticks([0, 10, 50, 100, 500, 1000, 2000])

ax.legend(loc="upper right", fontsize=11)
ax.set_title(
    "Dashboard de Aging Integrado: Distribuição de Clientes por Faixa de Atraso\n"
    "(Escala Logarítmica para visualização de volumes menores)",
    fontweight="bold", fontsize=16, pad=20,
)
ax.set_ylabel("Quantidade de Clientes (Escala SymLog)", fontsize=12, fontweight="bold")

sns.despine()
plt.tight_layout()

os.makedirs("imagens", exist_ok=True)
plt.savefig(
    "imagens/Dashboard_Aging.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ==============================================================================
# GRÁFICO 2: Aging empilhado por dimensão (mantém a leitura segmentada original)
# ==============================================================================
def plot_aging_por_dimensao(dimensao, titulo_dim, top_n=None):
    """Empilha as faixas de aging dentro de cada categoria da dimensão.

    Só considera clientes efetivamente em atraso — 'Sem Atraso' domina a base e
    esconderia a evolução das faixas críticas, que é o objeto da análise.
    """
    faixas_atraso = [f for f in ORDEM_AGING]

    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    especs = [
        ("AGING_PAGAMENTO", "Financeiro", axes[0]),
        ("AGING_COMODATO", "Comodato", axes[1]),
    ]

    for col_aging, rotulo, ax in especs:
        df_f = df_clientes[df_clientes[col_aging].astype(str) != "Sem Atraso"].copy()

        if top_n:
            top_items = df_f[dimensao].value_counts().nlargest(top_n).index
            df_f = df_f[df_f[dimensao].isin(top_items)]

        tab = (
            pd.crosstab(df_f[dimensao], df_f[col_aging])
            .reindex(columns=faixas_atraso, fill_value=0)
        )

        cores = sns.color_palette("Set2", len(faixas_atraso))
        tab.plot(kind="bar", stacked=True, ax=ax, color=cores, edgecolor="white", width=0.75)

        totais = tab.sum(axis=1)
        for idx, total in enumerate(totais):
            if total > 0:
                ax.text(idx, total, f"{int(total):,}", ha="center", va="bottom",
                        fontsize=10, fontweight="bold", color="#2C3E50")

        ax.set_title(f"{rotulo} — Aging por {titulo_dim}", fontweight="bold", fontsize=13)
        ax.set_ylabel("Clientes em Atraso", fontweight="bold")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=20)
        ax.legend(title="Faixa de Atraso", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)

    sns.despine()
    plt.tight_layout()
    plt.show()


plot_aging_por_dimensao("PERFIL", "Perfil do Cliente")
plot_aging_por_dimensao("CIDADE", "Localidade (Regiões Foco)")
plot_aging_por_dimensao("PAGAMENTO", "Forma de Pagamento (Top 5)", top_n=5)

### 💡 Conclusão: Análise de Aging e Padrões de Retenção Crítica

Para garantir uma leitura precisa do comportamento de atrasos, aplicamos uma **Escala Logarítmica Simétrica (SymLog)** na visualização. Essa técnica foi fundamental para contornar o efeito de "cauda longa" dos dados, permitindo que os blocos com volumes massivos (centenas ou milhares de pedidos com atraso curto) não esmagassem visualmente as ocorrências críticas (menor volume, porém de altíssimo risco).

A partir dessa visualização ajustada, destacamos os seguintes comportamentos iniciais:

1. **Concentração de Curto Prazo:** De maneira geral, a grande esmagadora maioria dos atrasos (tanto financeiros quanto de devolução de comodato) se resolve na janela de **até 15 dias**. Isso indica que boa parte da inadimplência não é má-fé, mas sim desorganização, esquecimento ou delay logístico natural.

2. **A Cauda Crítica (+30 Dias):** Embora o volume caia expressivamente após a barreira de 15 dias, a quantidade de pedidos que ultrapassam os 30 dias de atraso ainda é preocupante para o caixa e para a retenção de ativos da empresa.

3. **Proporcionalidade vs. Risco Real:** Nos atrasos curtos, a distribuição acompanha a proporção natural da base de clientes. No entanto, quando olhamos para a faixa mais crítica (21 a 30+ dias), começam a surgir desvios de comportamento que merecem atenção especial.

## 📅 3.5 - Série Histórica


### 📈 Panorama Histórico: Recência da Carteira e Faturamento por Perfil

> ⚠️ **Nota de limitação (mudança de grão dos dados).** A versão anterior desta análise apresentava a evolução do faturamento mês a mês, a partir da data de cada pedido. Com a migração para o dataset consolidado — uma linha por cliente — a data transacional deixou de existir na base: resta apenas o **mês da última compra** de cada cliente. A série deixou, portanto, de ser uma curva de receita ao longo do tempo e passou a ser uma **curva de recência da carteira**: cada cliente aparece uma única vez, no mês em que comprou pela última vez.

Feita a ressalva, a leitura estratégica continua válida. O gráfico a seguir mantém a estrutura de **Duplo Eixo**:

* **Gráfico de Barras Empilhadas (Faturamento Acumulado):** O eixo esquerdo controla as colunas, que representam o total já faturado pelos clientes cujo último pedido ocorreu naquele mês. As barras são fatiadas por cores para demonstrar, em valores absolutos e percentuais, a contribuição de cada Perfil de Cliente (Física, Jurídica e Estrangeiro).
* **Gráfico de Linhas (Volume de Clientes):** O eixo direito acompanha a quantidade de clientes cuja última compra caiu naquele mês.

Esse cruzamento permite identificar **onde a carteira está concentrada e onde ela está esfriando**: meses antigos com faturamento alto revelam clientes de peso que deixaram de comprar — exatamente o público que o modelo de risco precisa sinalizar.

In [ ]:
# =====================================================================
# 3.5 SÉRIE HISTÓRICA — RECOMPOSTA NO GRÃO DE CLIENTE
# =====================================================================
# ⚠️ LIMITAÇÃO: a série mensal de faturamento exigia DT_PEDIDO por transação.
# O dataset consolidado guarda apenas MES_ULTIMA_COMPRA (string "YYYY-MM"),
# então o eixo temporal deixou de ser "quando cada venda ocorreu" e passou a ser
# "quando o cliente comprou pela última vez". Um cliente aparece UMA única vez
# na série, no seu mês de recência — e não em todos os meses em que comprou.
# A leitura muda de "curva de receita" para "curva de recência da carteira":
# ainda mostra concentração e sazonalidade, mas não é mais uma série de vendas.

df_plot = df_clientes.copy()
df_plot["ANO_MES"] = df_plot["MES_ULTIMA_COMPRA"].astype(str)
df_plot = df_plot[df_plot["ANO_MES"].str.match(r"^\d{4}-\d{2}$", na=False)]

# Barras empilhadas: faturamento histórico dos clientes, fatiado por PERFIL.
df_barras = df_plot.groupby(["ANO_MES", "PERFIL"], observed=True)["TOTAL_GASTO"].sum().reset_index()
df_pivot = (
    df_barras.pivot(index="ANO_MES", columns="PERFIL", values="TOTAL_GASTO")
    .fillna(0)
    .sort_index()
)

# Linha: quantos clientes têm sua última compra naquele mês.
df_linha = (
    df_plot.groupby("ANO_MES").size().reset_index(name="QTD_CLIENTES").sort_values("ANO_MES")
)

# =====================================================================
# 2. PLOTAGEM DO GRÁFICO (Eixo Duplo)
# =====================================================================

fig, ax1 = plt.subplots(figsize=(16, 7))

# --- EIXO 1 (Esquerda): Barras Empilhadas (Faturamento por Perfil) ---

cores_perfis = ['#4C72B0', '#DD8452', '#55A868']

df_pivot.plot(kind='bar', stacked=True, ax=ax1, color=cores_perfis[: df_pivot.shape[1]],
              edgecolor='white', width=0.8)

# Configurações do Eixo 1
ax1.set_ylim(0, df_pivot.sum(axis=1).max() * 1.60)
ax1.set_xlabel('Mês da Última Compra', fontsize=12, fontweight='bold')
ax1.set_ylabel('Faturamento Acumulado (R$)', fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.legend(title='Perfil do Cliente', loc='upper left')

# Formatação de moeda no Eixo Y
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: fmt_moeda(x)))

# =====================================================================
# ADICIONAR RÓTULOS: % INTERNA E VALOR TOTAL NO TOPO
# =====================================================================

# 1. Calcula o faturamento total de cada mês (base para as % e para o rótulo do topo)
totais_mensais = df_pivot.sum(axis=1).values

# 2. Cria uma lista para rastrear a altura real do topo de cada barra
altura_topo_barra = [0] * len(df_pivot)

# 3. Loop para colocar as porcentagens (%) dentro de cada fatia da barra
for container in ax1.containers:
    for idx, patch in enumerate(container):
        altura_bloco = patch.get_height()
        y_inicio = patch.get_y()

        # Rastreia qual é o ponto mais alto alcançado pela barra deste mês
        if (y_inicio + altura_bloco) > altura_topo_barra[idx]:
            altura_topo_barra[idx] = y_inicio + altura_bloco

        if altura_bloco > 0:
            total_do_mes = totais_mensais[idx]
            porcentagem = (altura_bloco / total_do_mes) * 100

            # Só escreve a % se o bloco for maior que 3% (evita amontoar texto)
            if porcentagem > 3:
                x_centro = patch.get_x() + patch.get_width() / 2
                y_centro = y_inicio + altura_bloco / 2

                ax1.text(x_centro, y_centro, f'{porcentagem:.1f}%',
                         ha='center', va='center', color='white',
                         fontweight='bold', fontsize=9)

# 4. Loop para colocar o VALOR TOTAL DO MÊS no topo de cada barra empilhada
for idx, total in enumerate(totais_mensais):
    x_pos_barra = idx
    y_pos_topo = altura_topo_barra[idx]

    # Exibe o valor total do mês usando a sua função utilitária fmt_moeda
    ax1.text(x_pos_barra, y_pos_topo + (df_pivot.sum(axis=1).max() * 0.015), fmt_moeda(total),
             ha='center', va='bottom', color='#4A4A4A',
             fontweight='bold', fontsize=9, rotation=0)

# --- EIXO 2 (Direita): Gráfico de Linha (Quantidade de Clientes) ---
ax2 = ax1.twinx() # Compartilha o mesmo eixo X do ax1

# Garante o alinhamento perfeito da linha no centro das barras
x_pos = range(len(df_pivot.index))

ax2.plot(x_pos, df_linha['QTD_CLIENTES'], color='#4A4A4A', marker='o', linewidth=3,
         markersize=5, label='Qtd Clientes')

# Configurações do Eixo 2
ax2.set_ylabel('Quantidade de Clientes', fontsize=12, fontweight='bold', color='#4A4A4A')
ax2.tick_params(axis='y', labelcolor='#4A4A4A')

# Adiciona os números (rótulos) acima de cada ponto da linha
for i, v in enumerate(df_linha['QTD_CLIENTES']):
    ax2.text(i, v + (v * 0.02), str(v), color='#4A4A4A', ha='center', va='bottom',
             fontweight='bold', fontsize=9)

# --- FINALIZAÇÃO E AJUSTES VISUAIS ---
plt.title('Recência da Carteira: Faturamento por Perfil vs. Clientes por Mês de Última Compra',
          fontsize=16, fontweight='bold', pad=20)
ax1.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()

os.makedirs("imagens", exist_ok=True)
plt.savefig(
    "imagens/Serie_Historica_1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

#### 💡 Insights: Evolução Histórica e Sazonalidade

A análise do panorama histórico nos revela padrões de consumo fundamentais para o planejamento estratégico e para o nosso modelo preditivo:

1. **Potencial de Crescimento e Impulso Estrangeiro:** Ao compararmos a trajetória de vendas de 2024 com a tração observada em 2025, notamos um claro viés de alta no faturamento e no volume de transações. Esse avanço vem sendo impulsionado fortemente pela maior participação das vendas para o perfil **Estrangeiro**, reforçando a importância estratégica da operação transfronteiriça para a escalabilidade e o Ticket Médio do negócio.

2. **Forte Sazonalidade (Último Quadrimestre):** O gráfico evidencia picos operacionais sazonais muito bem definidos. Observamos uma alta considerável na demanda nos meses de reta final de ano (especialmente no período entre **setembro e dezembro**). Esse aquecimento natural — atrelado a festividades, eventos e altas temperaturas — exige da distribuidora um preparo logístico robusto não apenas para a entrega do chopp, mas para o gerenciamento de risco no comodato das chopeiras, que rodam com a capacidade máxima nesses meses.

### 📅 Exposição da Carteira por Mês de Última Compra

> ⚠️ **Nota de limitação (análise não reproduzível no novo grão).** A versão original desta seção cruzava, mês a mês, o **valor em R$ de parcelas vencidas e não pagas** com a **quantidade de equipamentos retidos**. Ambas as métricas dependiam da data de vencimento de cada parcela e de cada comodato — informação que o dataset consolidado não preserva, pois agrega tudo por cliente. **A curva histórica de inadimplência não pode ser reconstruída.**

Em seu lugar, adotamos uma pergunta vizinha e ainda aderente ao objetivo do projeto: **a carteira que parou de comprar em cada mês é mais ou menos arriscada?** Isso conecta recência (abandono) a risco, que é justamente a hipótese central do modelo.

**Os gráficos abaixo apresentam:**
1. 🔴 **Eixo Financeiro (Barras):** Valor total em Reais (R$) de parcelas dos clientes cuja última compra ocorreu naquele mês.
2. 🟠 **Eixo Logístico (Linha):** Quantidade de equipamentos em comodato na posse desses mesmos clientes.
3. 🟣 **Concentração de Risco Duplo (Linha):** Percentual de clientes classificados como `RISCO DUPLO` em cada mês de recência.

Cruzando essas métricas, visualizamos qual safra de clientes inativos concentra a maior exposição simultânea de caixa e de ativos imobilizados em campo.

In [ ]:
# ─── 3.5 Série Histórica ──────────────────────────────────────────────────────
# ⚠️ ANÁLISE PERDIDA: o gráfico original cruzava, mês a mês, o VALOR (R$) de
# parcelas vencidas e não pagas com a QUANTIDADE de equipamentos retidos. Ambas
# as métricas dependiam da data de vencimento de cada parcela/comodato, que não
# existe no dataset consolidado — ele guarda apenas o agregado por cliente
# (VALOR_TOTAL_PARCELAS, PARCELAS_ATRASADAS, QTD_EQUIPAMENTOS...). Não é
# possível reconstruir a curva de inadimplência ao longo do tempo.
#
# SUBSTITUIÇÃO: usamos MES_ULTIMA_COMPRA como eixo temporal para responder uma
# pergunta próxima e ainda útil — a carteira que parou de comprar em cada mês
# é mais ou menos arriscada? Isso conecta recência (abandono) com risco, que é
# justamente a hipótese do modelo.

df_risco_mes = df_clientes.copy()
df_risco_mes["ANO_MES"] = df_risco_mes["MES_ULTIMA_COMPRA"].astype(str)
df_risco_mes = df_risco_mes[df_risco_mes["ANO_MES"].str.match(r"^\d{4}-\d{2}$", na=False)]

df_hist_risco = (
    df_risco_mes.groupby("ANO_MES")
    .agg(
        VALOR_EM_ABERTO=("VALOR_TOTAL_PARCELAS", "sum"),
        EQUIPAMENTOS=("QTD_EQUIPAMENTOS", "sum"),
        CLIENTES=("ID_PESSOA", "count"),
        TAXA_RISCO_DUPLO=("PERFIL_RISCO", lambda s: (s == "RISCO DUPLO").mean() * 100),
    )
    .reset_index()
    .sort_values("ANO_MES")
)
df_hist_risco["MES_STR"] = df_hist_risco["ANO_MES"]

fig, ax1 = plt.subplots(figsize=(16, 7))
cor_fin, cor_com = "#E74C3C", "#F39C12"
bars = ax1.bar(
    df_hist_risco["MES_STR"],
    df_hist_risco["VALOR_EM_ABERTO"],
    color=cor_fin,
    alpha=0.8,
    label="R$ em Parcelas (clientes com última compra no mês)",
)
ax1.set_ylabel("Valor em Parcelas (R$)", color=cor_fin, fontweight="bold")
ax1.yaxis.set_major_formatter(FuncFormatter(fmt_moeda))
ax1.tick_params(axis="y", labelcolor=cor_fin)
ax1.tick_params(axis="x", rotation=45, labelsize=9)
ax1.set_ylim(0, df_hist_risco["VALOR_EM_ABERTO"].max() * 1.4)

ax2 = ax1.twinx()
ax2.plot(
    df_hist_risco["MES_STR"],
    df_hist_risco["EQUIPAMENTOS"],
    color=cor_com,
    marker="o",
    linewidth=2.5,
    markersize=7,
    label="Equipamentos em Comodato",
)
ax2.set_ylabel("Qtd. Equipamentos em Comodato", color=cor_com, fontweight="bold")
ax2.tick_params(axis="y", labelcolor=cor_com)
ax2.grid(False)

for bar in bars:
    h = bar.get_height()
    if h > 0:
        ax1.annotate(
            fmt_moeda(h),
            (bar.get_x() + bar.get_width() / 2, h),
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            color=cor_fin,
            xytext=(0, 3),
            textcoords="offset points",
        )

for x, y in zip(df_hist_risco['MES_STR'], df_hist_risco['EQUIPAMENTOS']):
    if y > 0:
        ax2.annotate(f'{int(y)} un', (x, y), ha='center', va='bottom', xytext=(0, 10),
                     textcoords='offset points', fontsize=8, fontweight='bold', color=cor_com)


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.grid(axis="y", linestyle="--", alpha=0.3)
plt.title(
    "Exposição da Carteira por Mês de Última Compra: Financeiro vs. Ativos em Campo",
    fontsize=15,
    fontweight="bold",
    pad=20,
)
sns.despine(right=False)
plt.tight_layout()

os.makedirs("imagens", exist_ok=True)
plt.savefig(
    "imagens/Serie_Historica_2.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Composição de risco por mês de recência: o complemento que substitui a curva
# de inadimplência perdida — mostra se quem sumiu há mais tempo é mais arriscado.
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_hist_risco["MES_STR"], df_hist_risco["TAXA_RISCO_DUPLO"],
        color="#8E44AD", marker="s", linewidth=2.5, markersize=6)
ax.set_ylabel("% Clientes em RISCO DUPLO", fontweight="bold", color="#8E44AD")
ax.yaxis.set_major_formatter(PercentFormatter())
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.set_title("Concentração de RISCO DUPLO por Mês da Última Compra",
             fontsize=14, fontweight="bold", pad=15)
ax.grid(axis="y", linestyle="--", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

#### 💡 Conclusão: Evolução Histórica e o Gargalo Operacional

A visão cronológica de duplo eixo nos permite cruzar as dores financeiras com as dores logísticas da distribuidora ao longo do tempo. A partir desta análise histórica, consolidamos os seguintes aprendizados vitais para o negócio:

1. **Correlação de Riscos (Efeito Cascata):** Existe uma clara tendência de acompanhamento entre os dois indicadores. Salvo raras exceções em meses específicos, quando a taxa de atraso de pagamento sobe, a retenção indevida de ativos (chopeiras/barris) sobe junto. Isso indica que o "cliente devedor" e o "cliente retentor" costumam ser a mesma persona ou são afetados pelos mesmos fatores externos.

2. **O Reflexo do Volume de Vendas:** O comportamento da curva de inadimplência espelha de forma quase idêntica o nosso gráfico de Volume de Vendas apresentado no início da EDA. O risco cresce à medida que a esteira de faturamento acelera.

3. **🚨 Ponto de Atenção Crítico (O Custo da Alta Temporada):** A descoberta mais sensível desta análise é constatar que **os atrasos mais críticos e volumosos ocorrem justamente nos períodos de pico de vendas**. 
   * **Impacto no Negócio:** Isso evidencia um forte gargalo operacional. Nos meses de maior demanda, a equipe da distribuidora foca todas as energias em "vender e entregar", perdendo a tração nas réguas de cobrança e na logística reversa (recolhimento). O resultado é um acúmulo perigoso de capital na rua e risco de perda de patrimônio exatamente quando a empresa mais precisa de fluxo de caixa e giro de equipamentos.

**Próximos Passos:** Com esse diagnóstico claro de que o aumento das vendas traz consigo um aumento desproporcional do risco na alta temporada, a necessidade de um **Modelo Preditivo** torna-se indiscutível. O algoritmo será fundamental para atuar como um "filtro inteligente", garantindo que a distribuidora escale suas vendas nos meses de pico apenas para perfis com alta probabilidade de pagamento e devolução.

## ☢️ 3.6 Radiografia do Risco Integrado por Cliente

Até o momento, analisamos os atrasos financeiros e logísticos de forma isolada. No entanto, o risco real de um cliente de comodato se manifesta na combinação dessas duas dimensões. Esta célula realiza um cruzamento matricial definitivo, gerando uma **radiografia integrada de comportamento** na granularidade de cliente (`ID_CLIENTE`).

O objetivo é classificar nossa carteira em 4 quadrantes claros de atrito (fricção), mapeando quem são os parceiros saudáveis e quem são os **Super Detratores** que geram prejuízo duplo para a operação.

---

#### 📊 **Os 4 Perfis de Risco Analisados (Fatias do Gráfico)**

| Perfil | Comportamento de Pagamento | Comportamento com o Comodato | Nível de Risco | Action / Estratégia |
| :--- | :--- | :--- | :--- | :--- |
| 🟢 **Bons Clientes** | Sempre em dia | Devolve os ativos no prazo | **Nulo** | Manter benefícios e incentivos. |
| 🟡 **Risco Financeiro** | Apresenta histórico de atrasos | Devolve os ativos no prazo | **Médio** | Adotar travas de crédito ou boleto antecipado. |
| 🟠 **Risco Logístico** | Sempre em dia | Retém barris/equipamentos além do prazo | **Médio** | Aplicar multas de retenção ou revisar contrato. |
| 🔴 **Super Detratores** | Apresenta histórico de atrasos | Retém barris/equipamentos além do prazo | **Crítico** | Bloqueio imediato de novos pedidos e recolhimento de ativos. |

---

#### 💡 **Importância Estratégica para o Modelo (Fase 5)**
Essa segmentação servirá como um validador empírico para a nossa variável resposta (`TARGET`). Se o modelo de Machine Learning for eficaz, ele deverá ser altamente sensível na identificação prévia dos clientes que caminham em direção ao grupo dos **Super Detratores**, permitindo que a empresa aja preventivamente antes que o ativo seja retido ou o calote financeiro se consolide.

In [ ]:
# ==============================================================================
# 3.6 BASE MESTRA DE CLIENTES (O UNIVERSO DE ANÁLISE)
# ==============================================================================
# Esta é a análise que melhor sobreviveu à mudança de grão: PERFIL_RISCO já é
# exatamente o cruzamento RISCO_FINANCEIRO × RISCO_COMODATO que a célula
# construía na mão a partir das três views. Todo o trabalho de merge e np.select
# migrou para a extração — aqui resta validar e visualizar.

df_consolidado = df_clientes.copy()

# Validação: a matriz 2x2 tem que reproduzir exatamente as classes de PERFIL_RISCO.
matriz_risco = pd.crosstab(
    df_consolidado["RISCO_FINANCEIRO"],
    df_consolidado["RISCO_COMODATO"],
    rownames=["Risco Financeiro"],
    colnames=["Risco Comodato"],
)

print("=" * 62)
print(f"{'☢️  MATRIZ DE RISCO INTEGRADO (Financeiro × Comodato)':^62}")
print("=" * 62)
display(matriz_risco)

# Renomeia para a taxonomia numerada do relatório (mantém a ordem de gravidade
# crescente na legenda e nas cores da pizza).
MAPA_PERFIL = {
    "SEM RISCO": "1. Sem Histórico de Atrasos",
    "SÓ FINANCEIRO": "2. Apenas Atraso Financeiro",
    "SÓ COMODATO": "3. Apenas Atraso Comodato",
    "RISCO DUPLO": "4. Risco Crítico (Fin + Comodato)",
}
df_consolidado["PERFIL_RISCO_LABEL"] = (
    df_consolidado["PERFIL_RISCO"].map(MAPA_PERFIL).fillna("Outros")
)

# ==============================================================================
# 5. VISUALIZAÇÃO - GRÁFICO PIZZA
# ==============================================================================
stats = df_consolidado['PERFIL_RISCO_LABEL'].value_counts().sort_index()
total_clientes = len(df_consolidado)

plt.figure(figsize=(11, 8))

cores = ['#2ECC71', '#E74C3C', '#F39C12', '#8E44AD']

plt.pie(
    stats,
    autopct=lambda p: f'{p:.1f}%\n({int(round(p * total_clientes / 100))})',
    startangle=140,
    colors=cores[: len(stats)],
    explode=[0.05] * len(stats),
    textprops={'fontsize': 11, 'fontweight': 'bold'}
)

plt.legend(stats.index, title="Classificação da Carteira", loc="center left", bbox_to_anchor=(1, 0.5))
plt.title(f'Radiografia do Risco Integrado por Cliente ({total_clientes} Clientes)', fontsize=16, fontweight='bold', pad=20)
plt.axis('equal')
plt.tight_layout()

os.makedirs("imagens", exist_ok=True)
plt.savefig(
    "imagens/Radiografia_Risco.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Contexto financeiro de cada classe — quanto de exposição está em cada balde.
resumo_risco = (
    df_consolidado.groupby("PERFIL_RISCO_LABEL")
    .agg(
        CLIENTES=("ID_PESSOA", "count"),
        FATURAMENTO=("TOTAL_GASTO", "sum"),
        VALOR_PARCELAS=("VALOR_TOTAL_PARCELAS", "sum"),
        EQUIPAMENTOS=("QTD_EQUIPAMENTOS", "sum"),
    )
    .reset_index()
    .sort_values("PERFIL_RISCO_LABEL")
)

print("\n" + "=" * 78)
print(f"{'💼 EXPOSIÇÃO FINANCEIRA POR CLASSE DE RISCO':^78}")
print("=" * 78)
for _, r in resumo_risco.iterrows():
    print(f"  {r['PERFIL_RISCO_LABEL']:<36}")
    print(f"     👥 {fmt_numero(r['CLIENTES']):>8} clientes   "
          f"💰 {fmt_moeda(r['FATURAMENTO']):>10} faturado   "
          f"🧾 {fmt_moeda(r['VALOR_PARCELAS']):>10} em parcelas   "
          f"🛢️ {fmt_numero(r['EQUIPAMENTOS']):>6} equip.")

### 💡 Conclusão: Radiografia da Carteira e Sobreposição de Riscos

A mudança de perspectiva do nível de "Pedido" para o nível de "Cliente" nos permitiu traçar o verdadeiro perfil de risco da nossa carteira ativa. O cruzamento das inadimplências revelou um cenário que exige forte atenção gerencial:

1. **A Minoria de Excelência (22%):** Apenas cerca de um quinto da nossa base atual é composta por clientes "livres de dor de cabeça" (sem nenhum histórico de atraso financeiro ou retenção de equipamentos). O fato de 78% dos clientes já terem gerado alguma fricção operacional mostra que atrasos são quase culturais nessa operação.

2. **A Dor Latente do Caixa (40%):** O atraso exclusivamente financeiro é o cenário mais recorrente. É o cliente que eventualmente escorrega no boleto ou na data do Pix, mas não gera passivo de maquinário. Exige uma esteira de cobrança ágil, mas ainda é um cliente que "gira" o produto.

3. **Inadimplência de Ativo (15%):** Uma fatia de clientes tem um comportamento curioso: pagam rigorosamente em dia, mas não devolvem a chopeira/cilindro no prazo. Para este grupo, o problema não é crédito, é logística reversa.

4. **🚨 A Zona de Perigo - Risco Crítico (22%):** Impressionantes 22% da base caem no pior cenário possível: **são ofensores duplos**. Eles comprometem o fluxo de caixa (não pagam) e bloqueiam o patrimônio da empresa (não devolvem o equipamento para ser alugado para outra pessoa). 
   * **Ação Estratégica:** Este é o grupo que corrói a margem de lucro da distribuidora. As políticas de concessão precisam ser severas com este perfil, exigindo, por exemplo, pagamento 100% antecipado e até cobrança de caução (garantia) para a liberação da chopeira.

**O Papel do Machine Learning:** O grande objetivo do nosso Modelo Preditivo daqui para frente será atuar como um "radar antecipado". Precisamos que o algoritmo consiga ler os dados de uma nova venda e prever, com base em padrões de histórico, idade, localização e forma de pagamento, se esse cliente tem probabilidade de cair na temida fatia dos **22% de Risco Crítico**, permitindo que a empresa bloqueie a venda antes que o prejuízo aconteça.

# 🔧 Fase 4 - Engenharia de Atributos e Preparação dos Datasets

Nesta etapa, transitamos da Análise Exploratória de Dados (EDA) para a construção da base que alimentará nossos modelos preditivos. O objetivo principal da **Engenharia de Atributos (Feature Engineering)** é transformar dados brutos e históricos de faturamento, entregas e interações em variáveis numéricas e categóricas de alto valor preditivo, capazes de capturar o comportamento de risco de inadimplência ou quebra de contrato no comodato de chopp.

### 🎯 Objetivos Estruturais
1. **Consolidação de Features:** Agregar os históricos transacionais (`VL_FINANCEIRO`, prazos de pagamento, médias de dias de atraso) na granularidade correta (por cliente/parceiro).
2. **Tratamento de Consistência:** Aplicar as regras de negócio para imputação de valores ausentes (`fillna`) e encoding de variáveis categóricas cruciais (como o `PERFIL` e a `CIDADE`).

### 📊 Estratégia de Divisão e Arquitetura dos DataFrames

Para garantir a reprodutibilidade e a aplicação prática do modelo em ambiente produtivo, segregamos nossa estrutura de dados em dois grandes blocos:

| Dataset | Finalidade no Projeto | Características |
| :--- | :--- | :--- |
| `df_ml_completo` | **Base de Produção / Inferência** | Contém a visão 360° de todos os clientes ativos com suas respectivas variáveis calculadas. É a base que receberá o *score* de risco diário ou mensal quando o modelo estiver rodando em produção. |
| `df_treino` / `df_teste` | **Desenvolvimento do Modelo** | Subconjunto extraído a partir do histórico elegível, contendo a variável resposta (`TARGET`) definida. Passará pela separação rigorosa (Holdout / Cross-Validation) para treinar, tunar e avaliar os algoritmos na Fase 5. |

---

## 👤 4.1 Consolidação da Visão Cadastral e Estruturação de Lookup por Cliente

O primeiro passo operacional da Engenharia de Atributos consiste em mudar a granularidade dos nossos dados. Enquanto as fases anteriores trabalharam a nível de transação/pedido, a modelagem preditiva exige uma linha única por cliente (`ID_CLIENTE`). 

Esta célula é responsável por construir a fundação dessa estrutura, consolidando as informações cadastrais e perfis históricos que servirão como base de cruzamento (*lookup*) para as próximas transformações.

---

#### 🛠️ **Processamento Executado na Célula**
1. **Definição da Entidade Central:** Isolamento do identificador único do cliente para garantir que não haja duplicidade de registros na base final.
2. **Mapeamento de Atributos Estuturais:** Resgate de variáveis qualitativas críticas a partir das dimensões tratadas, tais como:
   * **`PERFIL` / `TIPO_PESSOA`:** Essencial para capturar o comportamento de risco diferenciado entre pessoas físicas e jurídicas.
   * **`CIDADE`:** Variável de geolocalização relevante para análises de risco regionalizado na distribuição.
3. **Criação da Chave de Lookup:** Estruturação de um DataFrame base que vincula a identificação do cliente às suas características de cadastro estáveis, preparando o terreno para receber os agregados financeiros (`VL_FINANCEIRO`) e comportamentais nas células seguintes.



In [ ]:
# ─── FASE 4: Construção de df_ml_completo ─────────────────────────────────────
#
#  A agregação por cliente acontece na EXTRAÇÃO, onde é auditada (a soma do
#  faturamento por cliente é conferida contra a soma das linhas de origem).
#  Esta célula seleciona o universo de modelagem e valida o contrato — não
#  reagrega nada.
#
#  O recorte de core business (chopp/chopeira) também vem pronto: depende de
#  ID_PRODUTO, que só existe no grão transacional, então é marcado lá como a
#  flag CORE_BUSINESS. O REGEX que a define está em Extracao_Dados_Consolidados.
# ──────────────────────────────────────────────────────────────────────────────
print("🔄 Construindo df_ml_completo (unidade de análise: ID_PESSOA)")

# ── Universo de modelagem: clientes do core business ─────────────────────────
if "CORE_BUSINESS" not in df_clientes.columns:
    raise KeyError(
        "Coluna CORE_BUSINESS ausente — reexecute Extracao_Dados_Consolidados.ipynb. "
        "Sem ela não há como isolar os clientes de chopp/chopeira."
    )

df_ml_completo = df_clientes[df_clientes["CORE_BUSINESS"] == 1].copy()

print(f"\n   🍺 Clientes de core business : {len(df_ml_completo):>6,}")
print(f"   🚫 Fora do core business     : {len(df_clientes) - len(df_ml_completo):>6,}")
print(f"   💰 Faturamento do recorte    : R$ {df_ml_completo['TOTAL_GASTO'].sum():,.2f}")

# ── Validação do contrato com o painel de hiperparâmetros ────────────────────
# ALL_FEATURES vem de HP_ARQUITETURA (Fase 0). Se alguém acrescentar uma feature
# ao painel sem acrescentar a coluna de origem ao contrato da extração, o erro
# aparece AQUI com a mensagem certa — e não como KeyError na hora do fit.
_faltando = [c for c in ALL_FEATURES if c not in df_ml_completo.columns]
if _faltando:
    raise KeyError(
        f"Features declaradas em HP_ARQUITETURA e ausentes no dataset: {_faltando}. "
        f"Acrescente a coluna de origem em COLUNAS_NECESSARIAS "
        f"(Extracao_Dados_Consolidados.ipynb) e reexecute a extração."
    )

# ── Tratamento de ausentes ────────────────────────────────────────────────────
# Zero é a leitura correta para contagens e taxas: "nenhuma parcela" são 0
# parcelas, não um valor desconhecido.
_COLS_ZERO = [c for c in [
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG",
    "TAXA_ATRASO_PAGAMENTO", "TOTAL_COMODATOS", "COMODATOS_ATRASADOS",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "TAXA_ATRASO_COMODATO",
    "QTD_EQUIPAMENTOS",
] if c in df_ml_completo.columns]
df_ml_completo[_COLS_ZERO] = df_ml_completo[_COLS_ZERO].fillna(0)

for _col, _val in [("PERFIL", "NÃO INFORMADO"), ("CIDADE", "NÃO INFORMADO"),
                   ("PAGAMENTO", "NÃO INFORMADO"), ("NOME_CLIENTE", "N/D")]:
    if _col in df_ml_completo.columns:
        df_ml_completo[_col] = df_ml_completo[_col].fillna(_val)

# NaN numérico chegando ao StandardScaler vira erro no fit. Checagem explícita
# aqui dá uma mensagem muito mais útil do que a exceção do sklearn.
_com_nan = [c for c in FEATURES_NUM if df_ml_completo[c].isna().any()]
if _com_nan:
    raise ValueError(
        f"Features numéricas com NaN após o tratamento: {_com_nan}. "
        f"Verifique a agregação na extração."
    )

print(f"\n✅ df_ml_completo criado: {df_ml_completo.shape[0]:,} clientes × {df_ml_completo.shape[1]} colunas")
print(f"   ├─ Features do modelo : {len(ALL_FEATURES)} ({len(FEATURES_NUM)} num + {len(FEATURES_CAT)} cat)")
print(f"   └─ Identidade         : ID_PESSOA, NOME_CLIENTE (nunca entram no fit)")

print(f"\n   Distribuição das categóricas:")
for _cat in FEATURES_CAT:
    _vc = df_ml_completo[_cat].value_counts()
    print(f"   ├─ {_cat:<10} ({_vc.size} valores): {dict(list(_vc.head(3).items()))}")

display(df_ml_completo.head(10))

## 🔀 4.2 Separação Explícita de Escopo: 
## Base de Produção vs. Base de Treino

Nesta célula, executamos a segregação física e lógica dos dados em duas frentes independentes. Essa separação explícita é um pilar essencial de governança, garantindo que o modelo seja treinado em um cenário controlado e que a empresa possua uma estrutura pronta para gerar escoragem (*scoring*) de risco em ambiente produtivo.

---

#### 📐 Direcionamento dos DataFrames

1. **`df_ml_completo` (Visão 360° / Tabela de Produção):**
   * **Escopo:** Consolida 100% dos clientes ativos.
   * **Finalidade:** Atuará como barramento de consulta rápida (*lookup*). No ecossistema real (Fase 7 - Deploy), essa tabela alimentará a função `predict_by_id()`, permitindo que o gestor consulte instantaneamente a saúde preditiva de qualquer cliente da base.

2. **`df_treino` (Foco em Aprendizado de Máquina):**
   * **Escopo:** Filtra apenas o histórico elegível de clientes veteranos, eliminando o ruído e o viés de novos cadastros sem histórico transacional (*cold-start*).
   * **Finalidade:** É a base exclusiva que será enviada para a partição de modelagem (Treino/Teste) na Fase 5.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SEPARAÇÃO EXPLÍCITA:  df_ml_completo  →  df_treino
#
#  df_ml_completo : tabela de usuários — todos os clientes, com ID_PESSOA
#                   serve para lookup em produção (predict_by_id)
#
#  df_treino      : dataset de modelagem — apenas veteranos, sem colunas de
#                   identidade, com a variável-alvo calculada (ALTO_RISCO)
#
#  Todos os parâmetros vêm do painel da Fase 0 — nada é fixado aqui:
#    · elegibilidade e regra do alvo → HP_ARQUITETURA
#    · features                      → HP_ARQUITETURA['dados']
#    · partição treino/teste         → HP_TREINAMENTO['split']
# ══════════════════════════════════════════════════════════════════════════════

# ── Filtro de Veteranos (HP_ARQUITETURA['dados']['min_compras']) ──────────────
df_treino = df_ml_completo[df_ml_completo["FREQUENCIA_COMPRAS"] > MIN_COMPRAS].copy()

# ── Variável-Alvo (HP_ARQUITETURA['alvo']) ────────────────────────────────────
_combinador = HP_ARQUITETURA["alvo"]["combinador"]
_cond_pag = df_treino["TAXA_ATRASO_PAGAMENTO"] > LIMITE_ATRASO
_cond_com = df_treino["TAXA_ATRASO_COMODATO"] > LIMITE_ATRASO

if _combinador == "OU":
    _condicao_risco = _cond_pag | _cond_com
elif _combinador == "E":
    _condicao_risco = _cond_pag & _cond_com
else:
    raise ValueError(f"alvo.combinador='{_combinador}' inválido — use 'OU' ou 'E'.")

df_treino[TARGET] = np.where(_condicao_risco, 1, 0)

# ── Guarda de vazamento ───────────────────────────────────────────────────────
# As colunas que CONSTROEM o alvo não podem entrar como feature. A checagem é
# barata e evita o erro mais caro possível: um AUC alto que não significa nada.
_proibidas_no_X = set(HP_ARQUITETURA["dados"]["features_proibidas"]) & set(ALL_FEATURES)
if _proibidas_no_X:
    raise ValueError(f"❌ Vazamento: features proibidas em ALL_FEATURES: {sorted(_proibidas_no_X)}")

_faltando = [c for c in ALL_FEATURES if c not in df_treino.columns]
if _faltando:
    raise KeyError(f"❌ Features declaradas no painel e ausentes em df_treino: {_faltando}")

# ── Relatório de Distribuição do Target ───────────────────────────────────────
n_total = len(df_treino)
n_risco = int(df_treino[TARGET].sum())
n_bom = n_total - n_risco
pct_risco = n_risco / n_total * 100

print("=" * 66)
print(f"{'📐 SEPARAÇÃO DE DATASETS':^66}")
print("=" * 66)
print(f"\n  df_ml_completo             : {len(df_ml_completo):>5,} clientes  (lookup de produção)")
print(f"  df_treino                  : {n_total:>5,} clientes  (veteranos: compras > {MIN_COMPRAS})")
print(f"  Descartados                : {len(df_ml_completo)-n_total:>5,} clientes  (insuficiência de histórico)")
print(f"  % Clientes Utilizados      : {n_total/len(df_ml_completo)*100:.1f}%")

print(f"\n{'─'*66}")
print(f"{'📊 VARIÁVEL-ALVO: ' + TARGET:^66}")
print(f"{'─'*66}")
print(f"  Regra          : TAXA_ATRASO_PAG > {LIMITE_ATRASO:.0%} {_combinador} TAXA_ATRASO_COM > {LIMITE_ATRASO:.0%}")
print(f"  Alto Risco (1) : {n_risco:>5,}  ({pct_risco:.1f}%)")
print(f"  Bom Pagador (0): {n_bom:>5,}  ({100-pct_risco:.1f}%)")

if pct_risco > 35:
    print(f"\n  ⚠️  Desbalanceamento detectado → class_weight='balanced' definido no painel")

# ── Divisão Treino / Teste (HP_TREINAMENTO['split']) ──────────────────────────
X = df_treino[ALL_FEATURES].copy()
y = df_treino[TARGET].copy()

_split = HP_TREINAMENTO["split"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=_split["test_size"],
    random_state=RANDOM_STATE,
    shuffle=_split["shuffle"],
    stratify=y if _split["stratify"] else None,
)

# n do estrato minoritário no teste: governa a largura de TODOS os ICs adiante.
_n_neg_teste = int((y_test == 0).sum())

print(f"\n{'─'*66}")
print(f"{'DIVISÃO TREINO / TESTE':^66}")
print(f"{'─'*66}")
print(f"  X_train : {X_train.shape[0]:,} clientes  ({int((1-TEST_SIZE)*100)}%)")
print(f"  X_test  : {X_test.shape[0]:,} clientes  ({int(TEST_SIZE*100)}%)")
print(f"  Estratificado: {_split['stratify']}  |  semente: {RANDOM_STATE}")
print(f"\n  Features numéricas  : {len(FEATURES_NUM)}")
print(f"  Features categóricas: {len(FEATURES_CAT)}")
print(f"  Total de features   : {len(ALL_FEATURES)}")
print(f"\n  ⚠️  Negativos no teste: {_n_neg_teste} — é este n que determina a largura")
print(f"      dos intervalos de confiança de especificidade e AUC na Fase 6.")

## 💾 4.3 Materialização dos Datasets em Delta

O CSV gravado na célula anterior vive no disco efêmero do serverless e desaparece entre sessões. Esta etapa persiste os datasets no **Volume do Unity Catalog** em formato **Delta**, que carrega o schema junto dos dados — ao reler, `TICKET_MEDIO` volta como `double` e as datas como `timestamp`, sem a reconversão manual que o CSV exige e que é fonte silenciosa de divergência entre execuções.

| Dataset | Conteúdo | Para que serve |
| :--- | :--- | :--- |
| `dataset_producao` | `df_ml_completo` — todos os clientes, com identidade | tabela de lookup consultada por `predict_by_id()` |
| `dataset_treino` | elegíveis + features + alvo | **é a entrada do treinamento** — o dataframe final tratado |
| `dataset_split` | a partição treino/teste desta rodada | congela `X_train`/`X_test` para que as métricas sejam reproduzíveis |

O `dataset_split` carrega `_CONFIG_HASH` e `_EXPERIMENT_TAG`, amarrando a partição ao painel de controle que a gerou. Sem isso, reproduzir um resultado antigo depende de re-executar todo o pipeline e torcer para o split coincidir.

> Delta versiona a cada `overwrite`: rodadas anteriores continuam acessíveis por `option("versionAsOf", n)`, o que permite auditar o dataset exato de um experimento passado.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  4.3 MATERIALIZAÇÃO DOS DATASETS EM DELTA (Unity Catalog Volumes)
#
#  Por que materializar, e não apenas manter em memória:
#    · No serverless o disco local é efêmero — o CSV da célula anterior some
#      entre sessões. O Volume é durável.
#    · Delta guarda o SCHEMA junto dos dados: ao reler, TICKET_MEDIO volta como
#      double e DT_* como timestamp. CSV devolve tudo como string e obriga a
#      reconverter, o que é fonte silenciosa de divergência entre execuções.
#    · Congela a partição: X_train/X_test gravados são exatamente os que
#      produziram as métricas desta rodada. Sem isso, "reproduzir o resultado"
#      depende de re-executar todo o pipeline e torcer para o split coincidir.
#
#  O que é gravado (três níveis de granularidade, propósitos distintos):
#    dataset_producao  → df_ml_completo : TODOS os clientes, com identidade.
#                        É a tabela de lookup que predict_by_id() consulta.
#    dataset_treino    → df_treino      : elegíveis, com o alvo calculado.
#                        É a ENTRADA do treinamento — o dataframe final tratado.
#    split_train/test  → a partição exata desta rodada, com CONFIG_HASH.
# ══════════════════════════════════════════════════════════════════════════════

VOLUME_BASE = "/Volumes/projetointegrador/default/projetointegrador"
DELTA_DIR = f"{VOLUME_BASE}/datasets"

# Colunas de identidade: existem em df_ml_completo para o lookup em produção,
# mas NUNCA entram no modelo. Materializadas junto porque a tabela de produção
# precisa delas para responder "quem é o cliente 1042?".
COLS_IDENTIDADE = ["ID_PESSOA", "NOME_CLIENTE"]


def _preparar_para_delta(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza um DataFrame pandas para escrita em Delta.

    Duas correções necessárias e não óbvias:
      1. Spark não aceita nomes de coluna com espaço ou pontuação;
      2. colunas 'object' que contêm apenas NaN viram NullType no Spark e
         quebram a escrita — forçamos string.
    """
    out = df.copy()
    out.columns = [
        str(c).strip().replace(" ", "_").replace("-", "_").replace(".", "_")
        for c in out.columns
    ]
    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].astype(str)
    return out


def materializar_delta(
    df: pd.DataFrame,
    nome: str,
    descricao: str,
    modo: str = "overwrite",
) -> str:
    """
    Grava um DataFrame pandas como tabela Delta no Volume.

    mergeSchema/overwriteSchema: ao tunar o painel você muda o conjunto de
    features, e portanto o schema. Sem overwriteSchema a segunda rodada falha
    com AnalysisException em vez de sobrescrever.
    """
    caminho = f"{DELTA_DIR}/{nome}"
    sdf = spark.createDataFrame(_preparar_para_delta(df))

    (
        sdf.write
        .format("delta")
        .mode(modo)
        .option("overwriteSchema", "true")
        .save(caminho)
    )

    print(f"   ✅ {nome:<22} → {df.shape[0]:>6,} linhas × {df.shape[1]:>3} colunas")
    print(f"      {descricao}")
    print(f"      {caminho}")
    return caminho


print("=" * 78)
print(f"{'💾 MATERIALIZAÇÃO DOS DATASETS EM DELTA':^78}")
print("=" * 78)
print(f"  Destino    : {DELTA_DIR}")
print(f"  Rodada     : {EXPERIMENT_TAG}")
print(f"  Config hash: {CONFIG_HASH}\n")

# ── 1. Base de produção (lookup por ID) ───────────────────────────────────────
PATH_PRODUCAO = materializar_delta(
    df_ml_completo,
    "dataset_producao",
    "Todos os clientes + identidade — consumido por predict_by_id()",
)

# ── 2. Dataset de treino: A ENTRADA DO MODELO ────────────────────────────────
# Contém ALL_FEATURES + TARGET + identidade (para auditar quem foi classificado
# como o quê). O modelo lê só ALL_FEATURES; a identidade viaja junto para
# rastreabilidade, e a guarda em X_train/X_test impede que ela entre no fit.
_cols_treino = [c for c in COLS_IDENTIDADE if c in df_treino.columns] + ALL_FEATURES + [TARGET]
df_treino_final = df_treino[_cols_treino].copy()

PATH_TREINO = materializar_delta(
    df_treino_final,
    "dataset_treino",
    f"Elegíveis (compras > {MIN_COMPRAS}) + alvo {TARGET} — ENTRADA do treinamento",
)

# ── 3. Partição congelada desta rodada ────────────────────────────────────────
# _SPLIT marca a origem da linha; CONFIG_HASH amarra ao painel que a gerou.
_df_tr = X_train.copy()
_df_tr[TARGET] = y_train
_df_tr["_SPLIT"] = "train"

_df_te = X_test.copy()
_df_te[TARGET] = y_test
_df_te["_SPLIT"] = "test"

df_split = pd.concat([_df_tr, _df_te], ignore_index=True)
df_split["_CONFIG_HASH"] = CONFIG_HASH
df_split["_EXPERIMENT_TAG"] = EXPERIMENT_TAG

PATH_SPLIT = materializar_delta(
    df_split,
    "dataset_split",
    f"Partição {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} congelada — reproduz as métricas desta rodada",
)

# ── Registro no MLflow: liga os dados materializados ao run ──────────────────
# Sem isto, meses depois, "qual dataset gerou este modelo?" não tem resposta.
try:
    with mlflow.start_run(run_id=RUN_PAI_ID):
        mlflow.log_params({
            "delta_path_producao": PATH_PRODUCAO,
            "delta_path_treino": PATH_TREINO,
            "delta_path_split": PATH_SPLIT,
        })
        mlflow.log_dict(
            {
                "config_hash": CONFIG_HASH,
                "experiment_tag": EXPERIMENT_TAG,
                "gerado_em": datetime.now().isoformat(),
                "datasets": {
                    "dataset_producao": {
                        "caminho": PATH_PRODUCAO,
                        "linhas": int(df_ml_completo.shape[0]),
                        "colunas": list(df_ml_completo.columns),
                        "uso": "lookup de produção (predict_by_id)",
                    },
                    "dataset_treino": {
                        "caminho": PATH_TREINO,
                        "linhas": int(df_treino_final.shape[0]),
                        "colunas": list(df_treino_final.columns),
                        "features": ALL_FEATURES,
                        "target": TARGET,
                        "uso": "entrada do treinamento",
                    },
                    "dataset_split": {
                        "caminho": PATH_SPLIT,
                        "linhas": int(df_split.shape[0]),
                        "n_train": int(len(X_train)),
                        "n_test": int(len(X_test)),
                        "uso": "partição congelada desta rodada",
                    },
                },
            },
            "datasets_materializados.json",
        )
    print(f"\n   📌 Caminhos e schema registrados no run pai do MLflow.")
except NameError:
    # A materialização pode rodar antes da Fase 5 numa execução parcial.
    print(f"\n   ℹ️  RUN_PAI_ID ainda não existe — execute a Fase 5 para vincular ao MLflow.")
except Exception as e:
    print(f"\n   ⚠️  Registro no MLflow indisponível ({type(e).__name__}): {e}")

print("\n" + "-" * 78)
print("  COMO RELER (em qualquer sessão futura, sem re-executar o pipeline):")
print("-" * 78)
print(f'  df_treino = spark.read.format("delta").load(')
print(f'      "{DELTA_DIR}/dataset_treino").toPandas()')
print(f'  X = df_treino[ALL_FEATURES]; y = df_treino["{TARGET}"]')
print("\n  Delta versiona a cada overwrite. Para auditar uma rodada anterior:")
print(f'  spark.read.format("delta").option("versionAsOf", 0).load(...)')
print("=" * 78)

# 🤖 Fase 5 - Modelagem com Pipelines Modulares

Nesta fase, entramos no núcleo de Inteligência Artificial do projeto. Utilizando a base de dados refinada na Engenharia de Atributos, passamos a construir, treinar e validar os algoritmos de Machine Learning. O objetivo principal é desenvolver um classificador capaz de prever a probabilidade de um cliente se tornar um detrator de risco (inadimplência ou retenção de ativos), permitindo ações preventivas antes da consolidação do prejuízo.

### ⚙️ Estratégia de Modelagem
Para garantir um processo robusto, escalável e livre de vieses, nossa abordagem se baseia em três pilares técnicos:
1. **Pipelines Modulares:** Automação das etapas de pré-processamento, garantindo que transformações nos dados de treino nunca vazem para os dados de teste.
2. **Validação Cruzada (CV):** Divisão em múltiplos blocos (folds) para garantir que a performance do modelo seja estável e não fruto do acaso.
3. **Métricas Robustas ao Desbalanceamento:** com 86,8% de positivos, Accuracy e AUC isoladas enganam — prever "todos são risco" já acerta 86,8%. A seleção do campeão usa **MCC** e **Balanced Accuracy**, que só sobem quando *ambas* as classes são bem classificadas, sempre confrontadas com um **baseline** que responde a classe majoritária. O **Recall** da classe de risco continua sendo a métrica de negócio prioritária (falso negativo tem custo alto), mas é lido junto da especificidade, não isolado.

---

> 🗃️ **Governança de Modelos:** Todas as iterações, hiperparâmetros e métricas de performance serão centralizados em um repositório estruturado em memória (`MODEL_REGISTRY`), permitindo uma comparação justa e auditável para a escolha do modelo campeão que irá para produção.

---

## 🧱 5.1 Fábrica de Pipelines e Inicialização do Repositório (`MODEL_REGISTRY`)

Esta célula estabelece a infraestrutura de engenharia de software para os nossos modelos. Em vez de aplicar transformações nos dados de forma manual e isolada, encapsulamos o fluxo em **Pipelines do Scikit-Learn** — garantindo que qualquer tratamento aplicado no treino (escala de variáveis, codificação de categorias) seja replicado de forma idêntica e segura na base de produção (`df_ml_completo`).

A célula **não contém nenhum hiperparâmetro**. Ela é uma **fábrica**: lê `HP_ARQUITETURA` (Fase 0) e materializa os objetos sklearn correspondentes. A consequência prática é forte — o que o MLflow loga como configuração é, por construção, o que de fato foi treinado. Não existe caminho pelo qual os dois divirjam.

#### 🛠️ **Componentes Estruturados nesta Célula**

1. **`build_preprocessor()`** — traduz `HP_ARQUITETURA['preproc']` em um `ColumnTransformer`. Trocar `scaler_numerico` para `robust` ou `minmax` é uma edição de **uma palavra** na Fase 0.
2. **`build_classifier()`** — instancia cada algoritmo a partir de sua entrada no painel, injetando `random_state` apenas onde o estimador o aceita.
3. **`build_pipeline()`** — encadeia pré-processador + classificador.
4. **`MODEL_REGISTRY`** — repositório central de versões. Guarda, lado a lado, os **hiperparâmetros declarados** (`hp_declarados`, o que você pediu) e os **params efetivos** do objeto sklearn (`params`, o que o sklearn de fato usou, defaults incluídos). Divergência entre os dois é sintoma de um default entrando sem passar pelo painel.

#### ➕ Para testar um algoritmo novo

Importe a classe, acrescente uma entrada em `_CLASSES_MODELO` e um bloco em `HP_ARQUITETURA['modelos']`. Nada mais muda. Para desligar um candidato temporariamente sem perder sua configuração, basta `"ativo": False`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PIPELINES MODULARES — construídos A PARTIR de HP_ARQUITETURA
#
#  Nenhum hiperparâmetro é escrito nesta célula. Ela é uma FÁBRICA: lê o painel
#  de controle da Fase 0 e materializa os objetos sklearn correspondentes.
#  Consequência prática: o que o MLflow loga como configuração é, por
#  construção, o que de fato foi treinado — não há caminho para divergirem.
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.base import clone
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# ── Fábrica de pré-processador ────────────────────────────────────────────────
_SCALERS = {
    "standard": StandardScaler,
    "minmax": MinMaxScaler,
    "robust": RobustScaler,
    "none": None,
}


def build_preprocessor(cfg: dict = HP_ARQUITETURA) -> ColumnTransformer:
    """Materializa o ColumnTransformer descrito em HP_ARQUITETURA['preproc']."""
    p = cfg["preproc"]

    nome_scaler = p["scaler_numerico"]
    if nome_scaler not in _SCALERS:
        raise ValueError(
            f"scaler_numerico='{nome_scaler}' desconhecido. Opções: {list(_SCALERS)}"
        )
    classe_scaler = _SCALERS[nome_scaler]
    step_num = "passthrough" if classe_scaler is None else classe_scaler()

    if p["encoder_categorico"] != "onehot":
        raise ValueError(
            f"encoder_categorico='{p['encoder_categorico']}' não implementado."
        )
    step_cat = OneHotEncoder(
        handle_unknown=p["onehot_handle_unknown"],
        drop=p["onehot_drop"],
        sparse_output=False,
    )

    return ColumnTransformer(
        transformers=[
            ("num", step_num, cfg["dados"]["features_num"]),
            ("cat", step_cat, cfg["dados"]["features_cat"]),
        ],
        remainder=p["remainder"],
        verbose_feature_names_out=False,
    )


preprocessor = build_preprocessor()

# ── Cache de Pipeline ─────────────────────────────────────────────────────────
# NOTA (serverless): o caminho relativo cairia no workspace filesystem — lento e
# persistente — e /local_disk0 é bloqueado por permissão. Com 333 linhas o
# pré-processamento leva milissegundos, então o cache não paga o próprio risco.
# Mantido desligado; use tempfile.mkdtemp() se algum dia precisar reativar.
cache_dir = None


def build_pipeline(clf, preprocessor=preprocessor, cache=cache_dir) -> Pipeline:
    """
    Constrói um Pipeline sklearn com preprocessor + classificador.

    Parâmetros
    ----------
    clf         : estimador sklearn (LogisticRegression, RandomForest, etc.)
    preprocessor: ColumnTransformer pré-configurado (fábrica acima)
    cache       : joblib.Memory para cache de transformações

    Retorna: Pipeline com steps ['prep', 'clf']
    """
    # clone(): cada pipeline recebe a PRÓPRIA cópia não-treinada do
    # pré-processador. Sem isso os três pipelines compartilhariam a mesma
    # instância por referência, e o .fit() de um sobrescreveria in place o
    # StandardScaler.mean_ já ajustado do outro.
    # Hoje o efeito seria nulo (todos treinam no mesmo X_train), mas passa a
    # corromper silenciosamente assim que um candidato treinar em colunas ou
    # linhas diferentes — exatamente o experimento de ablação previsto no painel.
    return Pipeline(steps=[("prep", clone(preprocessor)), ("clf", clf)], memory=cache)


# ── Fábrica de classificadores ────────────────────────────────────────────────
# Mapa nome→classe. Para testar um algoritmo novo: importe-o, acrescente uma
# entrada aqui e um bloco em HP_ARQUITETURA['modelos'] — nada mais muda.
_CLASSES_MODELO = {
    "LogisticRegression": LogisticRegression,
    "DecisionTreeClassifier": DecisionTreeClassifier,
    "RandomForestClassifier": RandomForestClassifier,
}

# random_state é injetado só onde o estimador o aceita: passá-lo a um estimador
# determinístico levanta TypeError e derrubaria a rodada inteira.
_ACEITA_RANDOM_STATE = {
    "LogisticRegression",   # relevante para solvers estocásticos (saga/liblinear)
    "DecisionTreeClassifier",
    "RandomForestClassifier",
}


def build_classifier(nome: str, cfg_modelo: dict, random_state: int = RANDOM_STATE):
    """Instancia um classificador a partir de sua entrada em HP_ARQUITETURA."""
    nome_classe = cfg_modelo["classe"]
    if nome_classe not in _CLASSES_MODELO:
        raise ValueError(
            f"Modelo '{nome}': classe '{nome_classe}' não registrada em _CLASSES_MODELO."
        )
    params = dict(cfg_modelo["params"])
    if nome_classe in _ACEITA_RANDOM_STATE:
        params["random_state"] = random_state
    return _CLASSES_MODELO[nome_classe](**params)


# ── Instanciação dos candidatos ativos ────────────────────────────────────────
clfs = {
    nome: build_classifier(nome, cfg)
    for nome, cfg in HP_ARQUITETURA["modelos"].items()
    if cfg.get("ativo", True)
}

_modelos_inativos = [
    nome for nome, cfg in HP_ARQUITETURA["modelos"].items() if not cfg.get("ativo", True)
]

# ── MODEL_REGISTRY: dicionário central de versões ─────────────────────────────
# Guarda a config declarada (hp_declarados) LADO A LADO com os params efetivos
# do objeto (params). Divergência entre os dois é sintoma de default do sklearn
# entrando sem passar pelo painel — vale conferir na hora de comparar rodadas.
MODEL_REGISTRY = {}
for nome, clf in clfs.items():
    MODEL_REGISTRY[nome] = {
        "pipeline": build_pipeline(clf),
        "params": clf.get_params(),
        "hp_declarados": HP_ARQUITETURA["modelos"][nome]["params"],
        "versao": MODEL_VERSION,
        "config_hash": CONFIG_HASH,
        "experiment_tag": EXPERIMENT_TAG,
        "data_criacao": datetime.now().isoformat(),
        "metricas_cv": {},
        "metricas_teste": {},
        "treinado": False,
    }

print("✅ MODEL_REGISTRY construído a partir do painel de controle!")
print(f"{'─'*72}")
print(f"  Pré-processador: {HP_ARQUITETURA['preproc']['scaler_numerico']} "
      f"({len(FEATURES_NUM)} num) + onehot "
      f"({len(FEATURES_CAT)} cat, drop={HP_ARQUITETURA['preproc']['onehot_drop']})")
print(f"{'─'*72}")
for nome, entry in MODEL_REGISTRY.items():
    tipo_clf = type(entry["pipeline"].named_steps["clf"]).__name__
    hp_resumo = ", ".join(f"{k}={v}" for k, v in entry["hp_declarados"].items())
    print(f"  📦 {nome:<22} → {tipo_clf}")
    print(f"     └─ {hp_resumo}")
if _modelos_inativos:
    print(f"{'─'*72}")
    print(f"  ⏸️  Inativos (ativo=False): {', '.join(_modelos_inativos)}")
print(f"{'─'*72}")
print(f"  Config hash: {CONFIG_HASH}  |  Rodada: {EXPERIMENT_TAG}")
print(f"\n💡 Para tunar: edite HP_ARQUITETURA['modelos'][...]['params'] na Fase 0")
print(f"   e re-execute a partir de lá. Esta célula nunca precisa ser editada.")

print("\n🔍 Visualização do Pipeline (primeiro candidato ativo):")
display(MODEL_REGISTRY[next(iter(MODEL_REGISTRY))]["pipeline"])

---

## ⚙️ 5.2 Motor de Treinamento: Execução Instrumentada e Validação Cruzada (CV)

Esta célula contém o "motor" de execução do nosso experimento. Para evitar a repetição de blocos de código a cada algoritmo testado, implementamos uma **função mestre reutilizável** que automatiza o ciclo completo de vida de um modelo de classificação e registra tudo no MLflow.

Nenhum parâmetro de controle aparece na assinatura das funções — todos vêm de `HP_TREINAMENTO`. Isso é intencional: um valor default de função é exatamente o tipo de configuração que escapa do painel e depois não aparece no log.

#### 🔄 Ciclo de Execução Automatizado

1. **Validação Cruzada Estratificada:** o modelo é avaliado em `n_splits` partições que preservam a proporção da classe de risco, mitigando o efeito do desbalanceamento (86,8% de positivos).
2. **Diagnóstico de overfitting:** para cada métrica é logado o **gap treino − validação** (`cv_*_gap_overfit`). É este número — e não a métrica de validação isolada — que se observa ao tunar `max_depth`, `C` ou `min_samples_leaf`.
3. **Holdout com intervalos de confiança:** métricas para **ambas as classes**, com IC de Wilson (proporções) e de Hanley & McNeil (AUC). Com apenas ~19 negativos no teste, o IC é largo — e mostrar isso é mais honesto do que exibir três casas decimais de um AUC pontual.
4. **Varredura de limiar (`sweep_*`):** o corte de decisão é tunável **sem retreinar**. A grade de 0,20 a 0,80 é varrida e logada como série, transformando a escolha do ponto de operação em decisão baseada em curva.
5. **Alimentação do painel e do MLflow:** performance gravada no `MODEL_REGISTRY` e no tracking server.

#### 🗂️ Contrato de rastreabilidade

Cada run carrega o painel inteiro, com prefixos que revelam a origem de cada valor:

| Prefixo | Origem | Serve para |
| :--- | :--- | :--- |
| `arq.*` | `HP_ARQUITETURA` | isolar mudanças de **forma** do modelo |
| `trn.*` | `HP_TREINAMENTO` | isolar mudanças de **protocolo** de avaliação |
| `clf__*` | `estimador.get_params()` | conferir o que o sklearn **realmente** usou, defaults incluídos |

Comparar `arq.modelo.*` com `clf__*` revela qualquer default silencioso que tenha entrado sem passar pelo painel. E o artefato `config_experimento.json` traz a configuração completa: basta copiá-lo de volta para a Fase 0 para reproduzir a rodada.

> ⚠️ **Não há busca automática de hiperparâmetros** (`GridSearchCV` / `RandomizedSearchCV`) nesta versão. O tuning é **manual e deliberado**: uma configuração por rodada, cada rodada com sua etiqueta e seu hash. Com ~19 negativos no conjunto de teste, uma busca ampla otimizaria ruído — e o resultado pareceria melhor sem ser melhor.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR DE TREINAMENTO INSTRUMENTADO COM MLFLOW
#  Estrutura: 1 run PAI (a rodada) → N runs FILHOS (baseline + cada modelo)
#
#  CONTRATO DE RASTREABILIDADE desta célula:
#    · todo hiperparâmetro que influencia o resultado é logado como parâmetro,
#      com prefixo que revela a que bloco do painel pertence:
#          arq.*  → HP_ARQUITETURA   (a forma do modelo)
#          trn.*  → HP_TREINAMENTO   (como se aprende e decide)
#          clf__* → params efetivos do estimador sklearn
#    · config_hash identifica a rodada; runs com hash igual usaram config igual
#    · o painel inteiro vai como artefato JSON — reprodutível por cópia direta
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    fbeta_score,
)

# ─── Por que ESTAS métricas ───────────────────────────────────────────────────
# A classe positiva é MAIORIA (86,8%). Nesse regime:
#   · Accuracy engana → prever "todos são risco" já acerta 86,8%
#   · AUC engana      → o teste tem só ~19 negativos;
# A classe RARA aqui é a 0 (bom pagador). Por isso medimos as duas classes e
# incluímos métricas insensíveis a prevalência: balanced accuracy e MCC.
# SCORING_CV vem de HP_TREINAMENTO['scoring_cv'] (Fase 0) — não é redefinido aqui.


_Z_IC = HP_TREINAMENTO["intervalo_confianca"]["z"]


def _ic_wilson(acertos: int, n: int, z: float = _Z_IC) -> tuple:
    """IC 95% de Wilson — apropriado para n pequeno (aqui: ~19 negativos)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = acertos / n
    den = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / den
    margem = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (max(0.0, centro - margem), min(1.0, centro + margem))


def _ic_auc_hanley(auc: float, n_pos: int, n_neg: int, z: float = _Z_IC) -> tuple:
    """IC do AUC por Hanley & McNeil (1982).
    Crítico neste projeto: com ~19 negativos o IC é largo e mostra que a
    ordenação entre os modelos pode não ter suporte estatístico."""
    if n_pos == 0 or n_neg == 0:
        return (float("nan"), float("nan"), float("nan"))
    q1 = auc / (2 - auc)
    q2 = 2 * auc**2 / (1 + auc)
    se = float(np.sqrt(
        (auc * (1 - auc) + (n_pos - 1) * (q1 - auc**2) + (n_neg - 1) * (q2 - auc**2))
        / (n_pos * n_neg)
    ))
    return (max(0.0, auc - z * se), min(1.0, auc + z * se), se)


def avaliar_no_teste(pipeline, X_test, y_test, threshold: float = THRESHOLD_PADRAO) -> dict:
    """
    Conjunto completo de métricas de holdout, para AMBAS as classes.

    threshold: limiar aplicado a predict_proba. Vem de
    HP_TREINAMENTO['threshold']['classificacao']. Quando != 0.5 as métricas
    dependentes de rótulo (F1, Recall, MCC, matriz) mudam sem retreino —
    por isso o limiar usado é logado junto das métricas.
    """
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    report = cast(Dict[str, Any], classification_report(
        y_test, y_pred, output_dict=True, zero_division=0))
    c1 = report.get("1", {"precision": 0, "recall": 0, "f1-score": 0})
    c0 = report.get("0", {"precision": 0, "recall": 0, "f1-score": 0})

    auc = float(roc_auc_score(y_test, y_proba))
    n_pos = int((y_test == 1).sum())
    n_neg = int((y_test == 0).sum())
    auc_lo, auc_hi, auc_se = _ic_auc_hanley(auc, n_pos, n_neg)

    tp = int(((y_test == 1) & (y_pred == 1)).sum())
    tn = int(((y_test == 0) & (y_pred == 0)).sum())
    rec1_lo, rec1_hi = _ic_wilson(tp, n_pos)
    spec_lo, spec_hi = _ic_wilson(tn, n_neg)

    # AP da classe 0 (a rara): inverte rótulo e score
    ap_classe0 = float(average_precision_score(1 - np.asarray(y_test), 1 - y_proba))

    return {
        "AUC": auc,
        "AUC_ic_low": float(auc_lo),
        "AUC_ic_high": float(auc_hi),
        "AUC_se": float(auc_se),
        "AP": float(average_precision_score(y_test, y_proba)),
        "AP_classe0": ap_classe0,
        "F1": float(c1["f1-score"]),
        "Recall": float(c1["recall"]),
        "Recall_ic_low": float(rec1_lo),
        "Recall_ic_high": float(rec1_hi),
        "Precisão": float(c1["precision"]),
        "Specificity": float(c0["recall"]),           # recall da classe rara (bom pagador)
        "Specificity_ic_low": float(spec_lo),
        "Specificity_ic_high": float(spec_hi),
        "Precisão_classe0": float(c0["precision"]),
        "F1_classe0": float(c0["f1-score"]),
        "Accuracy": float(report["accuracy"]),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "MCC": float(matthews_corrcoef(y_test, y_pred)),
        "Brier": float(brier_score_loss(y_test, y_proba)),
        "F2": float(fbeta_score(y_test, y_pred, beta=2, zero_division=0)),
        "threshold": float(threshold),
        "y_pred": y_pred,
        "y_proba": y_proba,
    }


def _logar_metricas_teste(metricas: dict, prefixo: str = "test") -> None:
    """Loga só os escalares (y_pred/y_proba são arrays e não são métricas).
    Chaves em ASCII snake_case para o 'Compare' do Databricks alinhar os runs.
    A tradução vem de MAPA_METRICA_MLFLOW (Fase 0) — fonte única, compartilhada
    com o order_by do search_runs na Fase 6."""
    for chave_pt, chave_ascii in MAPA_METRICA_MLFLOW.items():
        valor = metricas.get(chave_pt)
        if isinstance(valor, float) and not np.isnan(valor):
            mlflow.log_metric(f"{prefixo}_{chave_ascii}", valor)


def _logar_config_painel(escopo_modelo: str | None = None) -> None:
    """
    Loga o painel de controle achatado como parâmetros do run ATIVO.

    Prefixos deliberados — no painel do MLflow você filtra por 'arq.' para
    isolar mudanças de arquitetura e por 'trn.' para mudanças de protocolo.
    Só o bloco do modelo em questão é logado (arq.modelo.*), evitando poluir
    o run de um candidato com os hiperparâmetros dos outros dois.
    """
    arq = {k: v for k, v in HP_ARQUITETURA.items() if k != "modelos"}
    mlflow.log_params(achatar_config(arq, "arq"))
    mlflow.log_params(achatar_config(HP_TREINAMENTO, "trn"))

    if escopo_modelo and escopo_modelo in HP_ARQUITETURA["modelos"]:
        mlflow.log_params(
            achatar_config(HP_ARQUITETURA["modelos"][escopo_modelo]["params"], "arq.modelo")
        )

    mlflow.log_params({
        "config_hash": CONFIG_HASH,
        "config_hash_arquitetura": CONFIG_HASH_ARQ,
        "config_hash_treinamento": CONFIG_HASH_TRN,
        "experiment_tag": EXPERIMENT_TAG,
        "versao_pipeline": MODEL_VERSION,
        "random_state": RANDOM_STATE,
    })


def _tags_padrao(**extras) -> dict:
    """Tags comuns a todo run. config_hash como TAG (além de param) porque
    tags são o eixo de agrupamento do painel; params são o de comparação."""
    return {
        **TAGS_PROJETO,
        "experiment_tag": EXPERIMENT_TAG,
        "config_hash": CONFIG_HASH,
        "config_hash_arq": CONFIG_HASH_ARQ,
        "config_hash_trn": CONFIG_HASH_TRN,
        "versao_modelo": MODEL_VERSION,
        "etapa": "modelagem",
        **extras,
    }


def _avaliar_metas(metricas: dict) -> dict:
    """Compara o resultado com as metas de KPI da Fase 1 (0/1 por meta).
    Logado como métrica para permitir ordenar runs por 'bateu a meta'."""
    metas = METAS_KPI
    return {
        "meta_auc_atingida": float(metricas["AUC"] >= metas["auc_roc"]),
        "meta_recall_atingida": float(metricas["Recall"] >= metas["recall_classe1"]),
        "meta_accuracy_atingida": float(metricas["Accuracy"] >= metas["accuracy"]),
    }


def train_and_evaluate(
    nome: str,
    pipeline: Pipeline,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> dict:
    """
    Treina, valida por CV e avalia no holdout — registrando tudo em um run
    FILHO do MLflow. Atualiza MODEL_REGISTRY[nome].

    Sem parâmetros de controle na assinatura: tudo vem de HP_TREINAMENTO.
    Isso é intencional — um default de função é exatamente o tipo de valor
    que escapa do painel e depois não aparece no log.
    """
    cfg_cv = HP_TREINAMENTO["cv"]
    cv = StratifiedKFold(
        n_splits=cfg_cv["n_splits"],
        shuffle=cfg_cv["shuffle"],
        random_state=RANDOM_STATE if cfg_cv["shuffle"] else None,
    )

    with mlflow.start_run(run_name=nome, nested=True) as run:
        clf = pipeline.named_steps["clf"]

        # ── 1. TAGS: dimensões pelas quais se filtra/agrupa no painel ─────────
        mlflow.set_tags(_tags_padrao(
            modelo=nome,
            algoritmo=type(clf).__name__,
            tipo_run="candidato",
        ))

        # ── 2. PARÂMETROS ─────────────────────────────────────────────────────
        # (a) painel de controle — a INTENÇÃO declarada
        _logar_config_painel(escopo_modelo=nome)

        # (b) params efetivos do estimador — o que o sklearn REALMENTE usou,
        #     defaults incluídos. Comparar (a) com (b) revela qualquer default
        #     silencioso que tenha entrado sem passar pelo painel.
        for k, v in clf.get_params().items():
            mlflow.log_param(f"clf__{k}", v)

        # (c) forma concreta dos dados desta rodada
        mlflow.log_params({
            "n_features_total": len(ALL_FEATURES),
            "n_features_num": len(FEATURES_NUM),
            "n_features_cat": len(FEATURES_CAT),
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_test_negativos": int((y_test == 0).sum()),
            "prevalencia_treino": round(float(y_train.mean()), 4),
            "prevalencia_teste": round(float(y_test.mean()), 4),
        })

        # ── 3. VALIDAÇÃO CRUZADA ──────────────────────────────────────────────
        cv_result = cross_validate(
            pipeline, X_train, y_train, cv=cv,
            scoring=SCORING_CV,
            return_train_score=cfg_cv["return_train_score"],
            n_jobs=cfg_cv["n_jobs"],
        )

        metricas_cv = {}
        for metrica in SCORING_CV:
            scores_val = cv_result[f"test_{metrica}"]
            metricas_cv[metrica] = float(scores_val.mean())

            mlflow.log_metric(f"cv_{metrica}_mean", float(scores_val.mean()))
            mlflow.log_metric(f"cv_{metrica}_std", float(scores_val.std()))
            # score fold a fold como série (step) → dispersão visível no gráfico
            for i, s in enumerate(scores_val):
                mlflow.log_metric(f"fold_{metrica}", float(s), step=i)
            # gap treino−validação: diagnóstico direto de overfitting.
            # É a métrica a observar ao tunar max_depth / C / min_samples_leaf.
            if cfg_cv["return_train_score"]:
                scores_tr = cv_result[f"train_{metrica}"]
                mlflow.log_metric(
                    f"cv_{metrica}_gap_overfit",
                    float(scores_tr.mean() - scores_val.mean()),
                )

        # Chaves em PT para compatibilidade com o restante do notebook
        metricas_cv_pt = {
            "AUC": metricas_cv["roc_auc"],
            "AUC_std": float(cv_result["test_roc_auc"].std()),
            "AP": metricas_cv["average_precision"],
            "F1": metricas_cv["f1"],
            "Recall": metricas_cv["recall"],
            "Precisão": metricas_cv["precision"],
            "Balanced_Accuracy": metricas_cv["balanced_accuracy"],
            "MCC": metricas_cv["matthews_corrcoef"],
        }

        # ── 4. TREINO FINAL + HOLDOUT ─────────────────────────────────────────
        pipeline.fit(X_train, y_train)
        metricas_teste = avaliar_no_teste(pipeline, X_test, y_test)
        _logar_metricas_teste(metricas_teste)
        mlflow.log_metrics(_avaliar_metas(metricas_teste))

        # ── 4b. SENSIBILIDADE AO THRESHOLD (out-of-fold) ──────────────────────
        # O limiar é tunável SEM retreinar, e varrer a grade transforma a escolha
        # do corte operacional em decisão baseada em curva.
        #
        # A varredura roda sobre predições OUT-OF-FOLD do TREINO, não sobre o
        # holdout. Motivo: escolher o corte olhando o test_* é tuning feito no
        # conjunto de teste — com ~19 negativos, o limiar "ótimo" ali é ruído,
        # e depois de escolhido o holdout já não mede generalização.
        # Aqui cada previsão vem de um fold que não viu aquele cliente no treino.
        _cfg_thr = HP_TREINAMENTO["threshold"]
        grade_thr = np.round(
            np.arange(_cfg_thr["sweep_inicio"],
                      _cfg_thr["sweep_fim"] + _cfg_thr["sweep_passo"] / 2,
                      _cfg_thr["sweep_passo"]),
            4,
        )
        try:
            proba_oof = cross_val_predict(
                pipeline, X_train, y_train, cv=cv,
                method="predict_proba", n_jobs=cfg_cv["n_jobs"],
            )[:, 1]
            y_oof = np.asarray(y_train)

            for passo, thr in enumerate(grade_thr):
                pred_thr = (proba_oof >= thr).astype(int)
                rep = cast(Dict[str, Any], classification_report(
                    y_oof, pred_thr, output_dict=True, zero_division=0))
                c1 = rep.get("1", {"precision": 0, "recall": 0})
                c0 = rep.get("0", {"recall": 0})

                mlflow.log_metric("sweep_threshold", float(thr), step=passo)
                mlflow.log_metric("sweep_oof_mcc", float(matthews_corrcoef(y_oof, pred_thr)), step=passo)
                mlflow.log_metric("sweep_oof_recall_classe1", float(c1["recall"]), step=passo)
                mlflow.log_metric("sweep_oof_precision_classe1", float(c1["precision"]), step=passo)
                mlflow.log_metric("sweep_oof_specificity_classe0", float(c0["recall"]), step=passo)
                mlflow.log_metric(
                    "sweep_oof_balanced_accuracy",
                    float(balanced_accuracy_score(y_oof, pred_thr)), step=passo)

            # Limiar que maximiza o MCC out-of-fold: sugestão honesta de ponto
            # de operação, obtida sem tocar no holdout.
            _mcc_por_thr = [
                (float(t), float(matthews_corrcoef(y_oof, (proba_oof >= t).astype(int))))
                for t in grade_thr
            ]
            _thr_otimo, _mcc_otimo = max(_mcc_por_thr, key=lambda p: p[1])
            mlflow.log_metric("threshold_otimo_oof", _thr_otimo)
            mlflow.log_metric("mcc_no_threshold_otimo_oof", _mcc_otimo)
            MODEL_REGISTRY[nome]["threshold_otimo_oof"] = _thr_otimo
        except Exception as e:
            print(f"   ⚠️  Sweep out-of-fold indisponível ({type(e).__name__}): {e}")

        # ── 5. MATRIZ DE CONFUSÃO em CONTAGENS ABSOLUTAS ──────────────────────
        # Absoluto, não normalizado: o percentual esconde que a linha dos
        # negativos tem apenas ~19 casos.
        tn, fp, fn, tp = confusion_matrix(y_test, metricas_teste["y_pred"]).ravel()
        mlflow.log_metrics({
            "cm_verdadeiro_negativo": int(tn),
            "cm_falso_positivo": int(fp),
            "cm_falso_negativo": int(fn),   # custo alto: risco não detectado
            "cm_verdadeiro_positivo": int(tp),
        })

        # ── 6. MODELO + assinatura (contrato de entrada/saída) ────────────────
        # MLflow 3.x: usa name=, não artifact_path= (deprecado).
        assinatura = infer_signature(X_train, pipeline.predict(X_train))
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="modelo",
            signature=assinatura,
            input_example=X_train.head(3),
        )

        # ── 7. Artefatos navegáveis ───────────────────────────────────────────
        mlflow.log_dict(
            cast(Dict[str, Any], classification_report(
                y_test, metricas_teste["y_pred"], output_dict=True, zero_division=0)),
            "relatorio_classificacao.json",
        )
        mlflow.log_dict(LIMITACOES_CONHECIDAS, "limitacoes_metodologicas.json")
        # O painel inteiro + os params efetivos: basta copiar este JSON de volta
        # para a Fase 0 para reproduzir a rodada exatamente.
        mlflow.log_dict(
            {
                **CONFIG_EXPERIMENTO,
                "modelo": nome,
                "params_efetivos_sklearn": {k: str(v) for k, v in clf.get_params().items()},
            },
            "config_experimento.json",
        )

        run_id = run.info.run_id

    MODEL_REGISTRY[nome]["metricas_cv"] = metricas_cv_pt
    MODEL_REGISTRY[nome]["metricas_teste"] = metricas_teste
    MODEL_REGISTRY[nome]["treinado"] = True
    MODEL_REGISTRY[nome]["run_id"] = run_id

    return metricas_teste


# ══════════════════════════════════════════════════════════════════════════════
#  SELEÇÃO DO CAMPEÃO — critério único, lido de HP_TREINAMENTO['selecao']
#  Definido em UM lugar e reutilizado pelas Fases 5, 6 e 7: antes a Fase 5
#  escolhia por MCC e a Fase 7 reescolhia por AUC, e as duas podiam divergir.
# ══════════════════════════════════════════════════════════════════════════════
def selecionar_campeao(registry: dict, cfg: dict = HP_TREINAMENTO) -> str:
    """Retorna o nome do modelo vencedor segundo o critério declarado."""
    sel = cfg["selecao"]
    chave_origem = "metricas_teste" if sel["origem"] == "teste" else "metricas_cv"
    metrica, desempate = sel["metrica"], sel["criterio_desempate"]
    sinal = 1 if sel["maior_melhor"] else -1

    candidatos = {
        k: v for k, v in registry.items()
        if v.get("treinado") and v.get(chave_origem)
    }
    if not candidatos:
        raise RuntimeError("Nenhum modelo treinado no registry — execute a Fase 5.")

    def _chave(nome_modelo: str):
        m = candidatos[nome_modelo][chave_origem]
        return (sinal * m.get(metrica, -np.inf), sinal * m.get(desempate, -np.inf))

    return max(candidatos, key=_chave)


# ══════════════════════════════════════════════════════════════════════════════
#  EXECUÇÃO DA RODADA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
NOME_RODADA = (
    f"{EXPERIMENT_TAG}__v{MODEL_VERSION}__{CONFIG_HASH}"
    f"__{datetime.now().strftime('%Y%m%d_%H%M')}"
)

# ─── Guarda contra colisão de etiqueta ───────────────────────────────────────
# Duas rodadas com a mesma EXPERIMENT_TAG e configs diferentes ficam
# indistinguíveis no filtro do painel. Aviso barato, erro caro de diagnosticar.
try:
    _runs_mesma_tag = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string=f"tags.experiment_tag = '{EXPERIMENT_TAG}'",
    )
    if len(_runs_mesma_tag) > 0:
        _hashes = set()
        if "tags.config_hash" in _runs_mesma_tag.columns:
            _hashes = set(_runs_mesma_tag["tags.config_hash"].dropna()) - {CONFIG_HASH}
        if _hashes:
            print(f"⚠️  A etiqueta '{EXPERIMENT_TAG}' já foi usada com OUTRA configuração")
            print(f"    (hashes {sorted(_hashes)}). Troque EXPERIMENT_TAG antes de rodar,")
            print(f"    senão as duas rodadas ficam indistinguíveis no painel.\n")
        else:
            print(f"ℹ️  Reexecução da mesma configuração ({CONFIG_HASH}) sob a mesma etiqueta.\n")
except Exception:
    pass   # search_runs indisponível não deve impedir o treino

print(f"🔁 Validação Cruzada Estratificada ({CV_FOLDS}-Fold) + Treinamento Final")
print(f"   Rodada     : {EXPERIMENT_TAG}")
print(f"   Config hash: {CONFIG_HASH}")
print(f"   Run pai    : {NOME_RODADA}\n")
print(f"{'─'*92}")

with mlflow.start_run(run_name=NOME_RODADA) as run_pai:
    RUN_PAI_ID = run_pai.info.run_id

    mlflow.set_tags(_tags_padrao(tipo_run="rodada_comparativa"))
    _logar_config_painel()
    mlflow.log_params({
        "n_clientes_total": len(df_ml_completo),
        "n_clientes_elegiveis": len(df_treino),
        "n_train": len(X_train),
        "n_test": len(X_test),
        # Governa a largura de TODOS os ICs da Fase 6 e muda quando se tuna
        # split.test_size ou min_compras — por isso vive no run pai também.
        "n_test_negativos": int((y_test == 0).sum()),
        "prevalencia_teste": round(float(y_test.mean()), 4),
        # 'planejados', não 'comparados': se um candidato falhar no treino o
        # número de runs-filhos será menor, e um param chamado "comparados"
        # estaria mentindo sobre o que o painel contém.
        "n_modelos_planejados": len(MODEL_REGISTRY) + int(HP_TREINAMENTO["baseline"]["ativo"]),
        "target": TARGET,
        "regra_target": (
            f"TAXA_ATRASO_PAG > {LIMITE_ATRASO} "
            f"{HP_ARQUITETURA['alvo']['combinador']} TAXA_ATRASO_COM > {LIMITE_ATRASO}"
        ),
        "features": ", ".join(ALL_FEATURES),
        "modelos_ativos": ", ".join(MODEL_REGISTRY.keys()),
        **{f"meta.{k}": v for k, v in METAS_KPI.items()},
    })
    mlflow.log_dict(LIMITACOES_CONHECIDAS, "limitacoes_metodologicas.json")
    mlflow.log_dict(CONFIG_EXPERIMENTO, "config_experimento.json")

    # ── Lineage do dataset: liga o run aos dados exatos que o produziram ──────
    try:
        ds_treino = mlflow.data.from_pandas(
            pd.concat([X_train, y_train], axis=1), targets=TARGET, name="chopp_treino")
        ds_teste = mlflow.data.from_pandas(
            pd.concat([X_test, y_test], axis=1), targets=TARGET, name="chopp_teste")
        mlflow.log_input(ds_treino, context="training")
        mlflow.log_input(ds_teste, context="testing")
    except Exception as e:
        print(f"   ⚠️  Lineage de dados indisponível ({type(e).__name__}).")

    # ── BASELINE: âncora obrigatória da comparação ────────────────────────────
    # Sem isto nenhuma métrica significa nada: prever "sempre risco" já entrega
    # Accuracy 0,868 e Recall 1,000 — acima das metas declaradas na Fase 1.
    BASELINE_METRICAS = None
    if HP_TREINAMENTO["baseline"]["ativo"]:
        with mlflow.start_run(run_name="Baseline (classe majoritária)", nested=True):
            dummy = build_pipeline(DummyClassifier(
                strategy=HP_TREINAMENTO["baseline"]["strategy"],
                random_state=RANDOM_STATE,
            ))
            dummy.fit(X_train, y_train)
            BASELINE_METRICAS = avaliar_no_teste(dummy, X_test, y_test)

            mlflow.set_tags(_tags_padrao(
                modelo="Baseline",
                algoritmo="DummyClassifier",
                tipo_run="baseline",
            ))
            _logar_config_painel()
            for k, v in dummy.named_steps["clf"].get_params().items():
                mlflow.log_param(f"clf__{k}", v)
            # Mesmos params de forma dos candidatos: sem eles a linha do baseline
            # fica com células vazias no 'Compare' do Databricks.
            mlflow.log_params({
                "n_features_total": len(ALL_FEATURES),
                "n_features_num": len(FEATURES_NUM),
                "n_features_cat": len(FEATURES_CAT),
                "n_train": len(X_train),
                "n_test": len(X_test),
                "n_test_negativos": int((y_test == 0).sum()),
                "prevalencia_treino": round(float(y_train.mean()), 4),
                "prevalencia_teste": round(float(y_test.mean()), 4),
            })
            _logar_metricas_teste(BASELINE_METRICAS)
            mlflow.log_metrics(_avaliar_metas(BASELINE_METRICAS))

            print(f"{'Baseline (maioria)':<27} | Acc={BASELINE_METRICAS['Accuracy']:.3f} "
                  f"| BalAcc={BASELINE_METRICAS['Balanced_Accuracy']:.3f} "
                  f"| MCC={BASELINE_METRICAS['MCC']:.3f} | F1={BASELINE_METRICAS['F1']:.3f}")

    # ── MODELOS CANDIDATOS ────────────────────────────────────────────────────
    for nome, entry in MODEL_REGISTRY.items():
        met = train_and_evaluate(nome, entry["pipeline"], X_train, y_train, X_test, y_test)
        cv_m = MODEL_REGISTRY[nome]["metricas_cv"]
        print(
            f"  ✅ {nome:<27} | CV AUC={cv_m['AUC']:.3f}±{cv_m['AUC_std']:.3f} "
            f"| Teste AUC={met['AUC']:.3f} [{met['AUC_ic_low']:.2f}–{met['AUC_ic_high']:.2f}] "
            f"| BalAcc={met['Balanced_Accuracy']:.3f} | MCC={met['MCC']:.3f}"
        )

    # ── Seleção do campeão (critério declarado no painel) ─────────────────────
    melhor_nome = selecionar_campeao(MODEL_REGISTRY)
    met_best = MODEL_REGISTRY[melhor_nome]["metricas_teste"]
    _sel = HP_TREINAMENTO["selecao"]

    mlflow.set_tags({
        "melhor_modelo": melhor_nome,
        "criterio_selecao": f"{_sel['metrica']} ({_sel['origem']})",
    })
    _metricas_campeao = {
        "melhor_test_mcc": met_best["MCC"],
        "melhor_test_auc": met_best["AUC"],
        "melhor_test_balanced_accuracy": met_best["Balanced_Accuracy"],
        "melhor_test_recall_classe1": met_best["Recall"],
    }
    if BASELINE_METRICAS is not None:
        _metricas_campeao.update({
            "baseline_accuracy": BASELINE_METRICAS["Accuracy"],
            "baseline_mcc": BASELINE_METRICAS["MCC"],
            "ganho_mcc_sobre_baseline": met_best["MCC"] - BASELINE_METRICAS["MCC"],
            "ganho_balacc_sobre_baseline": (
                met_best["Balanced_Accuracy"] - BASELINE_METRICAS["Balanced_Accuracy"]),
            "ganho_accuracy_sobre_baseline": met_best["Accuracy"] - BASELINE_METRICAS["Accuracy"],
        })
    mlflow.log_metrics(_metricas_campeao)

print(f"{'─'*92}")
print(f"\n✅ Modelos treinados, avaliados e registrados no MLflow!")
print(f"🏆 Campeão por {_sel['metrica']} ({_sel['origem']}, desempate {_sel['criterio_desempate']}): {melhor_nome}")
if BASELINE_METRICAS is not None:
    print(f"   ├─ MCC      : {met_best['MCC']:.4f}  (baseline: {BASELINE_METRICAS['MCC']:.4f})")
    print(f"   ├─ BalAcc   : {met_best['Balanced_Accuracy']:.4f}  (baseline: {BASELINE_METRICAS['Balanced_Accuracy']:.4f})")
    print(f"   ├─ Accuracy : {met_best['Accuracy']:.4f}  (baseline: {BASELINE_METRICAS['Accuracy']:.4f})")
else:
    print(f"   ├─ MCC      : {met_best['MCC']:.4f}   ⚠️  baseline desativado no painel")
print(f"   ├─ Config   : {CONFIG_HASH}  ({EXPERIMENT_TAG})")
print(f"   └─ Run pai  : {RUN_PAI_ID}")
print(f"\n💡 Menu lateral do Databricks → 'Experiments' → expanda o run pai")
print(f"   e selecione os filhos para usar o botão 'Compare'.")
print(f"   Para comparar RODADAS de tuning, filtre por tags.config_hash ou tags.experiment_tag.")

best_model = (melhor_nome, MODEL_REGISTRY[melhor_nome])
display(MODEL_REGISTRY[melhor_nome]["pipeline"])

# 📊 Fase 6 - Avalição dos Modelos

Nesta fase é realizada a avaliação final e o confronto direto entre os algoritmos treinados na etapa anterior. O objetivo principal da **Avaliação de Modelos** é validar a capacidade de generalização dos classificadores usando o conjunto de dados de teste (dados inéditos), garantindo que o modelo escolhido seja robusto e confiável para rodar em ambiente produtivo.

### 🎯 **Relembrando Nossos Critérios de Sucesso (Métricas de Negócio)**
Para alinhar a Inteligência Artificial aos objetivos estratégicos do negócio, relembramos os KPIs de avaliação estabelecidos na Fase 1:

* 🎯 **Recall (Sensibilidade) da Classe 1:** Nossa métrica prioritária. Precisamos garantir que o modelo identifique a maior porcentagem possível de clientes de risco (**Super Detratores**), minimizando os *Falsos Negativos* (clientes nocivos que o modelo prevê como bons).
* ⚖️ **F1-Score / Precisão:** Utilizados como balizadores para evitar um excesso de *Falsos Alarmes* (bloquear bons clientes por engano), o que prejudicaria a receita da operação.
* 📈 **Curva AUC-ROC:** Avalia a capacidade macro do modelo em distinguir as duas classes ao longo de diferentes pontos de corte (*thresholds*).

---


## 📉 6.1 Painel Comparativo de Performance: Matrizes de Confusão e Curva ROC

Esta célula é responsável por gerar o diagnóstico visual e estatístico completo de cada um dos três modelos testados. Através do cruzamento das métricas no conjunto de teste, construímos a base analítica necessária para a tomada de decisão do modelo campeão.

---

#### 🛠️ **Componentes de Diagnóstico Gerados:**
1. **Matriz de Confusão Combinada:** Exibe o volume exato de acertos e erros de cada algoritmo, permitindo auditar o impacto financeiro de cada modelo (contagem de falsos negativos vs. falsos alarmes).
2. **Gráfico da Curva ROC e AUC:** Plota o comportamento dos modelos em uma única tela. O algoritmo cuja linha estiver mais próxima do canto superior esquerdo (maior valor de AUC) possui o melhor poder de separação.
3. **Tabela Consolidada de Métricas:** Um compilado final trazendo **Acurácia, Precisão, Recall e F1-Score**, permitindo uma comparação analítica e objetiva para defender a escolha do modelo perante a banca examinadora.

In [ ]:
# =============================================================================
# FASE 6 — AVALIAÇÃO FINAL DOS MODELOS
# =============================================================================

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
    }
)
sns.set_style("whitegrid")

# ─────────────────────────────────────────────────────────────────────────────
# Ordena os modelos pela MESMA métrica declarada em HP_TREINAMENTO['selecao'].
# Ordenar por AUC aqui e eleger por MCC adiante faria os gráficos contarem
# uma história diferente da decisão — exatamente o descompasso que o painel
# de controle existe para eliminar.
# ─────────────────────────────────────────────────────────────────────────────
METRICA_SELECAO = HP_TREINAMENTO["selecao"]["metrica"]

modelos_avaliados = {
    nome: entry
    for nome, entry in MODEL_REGISTRY.items()
    if entry.get("metricas_teste") and METRICA_SELECAO in entry["metricas_teste"]
}

modelos_ordenados = dict(
    sorted(
        modelos_avaliados.items(),
        key=lambda x: x[1]["metricas_teste"][METRICA_SELECAO],
        reverse=HP_TREINAMENTO["selecao"]["maior_melhor"],
    )
)

nomes_modelos = list(modelos_ordenados.keys())
n_modelos = len(nomes_modelos)

palette = sns.color_palette("tab10", n_modelos)
baseline = y_test.mean()

fig = plt.figure(figsize=(26, 15))
gs = fig.add_gridspec(2, n_modelos, height_ratios=[1.1, 1])

# =============================================================================
# MATRIZES DE CONFUSÃO NORMALIZADAS
# =============================================================================
for idx, nome in enumerate(nomes_modelos):
    ax = fig.add_subplot(gs[0, idx])

    m = modelos_ordenados[nome]["metricas_teste"]
    # Contagens ABSOLUTAS: o percentual esconderia que a linha dos bons pagadores tem apenas ~19 casos no teste.
    cm = confusion_matrix(y_test, m["y_pred"])

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        annot_kws={"size": 13},
    )

    ax.set_title(f"{nome}\nAUC = {m['AUC']:.3f}", fontweight="bold")
    ax.set_xlabel("Predição")
    ax.set_ylabel("Real")
    ax.set_xticklabels(["Bom", "Alto Risco"])
    ax.set_yticklabels(["Bom", "Alto Risco"], rotation=0)

# =============================================================================
# CURVAS ROC + PR + BARRAS (FIGURA SEPARADA)
# =============================================================================
fig2, (ax_roc, ax_pr, ax_bar) = plt.subplots(
    1, 3, figsize=(24, 7), gridspec_kw={"width_ratios": [1.2, 1.2, 1]}
)

# ── ROC ───────────────────────────────────────────────────────────────────────
for idx, nome in enumerate(nomes_modelos):
    m = modelos_ordenados[nome]["metricas_teste"]
    fpr, tpr, _ = roc_curve(y_test, m["y_proba"])
    ax_roc.plot(fpr, tpr, lw=3, color=palette[idx], label=f"{nome} (AUC={m['AUC']:.3f})")

ax_roc.plot([0, 1], [0, 1], "k--", lw=2, alpha=0.6, label="Aleatório")
ax_roc.set_title("Curva ROC — Discriminação dos Modelos", fontweight="bold")
ax_roc.set_xlabel("Taxa de Falsos Positivos")
ax_roc.set_ylabel("Taxa de Verdadeiros Positivos")
ax_roc.legend()

# ── PR ────────────────────────────────────────────────────────────────────────
for idx, nome in enumerate(nomes_modelos):
    m = modelos_ordenados[nome]["metricas_teste"]
    p, r, _ = precision_recall_curve(y_test, m["y_proba"])
    ap = average_precision_score(y_test, m["y_proba"])
    
    ax_pr.plot(r, p, lw=3, color=palette[idx], label=f"{nome} (AP={ap:.3f})")

ax_pr.axhline(baseline, ls="--", color="black", label=f"Baseline = {baseline:.2%}")
ax_pr.set_title("Curva Precision-Recall — Classe Rara", fontweight="bold")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precisão")
ax_pr.legend()

# ── BARRAS HORIZONTAIS (MUITO MAIS EXECUTIVO) ────────────────────────────────
# MCC e Balanced_Accuracy entram no gráfico porque são as únicas métricas
# aqui que não são infladas pela prevalência de 86,8% da classe positiva.
_COLS_BARRA = ["MCC", "Balanced_Accuracy", "AUC", "F1", "Recall", "Precisão", "Accuracy"]
df_res = pd.DataFrame(
    {nome: entry["metricas_teste"] for nome, entry in modelos_ordenados.items()}
).T[_COLS_BARRA]

df_res = df_res.sort_values(METRICA_SELECAO)

df_res.plot.barh(ax=ax_bar, edgecolor="black")
ax_bar.set_title("Comparação Geral de Métricas", fontweight="bold")
ax_bar.set_xlim(0, 1)
ax_bar.grid(axis="x", linestyle="--", alpha=0.4)

for i, v in enumerate(df_res[METRICA_SELECAO]):
    ax_bar.text(v + 0.01, i, f"{v:.3f}", va="center")

plt.tight_layout()

# =============================================================================
# MELHOR MODELO — REAFIRMAÇÃO, NÃO NOVA ELEIÇÃO
#
# Esta célula NÃO reelege o campeão. Ela chama a mesma função da Fase 5, que
# lê o critério de HP_TREINAMENTO['selecao'] — assim o modelo anunciado aqui,
# o da Fase 5 e o promovido a produção na Fase 7 são necessariamente o mesmo.
# (Na v1.3 esta célula reelegia por AUC e podia contradizer as outras duas.)
# =============================================================================
melhor_nome = selecionar_campeao(MODEL_REGISTRY)
_m_best = modelos_ordenados[melhor_nome]["metricas_teste"]
_sel = HP_TREINAMENTO["selecao"]

print("\n" + "=" * 70)
print(f"🏆 MELHOR MODELO: {melhor_nome}")
print(f"   Critério: {_sel['metrica']} ({_sel['origem']}) = {_m_best[_sel['metrica']]:.4f}")
print(f"   AUC = {_m_best['AUC']:.4f} "
      f"[IC95% {_m_best['AUC_ic_low']:.3f}–{_m_best['AUC_ic_high']:.3f}]  |  "
      f"BalAcc = {_m_best['Balanced_Accuracy']:.4f}")

# O campeão por AUC pode NÃO ser o campeão pelo critério adotado. Quando isso
# acontece, dizer isso em voz alta é mais informativo do que esconder:
# significa que o modelo de melhor ordenação global classifica pior a classe rara.
_campeao_auc = max(modelos_ordenados, key=lambda k: modelos_ordenados[k]["metricas_teste"]["AUC"])
if _campeao_auc != melhor_nome:
    print(f"\n   ⚠️  Por AUC o vencedor seria outro: {_campeao_auc} "
          f"(AUC={modelos_ordenados[_campeao_auc]['metricas_teste']['AUC']:.4f}).")
    print(f"      Mantido {melhor_nome} porque {_sel['metrica']} exige acerto nas DUAS")
    print(f"      classes, e com 86,8% de positivos o AUC isolado é enganoso.")
print("=" * 70)

# ── Registra as figuras como artefatos do run pai ────────────────────────────
# log_figure() envia o objeto Figure direto ao tracking server: não usa savefig, não cria diretório local e não depende do disco efêmero.
with mlflow.start_run(run_id=RUN_PAI_ID):
    mlflow.log_figure(fig, "curvas/matrizes_confusao.png")
    mlflow.log_figure(fig2, "curvas/roc_pr_comparativo.png")
    print("Figuras registradas como artefatos do run pai.")

plt.show()

In [ ]:
# ─── 6B: Importância das Variáveis por Modelo ─────────────────────────────────

def get_feature_names(pipeline: Pipeline) -> list:
    """Extrai nomes das features após OneHotEncoding do preprocessor."""
    prep = pipeline.named_steps["prep"]
    cat_enc = prep.named_transformers_["cat"]
    cat_names = list(cat_enc.get_feature_names_out(FEATURES_CAT))
    return FEATURES_NUM + cat_names

modelos_com_importance = {
    nome: entry for nome, entry in modelos_avaliados.items() if entry.get("treinado")
}

fig, axes = plt.subplots(
    1, len(modelos_com_importance), figsize=(8 * len(modelos_com_importance), 7)
)
if len(modelos_com_importance) == 1:
    axes = [axes]

for ax, (nome, entry) in zip(axes, modelos_com_importance.items()):
    pipeline = entry["pipeline"]
    clf = pipeline.named_steps["clf"]
    all_names = get_feature_names(pipeline)
    cor = CORES_MODELO.get(nome, "#7F8C8D")

    if hasattr(clf, "feature_importances_"):
        df_imp = pd.DataFrame(
            {"Feature": all_names, "Importância": clf.feature_importances_}
        )
        df_imp = df_imp.sort_values("Importância", ascending=False).head(12)
        sns.barplot(
            data=df_imp,
            x="Importância",
            y="Feature",
            ax=ax,
            color=cor,
            edgecolor="white",
        )
        ax.set_title(f"{nome}", fontweight="bold")
        ax.set_xlabel("Importância Relativa")

    elif hasattr(clf, "coef_"):
        df_coef = pd.DataFrame({"Feature": all_names, "Coeficiente": clf.coef_[0]})
        df_coef = df_coef.reindex(df_coef["Coeficiente"].abs().sort_values(ascending=False).index).head(12)
        cores_coef = ["#E74C3C" if v > 0 else "#2ECC71" for v in df_coef["Coeficiente"]]
        ax.barh(
            df_coef["Feature"][::-1],
            df_coef["Coeficiente"][::-1],
            color=cores_coef[::-1],
            edgecolor="white",
        )
        ax.set_title(f"{nome}", fontweight="bold")
        ax.set_xlabel("Peso do Coeficiente")

plt.suptitle("Importância das Variáveis por Modelo\n\nTop 12 Features", fontsize=16, fontweight="bold", y=1.05)
sns.despine()
plt.tight_layout()

with mlflow.start_run(run_id=RUN_PAI_ID):
    mlflow.log_figure(fig, "curvas/importancia_features.png")
    print("Importâncias registradas como artefato do run pai.")

plt.show()

## 📊 6.2 Painel Comparativo Consolidado via MLflow

Esta célula **recupera do servidor de tracking** os runs registrados na rodada e monta a tabela de decisão. A diferença em relação às tabelas anteriores é importante: os números aqui não vêm de variáveis em memória, e sim do que ficou efetivamente persistido no MLflow — é a mesma fonte que a banca consultaria.

Três leituras são produzidas:

1. **Tabela comparativa** — CV e teste lado a lado, ordenada por MCC.
2. **Comparação contra o baseline** — responde à pergunta que derruba a maioria dos projetos: *o modelo supera o chute constante?*
3. **Intervalos de confiança do AUC** — com apenas ~19 negativos no teste, verifica se a diferença entre os modelos é distinguível ou se está dentro da margem de erro.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  6.2 PAINEL COMPARATIVO VIA MLFLOW — search_runs()
#  Recupera do tracking server os runs desta rodada e monta a tabela de decisão
# ══════════════════════════════════════════════════════════════════════════════

# ── Recupera SOMENTE os runs filhos desta rodada ──────────────────────────────
df_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string=f"tags.mlflow.parentRunId = '{RUN_PAI_ID}'",
    # Ordenação alinhada ao critério do painel. A chave MLflow vem do mapa —
    # deduzi-la com .lower() daria 'test_recall' onde o log tem
    # 'test_recall_classe1', e 'test_precisão' (com acento) que o MLflow rejeita.
    order_by=[
        f"metrics.test_{MAPA_METRICA_MLFLOW[HP_TREINAMENTO['selecao']['metrica']]} DESC"
    ],
)

COLUNAS_PAINEL = {
    "tags.modelo": "Modelo",
    "tags.algoritmo": "Algoritmo",
    "metrics.cv_roc_auc_mean": "CV AUC",
    "metrics.cv_roc_auc_std": "CV AUC ±",
    "metrics.cv_matthews_corrcoef_mean": "CV MCC",
    "metrics.test_auc": "Teste AUC",
    "metrics.test_balanced_accuracy": "Teste BalAcc",
    "metrics.test_mcc": "Teste MCC",
    "metrics.test_recall_classe1": "Recall (risco)",
    "metrics.test_specificity_classe0": "Specificity (bom)",
    "metrics.test_accuracy": "Accuracy",
}

cols_existentes = [c for c in COLUNAS_PAINEL if c in df_runs.columns]
df_painel = df_runs[cols_existentes].rename(columns=COLUNAS_PAINEL)

print("═" * 100)
print(f"{'PAINEL COMPARATIVO — RUNS DESTA RODADA (via MLflow)':^100}")
print("═" * 100)
display(df_painel.round(4))

# ── Leitura crítica: o modelo supera o chute constante? ───────────────────────
print(f"\n{'─'*100}")
print("LEITURA CRÍTICA — comparação obrigatória contra o baseline")
print(f"{'─'*100}\n")

if BASELINE_METRICAS is None:
    raise RuntimeError(
        "Baseline desativado em HP_TREINAMENTO['baseline']['ativo'] — "
        "sem ele a leitura critica abaixo nao tem ancora. Reative e re-execute a Fase 5."
    )

acc_base = BASELINE_METRICAS["Accuracy"]
print(f"O baseline (prever SEMPRE 'alto risco', sem olhar nenhum dado) obtém:")
print(f"Accuracy = {acc_base:.3f}   Recall classe 1 = {BASELINE_METRICAS['Recall']:.3f}   "
      f"F1 classe 1 = {BASELINE_METRICAS['F1']:.3f}")
print(f"MCC = {BASELINE_METRICAS['MCC']:.3f}   BalAcc = {BASELINE_METRICAS['Balanced_Accuracy']:.3f}\n")

_metas = METAS_KPI
print(f"As metas da Fase 1 (Accuracy ≥ {_metas['accuracy']} e Recall ≥ {_metas['recall_classe1']}) "
      f"são ATINGIDAS por esse chute constante. Metas abaixo do baseline não medem valor.\n")

for nome, entry in MODEL_REGISTRY.items():
    if not entry.get("treinado"):
        continue
    m = entry["metricas_teste"]
    ganho_acc = m["Accuracy"] - acc_base
    ganho_mcc = m["MCC"] - BASELINE_METRICAS["MCC"]
    veredito = "supera" if ganho_mcc > 0 else "NÃO supera"
    print(f"  · {nome:<25} ΔAccuracy={ganho_acc:+.3f}  ΔMCC={ganho_mcc:+.3f}  → {veredito} o baseline")

# ── Sobreposição de ICs: a ordenação tem suporte estatístico? ─────────────────
print(f"\n{'─'*100}")
print("INTERVALOS DE CONFIANÇA (95%) DO AUC — o ranking é confiável?")
print(f"{'─'*100}\n")

n_neg_teste = int((y_test == 0).sum())
print(f"Atenção: o conjunto de teste tem apenas {n_neg_teste} negativos.")
print(f"Cada negativo reclassificado move a specificity em ~{100/n_neg_teste:.1f} p.p.\n")

intervalos = []
for nome, entry in MODEL_REGISTRY.items():
    if not entry.get("treinado"):
        continue
    m = entry["metricas_teste"]
    intervalos.append((nome, m["AUC"], m["AUC_ic_low"], m["AUC_ic_high"]))
    print(f"  · {nome:<25} AUC = {m['AUC']:.3f}  IC95% [{m['AUC_ic_low']:.3f} – {m['AUC_ic_high']:.3f}]")

# Detecta sobreposição entre todos os pares
sobrepostos = []
for i in range(len(intervalos)):
    for j in range(i + 1, len(intervalos)):
        n1, _, lo1, hi1 = intervalos[i]
        n2, _, lo2, hi2 = intervalos[j]
        if lo1 <= hi2 and lo2 <= hi1:
            sobrepostos.append((n1, n2))

if sobrepostos:
    print(f"\nOs ICs se sobrepõem nos pares: "
          + "; ".join(f"{a} × {b}" for a, b in sobrepostos))
    print(f"Conclusão honesta: a diferença de AUC entre esses modelos NÃO é")
    print(f"estatisticamente distinguível com este tamanho de amostra.")
    print(f"A escolha do campeão deve considerar também parcimônia e interpretabilidade.")
else:
    print(f"\nNão há sobreposição de ICs — o ranking tem suporte estatístico.")

# ── Registro do painel como artefato do run pai ───────────────────────────────
with mlflow.start_run(run_id=RUN_PAI_ID):
    mlflow.log_table(df_painel, artifact_file="painel/comparativo_modelos.json")

print(f"\n{'═'*100}")
print(f"Painel registrado como artefato no run pai.")


# 🚀 Fase 7 - Deploy

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FASE 7 — DEPLOY: Versionamento + Predict por ID do Cliente
#
#  O PKL exportado carrega o painel de controle INTEIRO. Motivo: um modelo sem
#  a configuração que o produziu não é auditável — não se sabe qual limiar
#  aplicar, quais features esperar, nem qual regra de negócio ele reproduz.
# ══════════════════════════════════════════════════════════════════════════════

def save_model_version(
    nome: str,
    pipeline: Pipeline,
    metricas: dict,
    versao: str = MODEL_VERSION,
    model_dir: str = MODEL_DIR,
) -> str:
    """
    Salva pipeline + metadados + painel de controle em PKL versionado.
    Convenção de nome: modelo_chopp_v{versao}_{hash}_{algoritmo}_{data}.pkl
    O hash no nome permite distinguir dois PKLs da mesma versão semântica
    gerados por configurações de tuning diferentes.
    Retorna: caminho do arquivo salvo.
    """
    slug = nome.lower().replace(" ", "_").replace("(", "").replace(")", "")
    data_hoje = datetime.now().strftime("%Y%m%d")
    filename = f"modelo_chopp_v{versao}_{CONFIG_HASH}_{slug}_{data_hoje}.pkl"
    filepath = os.path.join(model_dir, filename)

    export = {
        "pipeline": pipeline,
        "nome_modelo": nome,
        "versao": versao,
        # ── Painel de controle congelado ──────────────────────────────────────
        "config_experimento": CONFIG_EXPERIMENTO,
        "config_hash": CONFIG_HASH,
        "experiment_tag": EXPERIMENT_TAG,
        "hp_arquitetura": HP_ARQUITETURA,
        "hp_treinamento": HP_TREINAMENTO,
        # ── Contrato de inferência ────────────────────────────────────────────
        "features_num": FEATURES_NUM,
        "features_cat": FEATURES_CAT,
        "all_features": ALL_FEATURES,
        "target": TARGET,
        "threshold_classificacao": HP_TREINAMENTO["threshold"]["classificacao"],
        "faixas_risco": {
            "baixo": HP_TREINAMENTO["threshold"]["faixa_baixo"],
            "medio": HP_TREINAMENTO["threshold"]["faixa_medio"],
        },
        # ── Regra de negócio e proveniência ───────────────────────────────────
        "limite_atraso": LIMITE_ATRASO,
        "min_compras": MIN_COMPRAS,
        "metricas_teste": {
            k: v
            for k, v in metricas.items()
            if not hasattr(v, "__len__") or isinstance(v, float)
        },
        "data_treinamento": datetime.now().isoformat(),
        "random_state": RANDOM_STATE,
        "run_id_mlflow": MODEL_REGISTRY.get(nome, {}).get("run_id"),
    }

    with open(filepath, "wb") as f:
        pickle.dump(export, f)

    tamanho_kb = os.path.getsize(filepath) / 1024
    print(f"Salvo: {filename}  ({tamanho_kb:.1f} KB)")
    return filepath


def load_model_version(filepath: str) -> dict:
    """
    Carrega modelo PKL e exibe card do modelo.
    Retorna o dicionário completo do modelo exportado.
    """
    with open(filepath, "rb") as f:
        obj = pickle.load(f)

    print(f"\n{'═'*62}")
    print(f"{'CARD DO MODELO':^62}")
    print(f"{'═'*62}")
    print(f"Nome       : {obj['nome_modelo']}")
    print(f"Versão     : {obj['versao']}")
    print(f"Rodada     : {obj.get('experiment_tag', 'N/D')}")
    print(f"Config hash: {obj.get('config_hash', 'N/D')}")
    print(f"Treinado em: {datetime.fromisoformat(obj['data_treinamento']).strftime('%d/%m/%Y %H:%M')}")

    m = obj.get("metricas_teste", {})
    for rotulo, chave in [("AUC-ROC", "AUC"), ("MCC", "MCC"),
                          ("Bal. Accuracy", "Balanced_Accuracy"),
                          ("F1-Score", "F1"), ("Recall", "Recall")]:
        valor = m.get(chave)
        print(f"{rotulo:<11}: {valor:.4f}" if isinstance(valor, float) else f"{rotulo:<11}: N/D")

    print(f"Features   : {len(obj['all_features'])} "
          f"({len(obj['features_num'])} num + {len(obj['features_cat'])} cat)")
    print(f"Threshold  : {obj.get('threshold_classificacao', 0.5):.2f}")
    print(f"Regra alvo : atraso > {obj.get('limite_atraso', float('nan')):.0%} | "
          f"elegíveis: compras > {obj.get('min_compras', 'N/D')}")
    print(f"Run MLflow : {obj.get('run_id_mlflow', 'N/D')}")
    print(f"{'═'*62}")
    return obj


# ─── Exportando todos os modelos treinados ────────────────────────────────────
print("Exportando modelos para PKL...\n")
caminhos_salvos = {}

for nome, entry in MODEL_REGISTRY.items():
    if entry["treinado"] and entry["metricas_teste"]:
        caminho = save_model_version(nome, entry["pipeline"], entry["metricas_teste"])
        caminhos_salvos[nome] = caminho
        MODEL_REGISTRY[nome]["filepath"] = caminho

# ─── Melhor modelo para produção ─────────────────────────────────────────────
# MESMO critério da Fase 5 — selecionar_campeao() lê HP_TREINAMENTO['selecao'].
# (Na v1.3 esta célula reselecionava por AUC enquanto a Fase 5 usava MCC; as
#  duas podiam apontar modelos diferentes e o PKL promovido não era o campeão
#  anunciado. Agora há um único critério, declarado no painel.)
melhor_nome = selecionar_campeao(MODEL_REGISTRY)
MODELO_PRODUCAO_PATH = MODEL_REGISTRY[melhor_nome]["filepath"]
_sel = HP_TREINAMENTO["selecao"]

print(f"\nModelos exportados : {len(caminhos_salvos)}")
print(f"Critério de seleção: {_sel['metrica']} ({_sel['origem']}), desempate por {_sel['criterio_desempate']}")
print(f"Modelo de produção : {melhor_nome}")
print(f"Arquivo            : {MODELO_PRODUCAO_PATH}")

# ══════════════════════════════════════════════════════════════════════════════
#  REGISTRO DO MODELO CAMPEÃO NO MLFLOW
#  O PKL acima vive no disco efêmero do serverless e some entre sessões.
#  O registro abaixo é a cópia durável, com assinatura e linhagem.
# ══════════════════════════════════════════════════════════════════════════════
RUN_ID_CAMPEAO = MODEL_REGISTRY[melhor_nome].get("run_id")

if RUN_ID_CAMPEAO:
    uri_modelo = f"runs:/{RUN_ID_CAMPEAO}/modelo"
    print(f"\nModelo campeão no MLflow: {uri_modelo}")

    # Anexa o PKL + metadados ao run como artefato adicional
    with mlflow.start_run(run_id=RUN_ID_CAMPEAO):
        mlflow.log_artifact(MODELO_PRODUCAO_PATH, artifact_path="modelo_pkl")
        mlflow.set_tags({
            "promovido_para": "producao",
            "criterio_promocao": f"{_sel['metrica']} ({_sel['origem']})",
        })

    # Registro no Model Registry (Unity Catalog usa nome de 3 níveis).
    NOME_REGISTRADO = "workspace.default.chopp_risco"
    try:
        versao = mlflow.register_model(
            model_uri=uri_modelo,
            name=NOME_REGISTRADO,
            tags={
                "config_hash": CONFIG_HASH,
                "experiment_tag": EXPERIMENT_TAG,
                "criterio_selecao": _sel["metrica"],
            },
        )
        print(f"Registrado como {NOME_REGISTRADO} versão {versao.version}")
    except Exception as e:
        print(f"Model Registry indisponível ({type(e).__name__}): {e}")
        print(f"O modelo continua acessível por: {uri_modelo}")

    print(f"\nPara carregar em produção:")
    print(f"modelo = mlflow.sklearn.load_model('{uri_modelo}')")
else:
    print("\nrun_id do campeão não encontrado — execute a Fase 5 antes.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  predict_by_id — Função de Produção
#  Simula: SELECT features FROM df_ml_completo WHERE ID_PESSOA = :id
#
#  Os limiares NÃO são literais nesta célula. A ordem de precedência é:
#     1. argumento explícito da chamada        (override pontual)
#     2. o que está gravado no PKL do modelo   (o limiar com que ele foi avaliado)
#     3. HP_TREINAMENTO['threshold']           (o painel da sessão atual)
#  Isso importa porque um PKL carregado de uma rodada antiga precisa continuar
#  usando o limiar DAQUELA rodada, e não o do painel que estiver aberto agora.
# ══════════════════════════════════════════════════════════════════════════════

def _resolver_thresholds(modelo_obj: dict, threshold=None) -> tuple:
    """Retorna (threshold_classificacao, faixa_baixo, faixa_medio)."""
    faixas_pkl = modelo_obj.get("faixas_risco", {})
    padrao = HP_TREINAMENTO["threshold"]

    thr = (
        threshold
        if threshold is not None
        else modelo_obj.get("threshold_classificacao", padrao["classificacao"])
    )
    baixo = faixas_pkl.get("baixo", padrao["faixa_baixo"])
    medio = faixas_pkl.get("medio", padrao["faixa_medio"])
    return float(thr), float(baixo), float(medio)


def _nivel_risco(prob: float, baixo: float, medio: float) -> str:
    """Faixa qualitativa da probabilidade — para leitura do gestor."""
    if prob < baixo:
        return "BAIXO"
    if prob < medio:
        return "MÉDIO"
    return "ALTO"


def predict_by_id(
    id_pessoa,
    df_usuario: pd.DataFrame,
    modelo_obj: dict,
    threshold: float | None = None,
) -> dict:
    """
    Classifica um cliente pelo seu ID, buscando as features em df_usuario.

    Parâmetros
    ----------
    id_pessoa  : valor do ID_PESSOA a ser consultado
    df_usuario : df_ml_completo — tabela de usuários (NÃO o df_treino)
    modelo_obj : dicionário retornado por load_model_version() ou direto do registry
    threshold  : limiar de probabilidade para classificar como Alto Risco.
                 None (padrão) ⇒ usa o limiar gravado no modelo, e na ausência
                 dele o de HP_TREINAMENTO['threshold']['classificacao'].
                 Reduza para 0.35-0.40 para priorizar recall (detectar mais riscos);
                 a curva completa está logada no MLflow como sweep_* na Fase 5.

    Retorna
    -------
    dict com classificacao, probabilidade_risco, nivel_risco, nome_cliente,
    modelo_utilizado, auc_modelo, features_usadas

    Exemplo
    -------
    resultado = predict_by_id(
        id_pessoa=1042,
        df_usuario=df_ml_completo,
        modelo_obj=load_model_version(MODELO_PRODUCAO_PATH)
    )
    """
    # ── SELECT pelo ID ─────────────────────────────────────────────────────────
    row = df_usuario[df_usuario['ID_PESSOA'] == id_pessoa]

    if row.empty:
        return {
            'erro'           : f'ID_PESSOA {id_pessoa} não encontrado em df_usuario.',
            'classificacao'  : None,
            'nivel_risco'    : 'DESCONHECIDO',
        }

    # ── Monta DataFrame de features ───────────────────────────────────────────
    pipeline   = modelo_obj['pipeline']
    all_feats  = modelo_obj['all_features']
    nome_cli   = row['NOME_CLIENTE'].values[0] if 'NOME_CLIENTE' in row.columns else 'N/D'

    faltando = [c for c in all_feats if c not in row.columns]
    if faltando:
        return {
            'erro'         : f'Features ausentes em df_usuario: {faltando}',
            'classificacao': None,
            'nivel_risco'  : 'DESCONHECIDO',
        }

    df_input = row[all_feats].copy()

    # ── Limiares vigentes ─────────────────────────────────────────────────────
    thr, faixa_baixo, faixa_medio = _resolver_thresholds(modelo_obj, threshold)

    # ── Predição ──────────────────────────────────────────────────────────────
    probabilidade = float(pipeline.predict_proba(df_input)[:, 1][0])
    classificacao = int(probabilidade >= thr)
    nivel = _nivel_risco(probabilidade, faixa_baixo, faixa_medio)

    metricas = modelo_obj.get('metricas_teste', {})

    return {
        'id_pessoa'         : id_pessoa,
        'nome_cliente'      : nome_cli,
        'classificacao'     : classificacao,   # 0 = Bom | 1 = Alto Risco
        'probabilidade_risco': round(probabilidade, 4),
        'nivel_risco'       : nivel,
        'modelo_utilizado'  : modelo_obj['nome_modelo'],
        'versao_modelo'     : modelo_obj.get('versao', 'N/D'),
        'config_hash'       : modelo_obj.get('config_hash', 'N/D'),
        'auc_modelo'        : round(float(metricas.get('AUC', 0)), 4),
        'mcc_modelo'        : round(float(metricas.get('MCC', 0)), 4),
        'threshold_usado'   : thr,
        'faixas_usadas'     : {'baixo': faixa_baixo, 'medio': faixa_medio},
        'features_usadas'   : df_input.iloc[0].to_dict(),
    }


# ─── Função de predição por dados manuais (sem necessidade de ID) ─────────────
def predict_manual(dados: dict, modelo_obj: dict, threshold: float | None = None) -> dict:
    """
    Classifica um cliente com dados informados manualmente (sem busca por ID).
    Útil para simular novos clientes ou testar cenários hipotéticos.
    Segue a mesma precedência de limiares de predict_by_id().
    """
    pipeline  = modelo_obj['pipeline']
    all_feats = modelo_obj['all_features']
    df_input  = pd.DataFrame([dados])[all_feats]

    thr, faixa_baixo, faixa_medio = _resolver_thresholds(modelo_obj, threshold)

    prob = float(pipeline.predict_proba(df_input)[:, 1][0])
    return {
        'classificacao'      : int(prob >= thr),
        'probabilidade_risco': round(prob, 4),
        'nivel_risco'        : _nivel_risco(prob, faixa_baixo, faixa_medio),
        'modelo_utilizado'   : modelo_obj['nome_modelo'],
        'threshold_usado'    : thr,
    }


_thr_p, _b_p, _m_p = (
    HP_TREINAMENTO["threshold"]["classificacao"],
    HP_TREINAMENTO["threshold"]["faixa_baixo"],
    HP_TREINAMENTO["threshold"]["faixa_medio"],
)
print("Funções de produção prontas!")
print("   ├─ predict_by_id(id_pessoa, df_ml_completo, modelo_obj)")
print("   └─ predict_manual(dados_dict, modelo_obj)")
print(f"\n   Limiares vigentes no painel (usados quando o PKL não traz os seus):")
print(f"   ├─ Classificação: prob ≥ {_thr_p:.2f} ⇒ ALTO_RISCO")
print(f"   └─ Faixas       : <{_b_p:.2f} BAIXO · <{_m_p:.2f} MÉDIO · ≥{_m_p:.2f} ALTO")
print(f"\n   💡 Para tunar o corte sem retreinar: altere")
print(f"      HP_TREINAMENTO['threshold'] na Fase 0 e consulte as métricas")
print(f"      sweep_* logadas no MLflow para escolher o ponto de operação.")

In [ ]:
# ─── Carrega modelo de produção e testa ──────────────────────────────────────
modelo_producao = load_model_version(MODELO_PRODUCAO_PATH)

# ══════════════════════════════════════════════════════════════════════════════
#  TESTE 1 — predict_by_id: amostragem inteligente garantindo os 3 níveis
#
#  Estratégia: rodar predict_proba em TODA a amostra de veteranos,
#  segregar por nivel_risco e pinçar 1 representante de cada bucket.
#  Isso garante BAIXO · MÉDIO · ALTO no output independentemente dos dados.
# ══════════════════════════════════════════════════════════════════════════════

EMOJI_NIVEL = {"BAIXO": "🟢", "MÉDIO": "🟡", "ALTO": "🔴"}

# ⚠️ Os três perfis sintéticos usados mais abaixo foram calibrados à mão para
#    cair em faixas distintas com os limiares padrão (0.35 / 0.65) e com as
#    features de atraso presentes. Se você tunar as faixas ou remover
#    MEDIA_DIAS_ATRASO_* (o experimento de ablação), os perfis podem cair todos
#    na mesma faixa — a célula continua rodando, mas perde a função didática.

# Amostra base: apenas veteranos com histórico suficiente
df_amostra = df_ml_completo[df_ml_completo["FREQUENCIA_COMPRAS"] > MIN_COMPRAS].copy()

# Predição vetorizada em toda a amostra (sem loop visível)
_probs = modelo_producao["pipeline"].predict_proba(df_amostra[ALL_FEATURES])[:, 1]
df_amostra = df_amostra.assign(_PROB=_probs)
# Reutiliza _nivel_risco() (Fase 7) em vez de reimplementar os cortes com
# pd.cut: duas fontes de verdade para a mesma regra acabam divergindo quando
# alguém tuna as faixas em só um dos lugares.
_faixas = modelo_producao.get("faixas_risco", {
    "baixo": HP_TREINAMENTO["threshold"]["faixa_baixo"],
    "medio": HP_TREINAMENTO["threshold"]["faixa_medio"],
})
df_amostra["_NIVEL"] = [
    _nivel_risco(p, _faixas["baixo"], _faixas["medio"]) for p in df_amostra["_PROB"]
]

# Seleciona 1 representante por nível (seed para reprodutibilidade)
ids_por_nivel = {}
for nivel in ["BAIXO", "MÉDIO", "ALTO"]:
    subset = df_amostra[df_amostra["_NIVEL"] == nivel]
    if not subset.empty:
        ids_por_nivel[nivel] = (
            subset.sample(1, random_state=RANDOM_STATE)["ID_PESSOA"].values[0]
        )

print("\n🧪 TESTE 1 — predict_by_id com clientes reais do dataset")
print(f"   Amostra base : {len(df_amostra):,} clientes veteranos")
print(f"   Estratégia   : 1 representante por nível de risco\n")

print("=" * 65)
print(f"  {'Nível':^7} {'ID':^10} {'Nome':^22} {'Prob':^7} {'Classificação'}")
print(f"  {'─'*7} {'─'*10} {'─'*22} {'─'*7} {'─'*16}")

for nivel, id_p in ids_por_nivel.items():
    res = predict_by_id(id_p, df_ml_completo, modelo_producao)
    emoji = EMOJI_NIVEL.get(res["nivel_risco"], "⚪")
    classi = "Alto Risco (1)" if res["classificacao"] == 1 else "Bom Pagador (0)"
    nome_s = str(res["nome_cliente"])[:20]
    print(
        f"  {emoji} {nivel:^7} {str(id_p):^10} {nome_s:<22} "
        f"{res['probabilidade_risco']:>5.1%}  {classi}"
    )

print(f"\n{'─'*65}")
print("  Detalhamento individual:\n")

for nivel, id_p in ids_por_nivel.items():
    res = predict_by_id(id_p, df_ml_completo, modelo_producao)
    emoji = EMOJI_NIVEL.get(res["nivel_risco"], "⚪")
    feats = res.get("features_usadas", {})
    print(f"  {emoji} {res['nome_cliente']} (ID {res['id_pessoa']})")
    print(f"     ├─ Nível de Risco  : {res['nivel_risco']}")
    print(f"     ├─ Probabilidade   : {res['probabilidade_risco']:.1%}")
    print(f"     ├─ Classificação   : {'Alto Risco (1)' if res['classificacao']==1 else 'Bom Pagador (0)'}")
    if feats:
        print(f"     ├─ Freq. Compras   : {int(feats.get('FREQUENCIA_COMPRAS', 0))} pedidos")
        print(f"     ├─ Ticket Médio    : R$ {feats.get('TICKET_MEDIO', 0):.0f}")
        print(f"     ├─ Dias Atr. Pag.  : {feats.get('MEDIA_DIAS_ATRASO_PAG', 0):.1f} dias")
        print(f"     └─ Dias Atr. Com.  : {feats.get('MEDIA_DIAS_ATRASO_COM', 0):.1f} dias")
    print()

# Distribuição geral na amostra
print(f"{'─'*65}")
print("  📊 Distribuição de risco — amostra de veteranos:\n")
dist = df_amostra["_NIVEL"].value_counts().sort_index()
total_amostra = len(df_amostra)
for nivel_d in ["BAIXO", "MÉDIO", "ALTO"]:
    n = dist.get(nivel_d, 0)
    pct = n / total_amostra * 100
    barra = "█" * max(1, int(pct / 3))
    print(
        f"  {EMOJI_NIVEL[nivel_d]} {nivel_d:^6} : {barra:<22} "
        f"{n:>4} clientes ({pct:.1f}%)"
    )

# ══════════════════════════════════════════════════════════════════════════════
#  TESTE 2 — predict_manual: 3 perfis sintéticos, 1 por nível de risco
#
#  Os valores foram calibrados para que cada perfil caia em um bucket
#  distinto: BAIXO (< 35%) · MÉDIO (35-65%) · ALTO (> 65%)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print("🧪 TESTE 2 — predict_manual: perfis sintéticos calibrados")
print(f"   Objetivo : cobrir os 3 possíveis outputs do modelo")
print(f"{'='*65}\n")

perfis = [
    (
        "Cliente Exemplar",
        "PJ consolidada · alta frequência · zero atrasos históricos",
        {
            "FREQUENCIA_COMPRAS": 18,
            "TICKET_MEDIO": 820.0,
            "TOTAL_GASTO": 14760.0,
            "DIAS_DESDE_PRIMEIRA_COMPRA": 1200,
            "DIAS_DESDE_ULTIMA_COMPRA": 8,
            "TOTAL_PARCELAS": 36,
            "TOTAL_COMODATOS": 8,
            "MEDIA_DIAS_ATRASO_PAG": 0.0,
            "MEDIA_DIAS_ATRASO_COM": 0.0,
            "PERFIL": "Pessoa Jurídica",
            "CIDADE": "PONTA PORÃ",
            "PAGAMENTO": "PIX",
        },
        "✅ APROVADO: Cliente elegível para novos pedidos e comodatos.",
    ),
    (
        "Cliente em Observação",
        "PF ativa · atrasos leves esporádicos · abaixo do threshold",
        {
            "FREQUENCIA_COMPRAS": 6,
            "TICKET_MEDIO": 390.0,
            "TOTAL_GASTO": 2340.0,
            "DIAS_DESDE_PRIMEIRA_COMPRA": 520,
            "DIAS_DESDE_ULTIMA_COMPRA": 48,
            "TOTAL_PARCELAS": 10,
            "TOTAL_COMODATOS": 2,
            "MEDIA_DIAS_ATRASO_PAG": 4.0,
            "MEDIA_DIAS_ATRASO_COM": 2.5,
            "PERFIL": "Pessoa Física",
            "CIDADE": "PONTA PORÃ",
            "PAGAMENTO": "BOLETO",
        },
        "⚡ ATENÇÃO: Monitorar próximas transações. Avaliar exigência de entrada.",
    ),
    (
        "Cliente Crítico",
        "PF nova · baixa frequência · atrasos severos financeiro e comodato",
        {
            "FREQUENCIA_COMPRAS": 3,
            "TICKET_MEDIO": 210.0,
            "TOTAL_GASTO": 630.0,
            "DIAS_DESDE_PRIMEIRA_COMPRA": 170,
            "DIAS_DESDE_ULTIMA_COMPRA": 220,
            "TOTAL_PARCELAS": 5,
            "TOTAL_COMODATOS": 3,
            "MEDIA_DIAS_ATRASO_PAG": 58.0,
            "MEDIA_DIAS_ATRASO_COM": 82.0,
            "PERFIL": "Pessoa Física",
            "CIDADE": "OUTRAS CIDADES",
            "PAGAMENTO": "OUTROS",
        },
        "⛔ BLOQUEADO: Não aprovar novas transações sem garantias adicionais.",
    ),
]

for label, descricao, dados, acao in perfis:
    res = predict_manual(dados, modelo_producao)
    emoji = EMOJI_NIVEL.get(res["nivel_risco"], "⚪")
    classi = "Alto Risco (1)" if res["classificacao"] == 1 else "Bom Pagador (0)"
    barra_prob = "█" * int(res["probabilidade_risco"] * 20)
    print(f"  {emoji}  {label}")
    print(f"     {descricao}")
    print(f"  {'─'*55}")
    print(f"  {'Probabilidade de Risco':<28} {res['probabilidade_risco']:.1%}  [{barra_prob:<20}]")
    print(f"  {'Classificação':<28} {classi}")
    print(f"  {'Nível':<28} {emoji} {res['nivel_risco']}")
    print(f"  {acao}")
    print()

print(f"{'─'*65}")
_thr = HP_TREINAMENTO["threshold"]
print(f"  ℹ️  Regra de decisão (HP_TREINAMENTO['threshold']):")
print(f"     Classificação binária : prob ≥ {_thr['classificacao']:.0%} ⇒ Alto Risco (1)")
print(f"     Faixa de leitura     : <{_thr['faixa_baixo']:.0%} BAIXO · "
      f"<{_thr['faixa_medio']:.0%} MÉDIO · ≥{_thr['faixa_medio']:.0%} ALTO")
print(f"     ⚠️  As faixas são rótulos de leitura para o gestor e NÃO coincidem")
print(f"        com o corte binário: com threshold {_thr['classificacao']:.2f}, parte da")
print(f"        faixa MÉDIO já e classificada como Alto Risco (1).")
print(f"\n  Modelo : {modelo_producao['nome_modelo']} v{modelo_producao['versao']}")
print(f"  AUC    : {modelo_producao['metricas_teste'].get('AUC', 0):.4f}"
      if isinstance(modelo_producao.get('metricas_teste', {}).get('AUC'), float) 
      else "")

In [0]:
# ─── Sumário Final do Projeto ──────────────────────────────────────────────────
print("═" * 65)
print(f"{'🍺 SUMÁRIO FINAL — INTELIGÊNCIA DE RISCO EM CHOPP':^65}")
print("═" * 65)

etapas = [
    ("Extração/Carga de Dados", "✅", "3 views + 7 tabelas dimensionais"),
    ("Filtro Core Business", "✅", "Regex Chopp/Chopeiras"),
    ("Auditoria Relacional", "✅", "IDs independentes ERP confirmados"),
    ("EDA: Perfil 360° + Risco + Aging", "✅", "Dashboards completos"),
    ("Feature Engineering (grão cliente)", "✅", "RFM + Comportamental + Demográfico"),
    ("df_ml_completo (lookup usuários)", "✅", f"{len(df_ml_completo):,} clientes"),
    ("df_treino (modelagem)", "✅", f"{len(df_treino):,} veteranos"),
    ("build_pipeline() modular", "✅", "Reaproveitável para qualquer clf"),
    ("MODEL_REGISTRY com versões", "✅", f"{len(MODEL_REGISTRY)} modelos treinados"),
    ("Validação Cruzada Estratificada", "✅", f"{CV_FOLDS}-Fold"),
    ("Export PKL versionado", "✅", f"{len(caminhos_salvos)} arquivos salvos"),
    ("predict_by_id(id_pessoa, ...)", "✅", "SELECT → Pipeline → Dict"),
    ("predict_manual(dados, ...)", "✅", "Simulação de novos clientes"),
]

print(f"\n  {'Etapa':<40} {'Status':<8} {'Detalhe'}")
print(f"  {'─'*40} {'─'*8} {'─'*20}")

for etapa, status, detalhe in etapas:
    print(f"  {etapa:<40} {status:<8} {detalhe}")

print(f"\n{'─'*65}")
print(f"  🏆 Melhor modelo em produção : {melhor_nome}")
print(f"  📄 Arquivo PKL               : {os.path.basename(MODELO_PRODUCAO_PATH)}")
print(f"  📊 AUC-ROC (teste)           : {MODEL_REGISTRY[melhor_nome]['metricas_teste']['AUC']:.4f}")
print(f"  🎯 Recall classe 1           : {MODEL_REGISTRY[melhor_nome]['metricas_teste']['Recall']:.4f}")

print(f"\n{'═'*65}")
print(f"{'🚀 PRÓXIMOS PASSOS RECOMENDADOS':^65}")
print(f"{'═'*65}")

proximos = [
    "1. Threshold calibration: ajustar para 0.35-0.40 se recall > precisão for prioridade",
    "2. Novos clientes: remover MEDIA_DIAS_ATRASO_* para cold-start sem histórico",
    "3. Monitoramento: recalibrar trimestralmente e logar versão usada por predição",
]

for p in proximos:
    print(f"  {p}")
print()